# FLIR: RandomMono + BTF / 10k iterations

Attach the FLIR Kaggle dataset, select **GPU** and enable **Internet** for
uncached dependencies/pretrained FCOS weights, then **Run All** in a fresh
kernel. Only this notebook is needed; project source is embedded below.

Base: freshly fetched `origin/feat/fcos-distillation` at `1cb91c5`.
The experiment changes source intermediate views only. Existing losses,
teacher EMA, phase-four RGB supervision and optional pseudo-label audits
are retained. There is no new L1/contrastive loss.

Edit the **Experiment settings** cell for batch size, paths, resume, or
ablations. Default training is 10,000 iterations. Checkpoints, training
logs, configuration, source manifest, preview and final COCO metrics are
saved under `/kaggle/working/output_randommono_btf`.


In [ ]:
# Keep the Kaggle CUDA torch/torchvision stack. Install only missing optional
# packages into a private directory, never another OpenCV alongside global cv2.
import importlib
import os
from pathlib import Path
import subprocess
import sys

DEPENDENCY_ROOT = Path("/kaggle/working/.flir_runtime_deps")
if not Path("/kaggle/working").is_dir():
    DEPENDENCY_ROOT = Path.cwd() / ".flir_runtime_deps"
if DEPENDENCY_ROOT.is_dir():
    sys.path.insert(0, str(DEPENDENCY_ROOT))

def ensure_notebook_dependencies():
    probes = {
        "opencv-contrib-python-headless==4.11.0.86":
            "import cv2; assert hasattr(getattr(cv2,'ximgproc',None),'bilateralTextureFilter')",
        "pycocotools==2.0.11": "from pycocotools.coco import COCO; from pycocotools.cocoeval import COCOeval",
    }
    missing = []
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([str(DEPENDENCY_ROOT), env.get("PYTHONPATH", "")])
    for package, probe in probes.items():
        run = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)
        if run.returncode:
            missing.append(package)
    if missing:
        if any(name == "cv2" or name.startswith("cv2.") for name in sys.modules) and any(p.startswith("opencv") for p in missing):
            raise RuntimeError("An incompatible cv2 is already loaded. Restart the kernel and Run All from the first cell.")
        DEPENDENCY_ROOT.mkdir(parents=True, exist_ok=True)
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--no-deps", "--only-binary=:all:", "--upgrade", "--target", str(DEPENDENCY_ROOT), *missing,
            ])
        except subprocess.CalledProcessError as error:
            raise RuntimeError("Optional dependencies could not be installed. Enable Kaggle Internet or provide the required wheels in this private dependency directory, then restart and Run All.") from error
        if str(DEPENDENCY_ROOT) not in sys.path:
            sys.path.insert(0, str(DEPENDENCY_ROOT))
        importlib.invalidate_caches()
    import cv2
    from pycocotools.coco import COCO
    if not hasattr(getattr(cv2, "ximgproc", None), "bilateralTextureFilter"):
        raise RuntimeError("The active OpenCV still has no BTF; restart the kernel after dependency installation.")
    print("Dependencies ready. OpenCV:", cv2.__version__, "| torch/torchvision were not replaced.")

ensure_notebook_dependencies()


## Embedded canonical source
Each `%%flir_module <name>` code cell below contains visible Python:
imports, classes and functions from the matching CLI module. You can edit
this code directly. The first line preserves its module namespace and
imports. Run every module cell and the loader before training. After a
code edit, run the edited module cell, then the loader and subsequent
cells, or use Run All.
No project .py files or git clone are needed.


In [ ]:
EXPECTED_SOURCE_MANIFEST = {'config': {'path': 'config.py',
            'is_package': False,
            'sha256': 'f7c86dde99c5d70f58d88c62e9ffa7c6fe2eb38a927d99c0b862bb6eb934652c'},
 'data': {'path': 'data/__init__.py',
          'is_package': True,
          'sha256': '6ad2a4462980e10b259550ddb58cfed767590228a5f2a25ad91f684989ccf3c6'},
 'data.augmentations': {'path': 'data/augmentations.py',
                        'is_package': False,
                        'sha256': 'e3a8f4831c61ba65aa778925c385c84d4acdb9ae8db25f2a7530e05b256f9a4f'},
 'data.dataloaders': {'path': 'data/dataloaders.py',
                      'is_package': False,
                      'sha256': 'dcfab22738140d9186f1bfa8e2a09efd22aff2caab314165edbcf17fecb66a06'},
 'data.datasets': {'path': 'data/datasets.py',
                   'is_package': False,
                   'sha256': '995293d2683f7b02d186d5ce2e37bd70d96e4c3b3093f094fc7bad1a39a0edac'},
 'data.ir_annotations': {'path': 'data/ir_annotations.py',
                         'is_package': False,
                         'sha256': '075c5e3244317a4ae125c7ae76a1394304914693945f726ee7be1905aedf8bed'},
 'data.preprocessing': {'path': 'data/preprocessing.py',
                        'is_package': False,
                        'sha256': 'd70ad53726471a9087184cf833c0da7b42b107238c0eec5b3b32a4dc1970ff1e'},
 'ema': {'path': 'ema.py',
         'is_package': False,
         'sha256': 'e296997de06b3c3489bfb80ff318f3868781e9468be9789cb091c1092037dd3b'},
 'evaluate_coco': {'path': 'evaluate_coco.py',
                   'is_package': False,
                   'sha256': 'b1e81ace95926978b4d8a4ab2bd3d70b054d638d5e8a62967b21f3acf9485510'},
 'evaluator': {'path': 'evaluator.py',
               'is_package': False,
               'sha256': 'c254bbe8dbc41e72befcf61c33dcf1c2a6dfac7f6377a0f58101ae120f4be410'},
 'example_flir': {'path': 'example_flir.py',
                  'is_package': False,
                  'sha256': 'bf67ca1a15d2d53bfa60294664fe36521f4b408a153318fea0981e928e87e548'},
 'intermediate_domain': {'path': 'intermediate_domain.py',
                         'is_package': False,
                         'sha256': '21b540fdde2b40a8e5554c20cfd2b1331769255d2ff49fdf493f0ff60a1636df'},
 'loss': {'path': 'loss/__init__.py',
          'is_package': True,
          'sha256': 'd5013e11244cba1c3c06dc9563b09f5f410e4722416682c3d3c4886ccaff7746'},
 'loss.d3t_criterion': {'path': 'loss/d3t_criterion.py',
                        'is_package': False,
                        'sha256': 'eee0d8d96d90b3dea34845c2e6f4e6d0f2036d0a757f1afbf083d9aced54be23'},
 'models': {'path': None,
            'is_package': True,
            'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'},
 'models.d3t_adapter': {'path': 'models/d3t_adapter.py',
                        'is_package': False,
                        'sha256': 'f7bc1c34044808e3487be3d5f6fec0f6084b48d98faa8aedbfe419e828ab1618'},
 'models.d3t_wrapper': {'path': 'models/d3t_wrapper.py',
                        'is_package': False,
                        'sha256': '52d49ac7ff238e477d40fdaefccaba9d58047b41f6d15b3533bbf9da6ead9856'},
 'models.fcos_factory': {'path': 'models/fcos_factory.py',
                         'is_package': False,
                         'sha256': '5ffa6817180b30206a8dddbc5252680a2b82aeacdf78e129defab4e03a376d3c'},
 'models.torchvision_fcos_adapter': {'path': 'models/torchvision_fcos_adapter.py',
                                     'is_package': False,
                                     'sha256': 'b10ab53e496bb49e8235fe90bade1bd5481192e4f1cf0d0e166390b7101c7750'},
 'notebook_support': {'path': 'notebook_support.py',
                      'is_package': False,
                      'sha256': '34b8b774b7cc115df8f2354bb689feeeb5de64f95c09a88d3094aaebb1bd63a7'},
 'optimization': {'path': 'optimization.py',
                  'is_package': False,
                  'sha256': '6dc2be42765956e7d5eaca5aa38a873f09502e56c4f134bad079e256e794450f'},
 'saga': {'path': 'saga.py',
          'is_package': False,
          'sha256': '52fe64e7c50509789877fbe5261edee39762c882103f64d1f5c46b83b41e4c74'},
 'scheduler': {'path': 'scheduler.py',
               'is_package': False,
               'sha256': 'e4e2a1f3e8363183c795955d961e2620f0922864b6f3f0c5b9a59392ac02cf75'},
 'trainer': {'path': 'trainer.py',
             'is_package': False,
             'sha256': 'eb80e902eb9a8efd57178e41d01ce93f8da3656cf7ada399f61d9e9ebea35100'},
 'training': {'path': 'training/__init__.py',
              'is_package': True,
              'sha256': '0650fcaa44404255fe48d01ddb7078b22c19a885fd026b342ccbfdb9367db074'},
 'training.bootstrap': {'path': 'training/bootstrap.py',
                        'is_package': False,
                        'sha256': '8f5b75b819728ae3abbd36cdfa4c6ee2d4a9dd192300dc3c4a2564d228e259f6'},
 'training.data_flow': {'path': 'training/data_flow.py',
                        'is_package': False,
                        'sha256': 'de0cd50290d068d7469c3a1cf8b777e0f63fdb1d2a103bc5489ceb6bb13ff69d'},
 'training.loss_flow': {'path': 'training/loss_flow.py',
                        'is_package': False,
                        'sha256': '3d56e0dd9a3a5771eaad99501b2ff2f086188aff2a76b918a1a1713600481d91'},
 'training.observability': {'path': 'training/observability.py',
                            'is_package': False,
                            'sha256': 'df71ddc6a6d0d001665de21b902e804ac952a310d9a69f87d690e5eab8a463c7'},
 'training.pseudo_label_eval': {'path': 'training/pseudo_label_eval.py',
                                'is_package': False,
                                'sha256': 'dcd36e863633920337f13d1f416c61289c4bd107e0ab4fcf39074a356026a874'},
 'training.teachers': {'path': 'training/teachers.py',
                       'is_package': False,
                       'sha256': '7e0056287e8ac0ba02fa3876fe567b9c2b813d7da32fb4c4add7c778c21e4cc1'},
 'utils': {'path': None,
           'is_package': True,
           'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'},
 'utils.helper': {'path': 'utils/helper.py',
                  'is_package': False,
                  'sha256': '723419d015e13af5ceb5a12ee477489b840081063d41a053c7931101ad068987'}}

# Register one notebook cell as one importable Python module.
# Module bodies stay visible/editable Python; no quoted source dictionaries.
import hashlib
from IPython.core.magic import register_cell_magic

EMBEDDED_SOURCES = {
    name: dict(item, source="") for name, item in EXPECTED_SOURCE_MANIFEST.items()
    if item["path"] is None
}

@register_cell_magic
def flir_module(line, cell):
    name = line.strip()
    if name not in EXPECTED_SOURCE_MANIFEST:
        raise ValueError(f"Unknown FLIR module: {name}")
    item = EXPECTED_SOURCE_MANIFEST[name]
    # Preserve a canonical missing final newline when possible. IPython also
    # normalizes blank-line whitespace; record the source actually received.
    if cell.endswith("\n") and hashlib.sha256(cell[:-1].encode()).hexdigest() == item["sha256"]:
        cell = cell[:-1]
    filename = "/__flir_embedded__/" + (item["path"] or name.replace(".", "/") + "/__init__.py")
    compile(cell, filename, "exec")
    EMBEDDED_SOURCES[name] = dict(item, source=cell, sha256=hashlib.sha256(cell.encode()).hexdigest())

print("Module cells are ready. Run every %%flir_module cell, then the loader cell.")


In [ ]:
%%flir_module config
"""
Configuration dataclasses for Curriculum Domain Adaptation framework.

Training flow:  RGB → MID(SAGA) → IR
"""

from dataclasses import dataclass, field
import math
from typing import Optional, Tuple
from enum import Enum

from models.torchvision_fcos_adapter import ClassificationInitMode

class Phase(Enum):
    """Enumeration of Curriculum Learning stages."""
    PHASE1_RGB_WARMUP = 1
    PHASE2_TRANSITION = 2  # Đảm bảo tên này tồn tại
    PHASE3_ADAPTATION = 3  # Đảm bảo tên này tồn tại
    PHASE4_IR_FOCUS   = 4

@dataclass
class TeacherSchedule:
    """Holds (ratio, min_hm) pairs for each curriculum phase."""
    # Format: (ratio, min_hm)
    phase1: Tuple[float, float]
    phase2: Tuple[float, float]
    phase3: Tuple[float, float]
    phase4: Tuple[float, float]

    def __post_init__(self):
        for phase_name in ("phase1", "phase2", "phase3", "phase4"):
            ratio, min_hm = getattr(self, phase_name)
            if not math.isfinite(ratio) or not 0.0 < ratio <= 1.0:
                raise ValueError(f"{phase_name} top ratio must be in (0, 1]")
            if not math.isfinite(min_hm) or not 0.0 <= min_hm <= 1.0:
                raise ValueError(f"{phase_name} min_hm must be in [0, 1]")

    def get_params(self, phase: Phase) -> Tuple[float, float]:
        mapping = {
            Phase.PHASE1_RGB_WARMUP: self.phase1,
            Phase.PHASE2_TRANSITION:    self.phase2,
            Phase.PHASE3_ADAPTATION:     self.phase3,
            Phase.PHASE4_IR_FOCUS:   self.phase4,
        }
        return mapping.get(phase, self.phase2)
    
@dataclass
class DistillConfig:
    # Adaptive Threshold Schedules for RGB and IR Teachers
    # RGB Teacher: Learns faster, more reliable early on
    rgb_teacher: TeacherSchedule = field(default_factory=lambda: TeacherSchedule(
        phase1=(0.010, 0.45), phase2=(0.020, 0.35), 
        phase3=(0.040, 0.25), phase4=(0.060, 0.20)
    ))
    
    # IR Teacher: Noisier, requires stricter filtering in early phases
    ir_teacher: TeacherSchedule = field(default_factory=lambda: TeacherSchedule(
        phase1=(0.002, 0.60), phase2=(0.005, 0.50), 
        phase3=(0.015, 0.35), phase4=(0.030, 0.25)
    ))

    # Harmony Measure (HM) parameters: (prob^alpha) * (iou^beta)
    hm_alpha: float = 0.5
    hm_beta: float = 0.5
    
    # Uncertainty Weighting: weight = exp(-(1-HM) / un_regular_alpha)
    un_regular_alpha: float = 1.0

    def __post_init__(self):
        if self.hm_alpha < 0 or self.hm_beta < 0 or self.un_regular_alpha <= 0:
            raise ValueError("invalid HM/uncertainty settings")
    

@dataclass
class FCOSModelConfig:
    """Effective Torchvision FCOS model settings."""
    num_classes: int = 3
    class_names: Tuple[str, ...] = ("person", "car", "bicycle")
    weights: Optional[str] = "DEFAULT"
    classification_init_mode: ClassificationInitMode = ClassificationInitMode.COCO_TOWER
    pretrained_backbone: bool = True
    trainable_backbone_layers: int = 3
    min_sizes: Tuple[int, ...] = (640, 672, 704, 736, 768, 800)
    max_size: int = 1333
    center_sampling_radius: float = 0.0
    score_thresh: float = 0.05
    nms_thresh: float = 0.6
    topk_candidates: int = 1000
    detections_per_img: int = 100
    
    # HM-Focal Loss (VFL) Hyperparameters
    vfl_alpha: float = 0.75
    vfl_gamma: float = 2.0
    vfl_weight_type: str = "iou"

    def __post_init__(self):
        self.class_names = tuple(self.class_names)
        self.min_sizes = tuple(self.min_sizes)
        self.classification_init_mode = ClassificationInitMode(
            self.classification_init_mode
        )
        if self.num_classes != len(self.class_names):
            raise ValueError("num_classes must match class_names")
        if self.class_names != ("person", "car", "bicycle"):
            raise ValueError("class_names must be ('person', 'car', 'bicycle')")
        if not 0 <= self.trainable_backbone_layers <= 5:
            raise ValueError("trainable_backbone_layers must be in [0, 5]")
        if not self.min_sizes or any(size <= 0 for size in self.min_sizes):
            raise ValueError("min_sizes must contain positive values")
        if self.min_sizes != tuple(sorted(self.min_sizes)):
            raise ValueError("min_sizes must be increasing")
        if self.max_size < max(self.min_sizes):
            raise ValueError("max_size must be at least the largest min_size")
        if self.center_sampling_radius < 0:
            raise ValueError("center_sampling_radius must be non-negative")
        if not 0 <= self.score_thresh <= 1 or not 0 <= self.nms_thresh <= 1:
            raise ValueError("score_thresh and nms_thresh must be in [0, 1]")
        if self.topk_candidates <= 0 or self.detections_per_img <= 0:
            raise ValueError("detection top-k values must be positive")

@dataclass
class EMAConfig:
    """Exponential Moving Average settings for teacher update."""
    alpha: float = 0.996          # EMA decay factor (higher = slower teacher update)
    start_steps: Optional[int] = None

    def __post_init__(self):
        if not math.isfinite(self.alpha) or not 0.0 <= self.alpha <= 1.0:
            raise ValueError("EMA alpha must be in [0, 1]")
        if self.start_steps is not None and (
            not isinstance(self.start_steps, int) or self.start_steps < 0
        ):
            raise ValueError("EMA start_steps must be None or a non-negative integer")


@dataclass
class SoftSAGAConfig:
    """SoftSAGA alpha per MID level (1.0=pure RGB, 0.0=pure gray)."""
    alpha_near_rgb:     float = 0.70   # objects mostly RGB
    alpha_intermediate: float = 0.50   # half RGB, half gray
    alpha_near_ir:      float = 0.25   # objects mostly gray


@dataclass
class IntermediateDomainConfig:
    """Source appearance bridge; independent color alpha and texture strength."""

    mode: str = "randommono_btf"
    texture_strength: float = 0.25
    btf_radius: int = 2
    btf_iterations: int = 1

    def __post_init__(self):
        if self.mode not in {"randommono_btf", "saga"}:
            raise ValueError("intermediate domain must be randommono_btf or saga")
        if not math.isfinite(self.texture_strength) or not 0 <= self.texture_strength <= 1:
            raise ValueError("texture_strength must be finite and in [0, 1]")
        for name in ("btf_radius", "btf_iterations"):
            value = getattr(self, name)
            if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
                raise ValueError(f"{name} must be a positive integer")


@dataclass
class StepRouting:
    """
    Defines the behavior for a specific training step.
    """
    ema_target: str = "none"      # Which teacher to update: "rgb" | "ir" | "none"
    use_gt: bool = True           # Whether to include Supervised Ground Truth loss
    student_saga_level: str = "rgb"  # "rgb" | "weak" | "mid" | "high" | "ir"
    teacher_saga_level: str = "rgb"  # "rgb" | "weak" | "mid" | "high" | "ir"
    teacher_names: Tuple[str, ...] = ()

@dataclass
class MidRoutingConfig:
    """
    Routes the data flow and EMA updates based on the DomainStep 
    returned by the CurriculumScheduler.
    """
    # PHASE 1: Supervised Warmup
    p1_rgb_supervised: StepRouting = field(default_factory=lambda: StepRouting(
        ema_target="none", use_gt=True, student_saga_level="rgb", teacher_saga_level="rgb",
        teacher_names=(),
    ))

    # PHASE 2: Transition (RGB Dominant)
    p2_rgb_flow: StepRouting = field(default_factory=lambda: StepRouting(
        ema_target="rgb", use_gt=True, student_saga_level="weak", teacher_saga_level="rgb",
        teacher_names=("rgb",),
    ))
    p2_ir_flow: StepRouting = field(default_factory=lambda: StepRouting(
        ema_target="ir", use_gt=True, student_saga_level="high", teacher_saga_level="mid",
        teacher_names=("ir",),
    ))

    # PHASE 3: Adaptation (IR Dominant)
    p3_rgb_flow: StepRouting = field(default_factory=lambda: StepRouting(
        ema_target="rgb", use_gt=True, student_saga_level="mid", teacher_saga_level="weak",
        teacher_names=("rgb",),
    ))
    p3_ir_flow: StepRouting = field(default_factory=lambda: StepRouting(
        ema_target="ir", use_gt=False, student_saga_level="ir", teacher_saga_level="high",
        teacher_names=("ir",),
    ))

    # PHASE 4: supervised RGB steps retain source-domain supervision, while
    # the existing IR route continues distillation and IR-teacher EMA updates.
    p4_rgb_supervised: StepRouting = field(default_factory=lambda: StepRouting(
        ema_target="none", use_gt=True,
        student_saga_level="rgb", teacher_saga_level="rgb", teacher_names=(),
    ))

    p4_ir_focus: StepRouting = field(default_factory=lambda: StepRouting(
        ema_target="ir", use_gt=False, student_saga_level="ir", teacher_saga_level="ir",
        teacher_names=("ir",),
    ))

    def get_routing(self, step_name: str) -> StepRouting:
        """Helper to fetch routing params using the step name string."""
        return getattr(self, step_name, self.p4_ir_focus)


@dataclass
class AugConfig:
    """Augmentation config for student (applied on top of DataLoader transforms)."""
    # Geometric
    hflip_prob:              float = 0.5   # horizontal flip
    # Photometric
    blur_prob:               float = 0.5
    blur_sigma_max:          float = 1.0
    brightness_prob:         float = 0.3
    brightness_mag:          float = 0.2   # ±20% brightness
    contrast_prob:           float = 0.3
    contrast_mag:            float = 0.2   # ±20% contrast


@dataclass
class CurriculumConfig:
    """Phase boundaries and step ratios for alternating domain training."""
    phase1_end: int = 2000
    phase2_end: int = 5000
    phase3_end: int = 8000

    phase2_rgb_sampling_ratio: float = 0.7 
    phase3_rgb_sampling_ratio: float = 0.3  
    phase4_rgb_sampling_ratio: float = 0.2

    def __post_init__(self):
        boundaries = (self.phase1_end, self.phase2_end, self.phase3_end)
        if any(not isinstance(value, int) or value < 0 for value in boundaries):
            raise ValueError("phase boundaries must be non-negative integers")
        if tuple(boundaries) != tuple(sorted(boundaries)):
            raise ValueError("phase boundaries must be increasing")
        for name in (
            "phase2_rgb_sampling_ratio", "phase3_rgb_sampling_ratio",
            "phase4_rgb_sampling_ratio",
        ):
            value = getattr(self, name)
            if not 0.0 <= value <= 1.0:
                raise ValueError(f"{name} must be in [0, 1]")



@dataclass
class LossConfig:
    """
    Phase-based loss weights to control the balance between 
    Supervised (Ground Truth) and Unsupervised (Knowledge Distillation) learning.
    """
    weight_logits: float = 2.0
    weight_deltas: float = 1.0
    weight_quality: float = 1.0
    weight_uhl: float = 1.0

    # Phase 1: Pure Supervised Warmup (RGB + GT)
    p1_sup_weight: float = 1.0
    p1_distill_weight: float = 0.0

    # Phase 2: Transition (RGB/IR flows with GT + Distill)
    p2_sup_weight: float = 1.0
    p2_distill_weight: float = 0.3  # Start trusting teachers slightly less than GT

    # Phase 3: Adaptation (Shift weight towards Distill)
    p3_sup_weight: float = 0.8      # Reduce GT reliance (only used in RGB flow)
    p3_distill_weight: float = 1.0

    # Phase 4: GT applies only to RGB steps; unlabeled IR still uses KD only.
    p4_sup_weight: float = 1.0
    p4_distill_weight: float = 1.0

    def get_phase_weights(self, phase: Phase) -> Tuple[float, float]:
        """Helper to retrieve (supervised_weight, distillation_weight) for a given phase."""
        mapping = {
            Phase.PHASE1_RGB_WARMUP: (self.p1_sup_weight, self.p1_distill_weight),
            Phase.PHASE2_TRANSITION: (self.p2_sup_weight, self.p2_distill_weight),
            Phase.PHASE3_ADAPTATION: (self.p3_sup_weight, self.p3_distill_weight),
            Phase.PHASE4_IR_FOCUS:   (self.p4_sup_weight, self.p4_distill_weight),
        }
        return mapping.get(phase, (1.0, 1.0))


@dataclass
class OptimizerConfig:
    """SGD and iteration-based LR schedule derived from the D3T solver."""

    base_lr: Optional[float] = None
    reference_lr: float = 0.016
    reference_batch_size: int = 16
    momentum: float = 0.9
    weight_decay: float = 1e-4
    warmup_iters: Optional[int] = None
    warmup_factor: float = 1.0 / 1000.0
    milestones: Tuple[int, ...] = (70000,)
    gamma: float = 0.1

    def __post_init__(self):
        self.milestones = tuple(self.milestones)
        positive_values = (self.reference_lr, self.momentum, self.gamma)
        if any(not math.isfinite(value) or value <= 0 for value in positive_values):
            raise ValueError("reference_lr, momentum, and gamma must be positive")
        if self.base_lr is not None and (
            not math.isfinite(self.base_lr) or self.base_lr <= 0
        ):
            raise ValueError("base_lr must be None or positive")
        if self.reference_batch_size <= 0:
            raise ValueError("reference_batch_size must be positive")
        if not math.isfinite(self.weight_decay) or self.weight_decay < 0:
            raise ValueError("weight_decay must be non-negative")
        if self.warmup_iters is not None and self.warmup_iters < 0:
            raise ValueError("warmup_iters must be None or non-negative")
        if not math.isfinite(self.warmup_factor) or not 0 < self.warmup_factor <= 1:
            raise ValueError("warmup_factor must be in (0, 1]")
        if any(step < 0 for step in self.milestones):
            raise ValueError("LR milestones must be non-negative")
        if self.milestones != tuple(sorted(self.milestones)):
            raise ValueError("LR milestones must be increasing")


@dataclass
class DataConfig:
    root: str = "/home/duongtt/ws/DA/datasets/flir_data/align"

@dataclass
class TrainLoaderConfig:
    batch_size: int = 4
    num_workers: int = 4
    shuffle: bool = True
    drop_last: bool = True


@dataclass
class EvalLoaderConfig:
    batch_size: int = 4
    num_workers: int = 2
    shuffle: bool = False


@dataclass
class DataLoaderConfig:
    train: TrainLoaderConfig = field(default_factory=TrainLoaderConfig)
    eval: EvalLoaderConfig   = field(default_factory=EvalLoaderConfig)


# ---------------------------------------------------------------------------
# Top-level config
# ---------------------------------------------------------------------------

@dataclass
class PseudoLabelEvalConfig:
    """Audit cadence, fixed population, and scoring; these settings never control KD."""

    enabled: bool = False
    interval: int = 500
    iou_threshold: float = 0.5
    sample_size: int = 256
    subset_seed: int = 0

    def __post_init__(self):
        if isinstance(self.interval, bool) or not isinstance(self.interval, int) or self.interval <= 0:
            raise ValueError("pseudo-label evaluation interval must be a positive integer")
        if not math.isfinite(self.iou_threshold) or not 0. <= self.iou_threshold <= 1.:
            raise ValueError("pseudo-label evaluation IoU threshold must be in [0, 1]")
        if isinstance(self.sample_size, bool) or not isinstance(self.sample_size, int) or self.sample_size <= 0:
            raise ValueError("pseudo-label evaluation sample size must be a positive integer")
        if isinstance(self.subset_seed, bool) or not isinstance(self.subset_seed, int) or self.subset_seed < 0:
            raise ValueError("pseudo-label evaluation subset seed must be a non-negative integer")


@dataclass
class TrainingConfig:
    """Master config for CurriculumDomainAdaptationTrainer."""
    model: FCOSModelConfig = field(default_factory=FCOSModelConfig)
    distill: DistillConfig = field(default_factory=DistillConfig)

    ema: EMAConfig = field(default_factory=EMAConfig)
    soft_saga: SoftSAGAConfig = field(default_factory=SoftSAGAConfig)
    intermediate_domain: IntermediateDomainConfig = field(default_factory=IntermediateDomainConfig)
    mid_routing: MidRoutingConfig = field(default_factory=MidRoutingConfig)
    aug: AugConfig = field(default_factory=AugConfig)
    curriculum: CurriculumConfig = field(default_factory=CurriculumConfig)
    loss: LossConfig = field(default_factory=LossConfig)
    optimizer: OptimizerConfig = field(default_factory=OptimizerConfig)

    data: DataConfig = field(default_factory=DataConfig)
    loader: DataLoaderConfig = field(default_factory=DataLoaderConfig)
    pseudo_label_eval: PseudoLabelEvalConfig = field(default_factory=PseudoLabelEvalConfig)

    max_iter: int = 10000
    total_iters: Optional[int] = None
    grad_clip: float = 10.0
    device: str = "cuda"
    log_interval: int = 50
    output_dir: str = "outputs"
    workflow: str = "curriculum"
    teacher_mode: str = "two_teacher"
    wandb: bool = False       
    eval_period: int = 500       

    def __post_init__(self):
        if self.total_iters is not None:
            if self.total_iters <= 0:
                raise ValueError("total_iters must be positive")
            self.max_iter = self.total_iters
        else:
            self.total_iters = self.max_iter
        if self.max_iter <= 0:
            raise ValueError("max_iter must be positive")
        if self.grad_clip < 0 or self.log_interval <= 0 or self.eval_period <= 0:
            raise ValueError("training intervals and grad_clip are invalid")
        if self.workflow not in {"curriculum", "rgb_baseline"}:
            raise ValueError("workflow must be 'curriculum' or 'rgb_baseline'")
        if self.teacher_mode not in {"rgb", "ir", "two_teacher"}:
            raise ValueError("teacher_mode must be 'rgb', 'ir', or 'two_teacher'")
        if self.optimizer.base_lr is None:
            self.optimizer.base_lr = (
                self.optimizer.reference_lr
                * self.loader.train.batch_size
                / self.optimizer.reference_batch_size
            )
        if self.optimizer.warmup_iters is None:
            self.optimizer.warmup_iters = min(1000, self.curriculum.phase1_end)
        if self.ema.start_steps is None:
            self.ema.start_steps = self.curriculum.phase1_end

    def get_phase(self, global_step: int) -> Phase:
        """Determines the phase using CurriculumConfig boundaries."""
        cfg = self.curriculum
        if global_step < cfg.phase1_end:
            return Phase.PHASE1_RGB_WARMUP
        elif global_step < cfg.phase2_end:
            return Phase.PHASE2_TRANSITION
        elif global_step < cfg.phase3_end:
            return Phase.PHASE3_ADAPTATION
        else:
            return Phase.PHASE4_IR_FOCUS


In [ ]:
%%flir_module data
from .datasets import (
    FLIRRGBDataset,
    FLIRIRDataset,
    FLIRIRValDataset,
)

from .dataloaders import (
    build_dataloaders,
    build_rgb_train_dataloader,
    build_rgb_val_dataloader,
    build_ir_train_dataloader,
    build_ir_val_dataloader,
    ir_collate,
    ir_val_collate,
    rgb_collate,
)

from .preprocessing import (
    FLIR_CLASSES,
    NUM_CLASSES,
)

__all__ = [
    "FLIRRGBDataset",
    "FLIRIRDataset",
    "FLIRIRValDataset",

    "rgb_collate",
    "ir_collate",
    "ir_val_collate",

    "build_dataloaders",
    "build_rgb_train_dataloader",
    "build_rgb_val_dataloader",
    "build_ir_train_dataloader",
    "build_ir_val_dataloader",

    "FLIR_CLASSES",
    "NUM_CLASSES"
]

In [ ]:
%%flir_module data.augmentations
from torchvision import transforms as T
import torch
import random
import torchvision.transforms.functional as TF
from typing import List, Dict, Optional, Tuple

def default_rgb_transform() -> T.Compose:
    """ToTensor only — FCOS GeneralizedRCNNTransform handles ImageNet normalisation."""
    return T.Compose([T.ToTensor()])


def default_ir_transform() -> T.Compose:
    """IR (thermal JPEG): convert to 3-channel float [0,1]. FCOS expects 3-ch input."""
    return T.Compose([
        T.Grayscale(num_output_channels=3),
        T.ToTensor(),
    ])

class StudentAugmentor:
    """
    Handles standard augmentations for Student and Teacher models.
    Supports Batch Tensors [B, 3, H, W] and Torchvision-style targets.
    """
    def __init__(self, config):
        """
        Args:
            config: The TrainingConfig object containing 'aug' settings.
        """
        self.cfg = config.aug

    def apply_weak_aug(
        self, 
        images: torch.Tensor, 
        targets: Optional[List[Dict[str, torch.Tensor]]] = None
    ) -> Tuple[torch.Tensor, Optional[List[Dict]], bool]:
        """
        Applies a stochastic horizontal flip to the entire batch.
        
        Returns:
            aug_images: Flipped images.
            aug_targets: Updated bounding boxes (if targets provided).
            did_flip: Boolean flag indicating if the flip was performed.
        """
        W = images.shape[-1]
        did_flip = False
        
        if random.random() < self.cfg.hflip_prob:
            did_flip = True
            images = torch.flip(images, dims=[-1])
            
            if targets is not None:
                new_targets = []
                for t in targets:
                    boxes = t["boxes"] # Expected format: [N, 4] (xyxy)
                    if boxes.numel() > 0:
                        flipped_boxes = boxes.clone()
                        # x1_new = W - x2, x2_new = W - x1
                        flipped_boxes[:, 0] = W - boxes[:, 2]
                        flipped_boxes[:, 2] = W - boxes[:, 0]
                        new_targets.append({**t, "boxes": flipped_boxes})
                    else:
                        new_targets.append(t)
                targets = new_targets
                
        return images, targets, did_flip

    def apply_photometric_aug(self, images: torch.Tensor) -> torch.Tensor:
        """
        Applies blur, brightness, and contrast adjustments.
        This does NOT affect bounding box coordinates.
        """
        # 1. Gaussian Blur
        if random.random() < self.cfg.blur_prob:
            sigma = random.uniform(0.1, self.cfg.blur_sigma_max)
            # kernel_size must be odd
            images = TF.gaussian_blur(images, kernel_size=[3, 3], sigma=[sigma, sigma])

        # 2. Brightness
        if random.random() < self.cfg.brightness_prob:
            factor = 1.0 + random.uniform(-self.cfg.brightness_mag, self.cfg.brightness_mag)
            images = torch.clamp(images * factor, 0.0, 1.0)

        # 3. Contrast
        if random.random() < self.cfg.contrast_prob:
            mean = images.mean(dim=[-1, -2], keepdim=True)
            factor = 1.0 + random.uniform(-self.cfg.contrast_mag, self.cfg.contrast_mag)
            images = torch.clamp((images - mean) * factor + mean, 0.0, 1.0)

        return images

    def apply_strong_aug(
        self, 
        images: torch.Tensor, 
        targets: Optional[List[Dict]] = None
    ) -> Tuple[torch.Tensor, Optional[List[Dict]]]:
        """
        Strong Augmentation: Combined Geometric (Weak) + Photometric transformations.
        Used for the Student model.
        """
        images, targets, _ = self.apply_weak_aug(images, targets)
        images = self.apply_photometric_aug(images)
        
        return images, targets


In [ ]:
%%flir_module data.dataloaders
from typing import Dict, List, Tuple
import torch
from torch.utils.data import DataLoader

from .datasets import (
    FLIRRGBDataset,
    FLIRIRDataset,
    FLIRIRValDataset,
)


# ---------------------------------------------------------------------------
# Collate functions
# ---------------------------------------------------------------------------

def rgb_collate(batch: List) -> Tuple[torch.Tensor, List[Dict], Tuple[str, ...]]:
    return (
        torch.stack([b[0] for b in batch]),
        [b[1] for b in batch],
        tuple(b[2] for b in batch),
    )


def ir_collate(batch: List) -> Tuple[torch.Tensor, Tuple[str, ...]]:
    return torch.stack([b[0] for b in batch]), tuple(b[1] for b in batch)


def ir_val_collate(batch: List) -> Tuple[torch.Tensor, List[Dict], Tuple[str, ...]]:
    return (
        torch.stack([b[0] for b in batch]),
        [b[1] for b in batch],
        tuple(b[2] for b in batch),
    )


# ---------------------------------------------------------------------------
# Builders
# ---------------------------------------------------------------------------

def build_rgb_train_dataloader(config):
    dataset = FLIRRGBDataset(
        root=config.data.root,
        split="train",
    )

    return DataLoader(
        dataset,
        batch_size=config.loader.train.batch_size,
        shuffle=config.loader.train.shuffle,
        num_workers=config.loader.train.num_workers,
        collate_fn=rgb_collate,
        pin_memory=True,
        drop_last=config.loader.train.drop_last,
    )


def build_ir_train_dataloader(config):
    dataset = FLIRIRDataset(
        root=config.data.root,
        split="train",
    )

    return DataLoader(
        dataset,
        batch_size=config.loader.train.batch_size,
        shuffle=config.loader.train.shuffle,
        num_workers=config.loader.train.num_workers,
        collate_fn=ir_collate,
        pin_memory=True,
        drop_last=config.loader.train.drop_last,
    )


def build_rgb_val_dataloader(config):
    dataset = FLIRRGBDataset(
        root=config.data.root,
        split="validation",
    )

    return DataLoader(
        dataset,
        batch_size=config.loader.eval.batch_size,
        shuffle=config.loader.eval.shuffle,
        num_workers=config.loader.eval.num_workers,
        collate_fn=rgb_collate,
        pin_memory=True,
    )


def build_ir_val_dataloader(config):
    dataset = FLIRIRValDataset(
        root=config.data.root,
        split="validation",
    )

    return DataLoader(
        dataset,
        batch_size=config.loader.eval.batch_size,
        shuffle=config.loader.eval.shuffle,
        num_workers=config.loader.eval.num_workers,
        collate_fn=ir_val_collate,
        pin_memory=True,
    )


# ---------------------------------------------------------------------------
# Unified builder
# ---------------------------------------------------------------------------

def build_dataloaders(config) -> Dict[str, DataLoader]:
    return {
        "rgb_train": build_rgb_train_dataloader(config),
        "rgb_val":   build_rgb_val_dataloader(config),
        "ir_train":  build_ir_train_dataloader(config),
        "ir_val":    build_ir_val_dataloader(config),
    }


In [ ]:
%%flir_module data.datasets
"""
FLIR ADAS Aligned Dataset Loader — VOC XML format.

Actual dataset structure:
  align/
  ├── JPEGImages/
  │   ├── FLIR_XXXXX_PreviewData.jpeg   ← IR (thermal) image
  │   ├── FLIR_XXXXX_RGB.jpg            ← RGB image (spatially aligned with IR)
  │   └── ...
  ├── Annotations/
  │   ├── FLIR_XXXXX_PreviewData.xml    ← VOC XML annotation for IR image
  │   └── ...
  ├── align_train.txt                   ← stems: "FLIR_XXXXX_PreviewData"
  └── align_validation.txt

The split files may also use the VOC-compatible ImageSets/Main/ directory.

Key conventions:
  - stem  = "FLIR_XXXXX_PreviewData"
  - IR  image  : JPEGImages/{stem}.jpeg
  - RGB image  : JPEGImages/{stem.replace('_PreviewData','')}_RGB.jpg
  - Annotation : Annotations/{stem}.xml

Classes (FCOS 0-indexed, background excluded):
  person  → 0
  car     → 1
  bicycle → 2

Ignored labels: "FLIR" (source tag in XML), "dog" (too rare)

Domain adaptation roles:
  FLIRRGBDataset   → source domain  (labeled RGB, annotations from aligned IR XML)
  FLIRIRDataset    → target domain  (unlabeled IR, training only)
  FLIRIRValDataset → evaluation     (labeled IR, mAP computation)
"""

from pathlib import Path
from typing import Callable, Dict, Optional, Tuple
import torch
from PIL import Image
from torch.utils.data import Dataset
from utils.helper import ir_stem_to_rgb_filename, read_split_file
from data.augmentations import default_rgb_transform, default_ir_transform
from data.preprocessing import parse_voc_xml, objects_to_tensors


def _resolve_split_file(root: Path, split: str) -> Path:
    filename = f"align_{split}.txt"
    candidates = (
        root / filename,
        root / "ImageSets" / "Main" / filename,
    )
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    searched = ", ".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(
        f"FLIR split file '{filename}' was not found; searched: {searched}"
    )


# ---------------------------------------------------------------------------
# Datasets
# ---------------------------------------------------------------------------

class FLIRRGBDataset(Dataset):
    """
    Labeled RGB source domain.

    Loads: JPEGImages/FLIR_XXXXX_RGB.jpg
    Labels: parsed from Annotations/FLIR_XXXXX_PreviewData.xml
    (annotations are aligned → valid for paired RGB images)

    Args:
        root       : path to the `align/` directory
        split      : "train" or "validation"
        transform  : image transform (default: ToTensor)
        min_area   : skip boxes smaller than this (pixels²)
    """

    def __init__(
        self,
        root: str,
        split: str = "train",
        transform: Optional[Callable] = None,
        min_area: float = 16.0,
    ) -> None:
        self.root      = Path(root)
        self.transform = transform or default_rgb_transform()
        self.min_area  = min_area

        split_file = _resolve_split_file(self.root, split)
        self.stems = read_split_file(split_file)

        # Filter to stems where the RGB image actually exists
        self.stems = [
            s for s in self.stems
            if (self.root / "JPEGImages" / ir_stem_to_rgb_filename(s)).exists()
        ]

    def __len__(self) -> int:
        return len(self.stems)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict, str]:
        stem     = self.stems[idx]
        rgb_path = self.root / "JPEGImages" / ir_stem_to_rgb_filename(stem)
        ann_path = self.root / "Annotations" / f"{stem}.xml"

        img    = Image.open(rgb_path).convert("RGB")
        img_t  = self.transform(img)

        objects       = parse_voc_xml(ann_path)
        boxes, labels = objects_to_tensors(objects, self.min_area)

        target = {"boxes": boxes, "labels": labels, "stem": stem}
        return img_t, target, stem


class FLIRIRDataset(Dataset):
    """
    Unlabeled IR target domain — for training only.

    Loads: JPEGImages/FLIR_XXXXX_PreviewData.jpeg  (no labels)

    Args:
        root      : path to the `align/` directory
        split     : "train" or "validation"
        transform : image transform (default: Grayscale→3ch, ToTensor)
    """

    def __init__(
        self,
        root: str,
        split: str = "train",
        transform: Optional[Callable] = None,
    ) -> None:
        self.root      = Path(root)
        self.transform = transform or default_ir_transform()

        split_file = _resolve_split_file(self.root, split)
        all_stems  = read_split_file(split_file)

        self.ir_paths = [
            self.root / "JPEGImages" / f"{s}.jpeg"
            for s in all_stems
            if (self.root / "JPEGImages" / f"{s}.jpeg").exists()
        ]

    def __len__(self) -> int:
        return len(self.ir_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, str]:
        img   = Image.open(self.ir_paths[idx])
        img_t = self.transform(img)
        return img_t, self.ir_paths[idx].stem


class FLIRIRValDataset(Dataset):
    """
    Labeled IR target domain — for evaluation only.

    Loads: JPEGImages/FLIR_XXXXX_PreviewData.jpeg
    Labels: Annotations/FLIR_XXXXX_PreviewData.xml

    Args:
        root      : path to the `align/` directory
        split     : "train" or "validation"
        transform : image transform
        min_area  : skip boxes smaller than this (pixels²); defaults to 0 so
                    evaluation retains every valid annotation
    """

    def __init__(
        self,
        root: str,
        split: str = "validation",
        transform: Optional[Callable] = None,
        min_area: float = 0.0,
    ) -> None:
        self.root      = Path(root)
        self.transform = transform or default_ir_transform()
        self.min_area  = min_area

        split_file = _resolve_split_file(self.root, split)
        all_stems  = read_split_file(split_file)

        # Keep only stems where both image and annotation exist
        self.stems = [
            s for s in all_stems
            if (self.root / "JPEGImages"  / f"{s}.jpeg").exists()
            and (self.root / "Annotations" / f"{s}.xml").exists()
        ]

    def __len__(self) -> int:
        return len(self.stems)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict, str]:
        stem     = self.stems[idx]
        ir_path  = self.root / "JPEGImages"  / f"{stem}.jpeg"
        ann_path = self.root / "Annotations" / f"{stem}.xml"

        img    = Image.open(ir_path)
        img_t  = self.transform(img)

        objects       = parse_voc_xml(ann_path)
        boxes, labels = objects_to_tensors(objects, self.min_area)

        target = {"boxes": boxes, "labels": labels, "stem": stem}
        return img_t, target, stem
    



In [ ]:
%%flir_module data.ir_annotations
"""Read audit-only IR labels without changing training datasets or sampling."""

from dataclasses import dataclass
import math
from pathlib import Path
import xml.etree.ElementTree as ET

from data.preprocessing import FLIR_CLASS_TO_IDX, objects_to_tensors


@dataclass(frozen=True)
class IRAnnotation:
    """Missing/invalid labels are unknown; an available empty target is known."""

    status: str
    target: dict | None = None


class IRAnnotationReader:
    def __init__(self, data_root):
        # Share the dataset root and its existing VOC annotation layout.
        self.root = Path(data_root)

    def read(self, sample_ids, image_sizes, *, hflipped: bool):
        """Read by training sample identity and apply its already sampled flip."""
        if len(sample_ids) != len(image_sizes):
            raise ValueError("IR annotation IDs and image sizes must align")
        return tuple(self._read_one(stem, size, hflipped) for stem, size in
                     zip(sample_ids, image_sizes))

    def _read_one(self, stem, image_size, hflipped):
        # Reject unavailable/corrupt annotations instead of counting them as
        # background. Do not filter images out of the existing training loader.
        if not isinstance(stem, str) or Path(stem).name != stem or stem in {"", ".", ".."}:
            raise ValueError("IR annotation ID must be a file stem")
        path = self.root / "Annotations" / f"{stem}.xml"
        try:
            root = ET.parse(path).getroot()
        except FileNotFoundError:
            return IRAnnotation("missing")
        except (ET.ParseError, OSError):
            return IRAnnotation("invalid")
        height, width = image_size
        try:
            # FLIR exports use both <annotation> and <Annotation> roots.
            if root.tag.lower() != "annotation":
                raise ValueError("expected VOC annotation")
            size = root.find("size")
            if size is not None and (
                int(size.findtext("height")) != height
                or int(size.findtext("width")) != width
            ):
                raise ValueError("annotation and IR image sizes differ")
            objects = []
            for obj in root.findall("object"):
                name = obj.findtext("name", "").strip()
                if name not in FLIR_CLASS_TO_IDX:
                    continue
                box = obj.find("bndbox")
                coords = [float(box.findtext(key)) for key in ("xmin", "ymin", "xmax", "ymax")]
                x1, y1, x2, y2 = coords
                if not all(math.isfinite(v) for v in coords) or x2 <= x1 or y2 <= y1:
                    raise ValueError("invalid annotation box")
                objects.append({"label": name, "box": coords})
        except (TypeError, ValueError, AttributeError):
            return IRAnnotation("invalid")

        # Match IR evaluation's class mapping and retain small boxes. Geometry
        # follows the existing batch flip; no augmentation or RNG is run here.
        boxes, labels = objects_to_tensors(objects, min_area=0.)
        if hflipped:
            boxes[:, [0, 2]] = width - boxes[:, [2, 0]]
        return IRAnnotation("available", {"boxes": boxes, "labels": labels, "stem": stem})


In [ ]:
%%flir_module data.preprocessing
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple
import torch

FLIR_CLASSES    = ["person", "car", "bicycle"]
FLIR_CLASS_TO_IDX: Dict[str, int] = {c: i for i, c in enumerate(FLIR_CLASSES)}
NUM_CLASSES     = len(FLIR_CLASSES)   # 3

# Labels to silently skip (noise / artefacts in the XML)
_IGNORE_LABELS  = {"FLIR", "dog"}

def parse_voc_xml(xml_path: Path) -> List[Dict]:
    """
    Parse a VOC-format XML file and return a list of object dicts.

    Each dict:  {"label": str,  "box": [xmin, ymin, xmax, ymax]}
    """
    try:
        tree = ET.parse(xml_path)
    except ET.ParseError:
        return []

    root = tree.getroot()
    objects = []
    for obj in root.findall("object"):
        name = obj.findtext("name", default="").strip()
        if name in _IGNORE_LABELS or name not in FLIR_CLASS_TO_IDX:
            continue
        bnd = obj.find("bndbox")
        if bnd is None:
            continue
        try:
            xmin = float(bnd.findtext("xmin"))
            ymin = float(bnd.findtext("ymin"))
            xmax = float(bnd.findtext("xmax"))
            ymax = float(bnd.findtext("ymax"))
        except (TypeError, ValueError):
            continue
        if xmax > xmin and ymax > ymin:
            objects.append({"label": name, "box": [xmin, ymin, xmax, ymax]})
    return objects


def objects_to_tensors(
    objects: List[Dict],
    min_area: float = 16.0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Convert parsed objects to (boxes [N,4], labels [N]) tensors."""
    boxes, labels = [], []
    for obj in objects:
        x1, y1, x2, y2 = obj["box"]
        if (x2 - x1) * (y2 - y1) < min_area:
            continue
        boxes.append([x1, y1, x2, y2])
        labels.append(FLIR_CLASS_TO_IDX[obj["label"]])

    if boxes:
        return (
            torch.as_tensor(boxes,  dtype=torch.float32),
            torch.as_tensor(labels, dtype=torch.int64),
        )
    return (
        torch.zeros((0, 4), dtype=torch.float32),
        torch.zeros((0,),   dtype=torch.int64),
    )

In [ ]:
%%flir_module ema
"""
EMA (Exponential Moving Average) teacher update.

teacher_params ← alpha * teacher_params + (1 - alpha) * student_params

Warmup schedule: during early iterations alpha is ramped up so the teacher
does not diverge immediately from a random student.
"""

from typing import Optional

import torch
import torch.nn as nn


def ema_update(
    teacher: nn.Module,
    student: nn.Module,
    alpha: float = 0.9996,
    global_step: Optional[int] = None,
) -> None:
    """
    Update teacher parameters in-place with EMA from student.

    Args:
        teacher      : teacher model (requires_grad=False)
        student      : student model (being trained)
        alpha        : EMA decay. Higher = slower teacher change.
        global_step  : current iteration. If provided, applies warmup:
                       effective_alpha = min(alpha, (1 + step) / (10 + step))
                       This prevents the teacher from diverging too fast at
                       the very start when the student is still random.
    """
    if global_step is not None:
        # Warmup: alpha ramps from ~0.09 → target_alpha over first ~10k steps
        warmup_alpha = (1.0 + global_step) / (10.0 + global_step)
        alpha = min(alpha, warmup_alpha)

    with torch.no_grad():
        # Update learnable parameters
        t_params = dict(teacher.named_parameters())
        s_params = dict(student.named_parameters())

        for name, t_p in t_params.items():
            if name in s_params:
                t_p.data.mul_(alpha).add_(s_params[name].data, alpha=1.0 - alpha)

        # Update buffers (e.g. BatchNorm running_mean / running_var)
        t_bufs = dict(teacher.named_buffers())
        s_bufs = dict(student.named_buffers())

        for name, t_b in t_bufs.items():
            if name not in s_bufs:
                continue
            s_b = s_bufs[name]
            if t_b.is_floating_point():
                t_b.data.mul_(alpha).add_(s_b.data, alpha=1.0 - alpha)
            else:
                # Counters and other integral buffers have no meaningful
                # interpolation; follow the student exactly.
                t_b.data.copy_(s_b.data)


def copy_student_to_teacher(teacher: nn.Module, student: nn.Module) -> None:
    """Hard copy student weights into teacher (alpha=0). Used for initialization."""
    with torch.no_grad():
        for t_p, s_p in zip(teacher.parameters(), student.parameters()):
            t_p.data.copy_(s_p.data)
        for t_b, s_b in zip(teacher.buffers(), student.buffers()):
            t_b.data.copy_(s_b.data)


In [ ]:
%%flir_module evaluate_coco
"""Evaluate a D3T FCOS checkpoint with the official COCO detection metric."""

from __future__ import annotations

import argparse
import json
import logging
from pathlib import Path
from typing import Mapping, Sequence

import numpy as np
import torch
from torch.utils.data import DataLoader

from config import FCOSModelConfig, TrainingConfig
from data import (
    FLIRIRValDataset,
    FLIRRGBDataset,
    ir_val_collate,
    rgb_collate,
)
from models.fcos_factory import CHECKPOINT_SCHEMA_VERSION, build_fcos_d3t_model


logger = logging.getLogger(__name__)


def _xyxy_to_xywh(box: Sequence[float]) -> list[float]:
    x1, y1, x2, y2 = (float(value) for value in box)
    return [x1, y1, x2 - x1, y2 - y1]


def build_coco_inputs(predictions, targets, class_names: Sequence[str]):
    """Convert zero-based FLIR tensors into one-based COCO dictionaries."""

    if len(predictions) != len(targets):
        raise ValueError("predictions and targets must contain the same images")
    categories = [
        {"id": index + 1, "name": name}
        for index, name in enumerate(class_names)
    ]
    images = [{"id": index + 1} for index in range(len(targets))]
    annotations = []
    detections = []
    annotation_id = 1

    for image_index, (prediction, target) in enumerate(
        zip(predictions, targets), start=1
    ):
        for box, label in zip(target["boxes"], target["labels"]):
            category_id = int(label) + 1
            if not 1 <= category_id <= len(categories):
                raise ValueError("target label is outside the configured class range")
            xywh = _xyxy_to_xywh(box.tolist())
            annotations.append({
                "id": annotation_id,
                "image_id": image_index,
                "category_id": category_id,
                "bbox": xywh,
                "area": xywh[2] * xywh[3],
                "iscrowd": 0,
            })
            annotation_id += 1

        for box, score, label in zip(
            prediction["boxes"], prediction["scores"], prediction["labels"]
        ):
            category_id = int(label) + 1
            if not 1 <= category_id <= len(categories):
                raise ValueError("prediction label is outside the configured class range")
            detections.append({
                "image_id": image_index,
                "category_id": category_id,
                "bbox": _xyxy_to_xywh(box.tolist()),
                "score": float(score),
            })

    dataset = {
        "info": {},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }
    return dataset, detections


def _mean_valid(values: np.ndarray) -> float:
    valid = values[values > -1]
    return float(valid.mean()) if valid.size else float("nan")


def summarize_coco_eval(coco_eval, class_names: Sequence[str]) -> dict[str, float]:
    """Return standard COCO summary metrics plus AP for each category."""

    stat_names = (
        "AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
        "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large",
    )
    metrics = {
        name: float(coco_eval.stats[index])
        for index, name in enumerate(stat_names)
    }
    precision = coco_eval.eval["precision"]  # [IoU, recall, class, area, maxDet]
    area_index = list(coco_eval.params.areaRngLbl).index("all")
    max_det_index = len(coco_eval.params.maxDets) - 1
    iou_thresholds = np.asarray(coco_eval.params.iouThrs)

    for class_index, class_name in enumerate(class_names):
        class_precision = precision[:, :, class_index, area_index, max_det_index]
        metrics[f"AP/{class_name}"] = _mean_valid(class_precision)
        for threshold, label in ((0.5, "AP50"), (0.75, "AP75")):
            matches = np.flatnonzero(np.isclose(iou_thresholds, threshold))
            metrics[f"{label}/{class_name}"] = (
                _mean_valid(class_precision[matches[0]])
                if matches.size
                else float("nan")
            )
    return metrics


def _load_checkpoint_model(checkpoint_path: str, device: torch.device, score_thresh: float):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    metadata = checkpoint.get("metadata")
    if not isinstance(metadata, Mapping):
        raise ValueError("checkpoint has no D3T metadata")
    if metadata.get("schema_version") != CHECKPOINT_SCHEMA_VERSION:
        raise ValueError("checkpoint schema is not supported")
    saved_model = metadata.get("config", {}).get("model", {})
    class_names = tuple(metadata.get("class_names", ()))
    if not class_names or int(metadata.get("num_classes", 0)) != len(class_names):
        raise ValueError("checkpoint class metadata is invalid")

    model_config = FCOSModelConfig(
        num_classes=len(class_names),
        class_names=class_names,
        weights=None,
        classification_init_mode=metadata["classification_init_mode"],
        pretrained_backbone=False,
        trainable_backbone_layers=int(saved_model.get("trainable_backbone_layers", 3)),
        min_sizes=tuple(saved_model.get("min_sizes", (640, 672, 704, 736, 768, 800))),
        max_size=int(saved_model.get("max_size", 1333)),
        center_sampling_radius=float(saved_model.get("center_sampling_radius", 0.0)),
        score_thresh=score_thresh,
        nms_thresh=float(saved_model.get("nms_thresh", 0.6)),
        topk_candidates=int(saved_model.get("topk_candidates", 1000)),
        detections_per_img=100,
        vfl_alpha=float(saved_model.get("vfl_alpha", 0.75)),
        vfl_gamma=float(saved_model.get("vfl_gamma", 2.0)),
        vfl_weight_type=str(saved_model.get("vfl_weight_type", "iou")),
    )
    config = TrainingConfig(
        model=model_config,
        teacher_mode=str(metadata.get("teacher_mode", "two_teacher")),
        device=str(device),
    )
    model = build_fcos_d3t_model(config)
    model.load_state_dict(checkpoint["student"])
    model.to(device).eval()
    return model, class_names


def _build_loader(data_root: str, domain: str, batch_size: int, workers: int):
    if domain == "ir":
        dataset = FLIRIRValDataset(data_root, split="validation")
        collate_fn = ir_val_collate
    else:
        dataset = FLIRRGBDataset(data_root, split="validation")
        collate_fn = rgb_collate
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=workers,
        pin_memory=True,
        collate_fn=collate_fn,
    )


def _collect(model, loader, device: torch.device):
    predictions = []
    targets = []
    with torch.inference_mode():
        for batch_index, (images, batch_targets, sample_ids) in enumerate(loader, start=1):
            images = images.to(device, non_blocking=True)
            output = model(images, sample_ids=sample_ids)
            predictions.extend(
                {key: value.detach().cpu() for key, value in item.items()}
                for item in output
            )
            targets.extend(
                {
                    key: value.detach().cpu()
                    for key, value in item.items()
                    if isinstance(value, torch.Tensor)
                }
                for item in batch_targets
            )
            if batch_index % 50 == 0:
                logger.info("Evaluated %d/%d batches", batch_index, len(loader))
    return predictions, targets


def evaluate(args) -> dict[str, float]:
    try:
        from pycocotools.coco import COCO
        from pycocotools.cocoeval import COCOeval
    except ImportError as error:
        raise RuntimeError(
            "COCO evaluation requires pycocotools; install it with "
            "'pip install pycocotools' inside the training environment"
        ) from error

    device = torch.device(args.device)
    model, class_names = _load_checkpoint_model(
        args.checkpoint, device, args.score_thresh
    )
    loader = _build_loader(
        args.data_root, args.domain, args.batch_size, args.workers
    )
    logger.info(
        "COCO evaluation: checkpoint=%s domain=%s images=%d device=%s",
        args.checkpoint, args.domain.upper(), len(loader.dataset), device,
    )
    predictions, targets = _collect(model, loader, device)
    dataset, detections = build_coco_inputs(predictions, targets, class_names)

    coco_gt = COCO()
    coco_gt.dataset = dataset
    coco_gt.createIndex()
    if detections:
        coco_dt = coco_gt.loadRes(detections)
    else:
        coco_dt = COCO()
        coco_dt.dataset = {
            "images": dataset["images"],
            "categories": dataset["categories"],
            "annotations": [],
        }
        coco_dt.createIndex()
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.params.catIds = list(range(1, len(class_names) + 1))
    coco_eval.params.imgIds = list(range(1, len(targets) + 1))
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    metrics = summarize_coco_eval(coco_eval, class_names)

    print("\nPer-class COCO metrics")
    for class_name in class_names:
        print(
            f"{class_name:>10}: "
            f"AP={metrics[f'AP/{class_name}']:.4f}  "
            f"AP50={metrics[f'AP50/{class_name}']:.4f}  "
            f"AP75={metrics[f'AP75/{class_name}']:.4f}"
        )
    if args.output:
        output_path = Path(args.output)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(
            json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8"
        )
        logger.info("Metrics saved -> %s", output_path)
    return metrics


def parse_args(argv=None):
    parser = argparse.ArgumentParser(
        description="Evaluate a D3T FCOS checkpoint with official COCO bbox metrics"
    )
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--data-root", "--data_root", dest="data_root", required=True)
    parser.add_argument("--domain", choices=("ir", "rgb"), default="ir")
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--device", default="cuda")
    parser.add_argument(
        "--score-thresh",
        type=float,
        default=0.001,
        help="Low threshold preserves candidates needed by COCO AP ranking",
    )
    parser.add_argument("--output", default=None, help="Optional metrics JSON path")
    return parser.parse_args(argv)


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
    evaluate(parse_args())


In [ ]:
%%flir_module evaluator
"""Small dependency-free detection evaluator and phase callback."""

from collections import defaultdict
import copy
import logging
import time
from typing import Dict, List, Optional

import torch
from torch.utils.data import DataLoader
from torchvision.ops import box_iou

logger = logging.getLogger(__name__)


def _compute_ap_voc11(recalls: torch.Tensor, precisions: torch.Tensor) -> float:
    return sum(
        precisions[recalls >= threshold].max().item()
        if (recalls >= threshold).any()
        else 0.0
        for threshold in torch.linspace(0.0, 1.0, 11)
    ) / 11.0


def _compute_ap_auc(recalls: torch.Tensor, precisions: torch.Tensor) -> float:
    mrec = torch.cat([torch.tensor([0.0]), recalls, torch.tensor([1.0])])
    mpre = torch.cat([torch.tensor([0.0]), precisions, torch.tensor([0.0])])
    for index in range(len(mpre) - 2, -1, -1):
        mpre[index] = torch.maximum(mpre[index], mpre[index + 1])
    indices = torch.where(mrec[1:] != mrec[:-1])[0]
    return ((mrec[indices + 1] - mrec[indices]) * mpre[indices + 1]).sum().item()


def _compute_class_ap(
    pred_list: List[Dict],
    gt_list: List[Dict],
    iou_thresh: float = 0.5,
    interp: str = "voc11",
) -> float:
    num_gt = sum(len(item["boxes"]) for item in gt_list)
    if num_gt == 0:
        return float("nan")
    scores: List[float] = []
    true_positives: List[int] = []
    for prediction, target in zip(pred_list, gt_list):
        pred_boxes, pred_scores, gt_boxes = (
            prediction["boxes"], prediction["scores"], target["boxes"]
        )
        if len(pred_boxes) == 0:
            continue
        order = pred_scores.argsort(descending=True)
        matched = torch.zeros(len(gt_boxes), dtype=torch.bool)
        for index in order:
            scores.append(float(pred_scores[index]))
            if len(gt_boxes) == 0:
                true_positives.append(0)
                continue
            ious = box_iou(pred_boxes[index].unsqueeze(0), gt_boxes)[0]
            best_iou, best_index = ious.max(dim=0)
            is_true_positive = best_iou >= iou_thresh and not matched[best_index]
            true_positives.append(int(is_true_positive))
            if is_true_positive:
                matched[best_index] = True
    if not scores:
        return 0.0
    order = sorted(range(len(scores)), key=scores.__getitem__, reverse=True)
    tp = torch.tensor([true_positives[index] for index in order], dtype=torch.float32)
    cumulative_tp = tp.cumsum(0)
    cumulative_fp = (1.0 - tp).cumsum(0)
    recalls = cumulative_tp / num_gt
    precisions = cumulative_tp / (cumulative_tp + cumulative_fp + 1e-9)
    return (_compute_ap_voc11 if interp == "voc11" else _compute_ap_auc)(
        recalls, precisions
    )


class DetectionEvaluator:
    def __init__(
        self,
        num_classes: int,
        class_names: Optional[List[str]] = None,
        iou_thresholds: Optional[List[float]] = None,
        interp: str = "voc11",
    ) -> None:
        self.num_classes = num_classes
        self.class_names = class_names or [f"cls_{i}" for i in range(num_classes)]
        self.iou_thresholds = iou_thresholds or [0.5]
        self.interp = interp
        self.reset()

    def reset(self) -> None:
        self._preds: List[Dict] = []
        self._gts: List[Dict] = []

    def update(self, predictions, targets) -> None:
        if len(predictions) != len(targets):
            raise ValueError("predictions and targets must have equal batch length")
        for prediction, target in zip(predictions, targets):
            self._preds.append({
                key: value.detach().cpu()
                for key, value in prediction.items()
                if isinstance(value, torch.Tensor)
            })
            self._gts.append({
                key: value.detach().cpu()
                for key, value in target.items()
                if isinstance(value, torch.Tensor)
            })

    def compute(self) -> Dict[str, float]:
        if not self._preds:
            return {"mAP@0.5": 0.0}
        by_class_predictions = defaultdict(lambda: [{} for _ in self._preds])
        by_class_targets = defaultdict(lambda: [{} for _ in self._gts])
        for image_index, (prediction, target) in enumerate(zip(self._preds, self._gts)):
            for class_index in range(self.num_classes):
                pred_mask = prediction["labels"] == class_index
                target_mask = target["labels"] == class_index
                by_class_predictions[class_index][image_index] = {
                    "boxes": prediction["boxes"][pred_mask],
                    "scores": prediction["scores"][pred_mask],
                }
                by_class_targets[class_index][image_index] = {
                    "boxes": target["boxes"][target_mask]
                }
        results: Dict[str, float] = {}
        maps = []
        for threshold in self.iou_thresholds:
            class_aps = []
            for class_index in range(self.num_classes):
                ap = _compute_class_ap(
                    by_class_predictions[class_index],
                    by_class_targets[class_index],
                    threshold,
                    self.interp,
                )
                if ap == ap:
                    class_aps.append(ap)
                    if threshold == 0.5:
                        results[f"AP@0.5/{self.class_names[class_index]}"] = round(ap, 4)
            mean_ap = sum(class_aps) / len(class_aps) if class_aps else 0.0
            maps.append(mean_ap)
            if threshold == 0.5:
                results["mAP@0.5"] = round(mean_ap, 4)
        if len(self.iou_thresholds) > 1:
            results["mAP@0.5:0.95"] = round(sum(maps) / len(maps), 4)
        return results


class PhaseEvaluator:
    """Evaluate domains and invoke the best callback exactly on improvement."""

    def __init__(
        self,
        evaluator: DetectionEvaluator,
        ir_val_loader: Optional[DataLoader],
        device: torch.device,
        rgb_val_loader: Optional[DataLoader] = None,
        eval_every_n: Optional[int] = 500,
        vis_dir: Optional[str] = None,
    ) -> None:
        self.evaluator = evaluator
        self.ir_val_loader = ir_val_loader
        self.rgb_val_loader = rgb_val_loader
        self.device = device
        self.eval_every_n = eval_every_n
        self.vis_dir = vis_dir
        self.best_ir_map = -1.0
        self.best_rgb_map = -1.0
        self._last_phase = None
        self.history = []
        self.on_new_best_fn = None

    def register_best_fn(self, fn) -> None:
        self.on_new_best_fn = fn

    def state_dict(self) -> Dict:
        """Return persistent evaluation state without serializing callbacks/loaders."""

        return {
            "best_ir_map": float(self.best_ir_map),
            "best_rgb_map": float(self.best_rgb_map),
            "last_phase": self._last_phase,
            "history": copy.deepcopy(self.history),
        }

    def load_state_dict(self, state: Dict) -> None:
        """Restore evaluation progress used by periodic and best-model decisions."""

        if not isinstance(state, dict):
            raise TypeError("phase evaluator state must be a dictionary")
        self.best_ir_map = float(state.get("best_ir_map", -1.0))
        self.best_rgb_map = float(state.get("best_rgb_map", -1.0))
        self._last_phase = state.get("last_phase")
        self.history = copy.deepcopy(state.get("history", []))

    def step(self, model, global_step: int, current_phase) -> Optional[Dict]:
        phase_changed = self._last_phase is not None and current_phase != self._last_phase
        periodic = self.eval_every_n is not None and global_step > 0 and global_step % self.eval_every_n == 0
        self._last_phase = current_phase
        if phase_changed or periodic:
            return self.evaluate(model, global_step, current_phase)
        return None

    @staticmethod
    def _phase_name(phase) -> str:
        return phase.name if hasattr(phase, "name") else str(phase)

    def evaluate(self, model, global_step: int, current_phase, trigger_reason: str = "manual") -> Dict:
        was_training = model.training
        model.eval()
        phase_name = self._phase_name(current_phase)
        results = {"global_step": global_step, "phase": phase_name, "trigger": trigger_reason}
        try:
            if self.ir_val_loader is not None:
                ir_results = self._run_eval_on_loader(model, self.ir_val_loader, "IR")
            else:
                ir_results = {"mAP@0.5": 0.0}
            results.update(ir_results)
            if ir_results.get("mAP@0.5", 0.0) > self.best_ir_map:
                self.best_ir_map = ir_results["mAP@0.5"]
                results["is_best_ir"] = True
                if self.on_new_best_fn is not None:
                    self.on_new_best_fn(results)

            if self.rgb_val_loader is not None and phase_name in {
                "PHASE1_RGB_WARMUP", "PHASE2_TRANSITION"
            }:
                rgb_results = self._run_eval_on_loader(model, self.rgb_val_loader, "RGB")
                results.update({f"rgb_{key}": value for key, value in rgb_results.items()})
                if rgb_results.get("mAP@0.5", 0.0) > self.best_rgb_map:
                    self.best_rgb_map = rgb_results["mAP@0.5"]
                    results["is_best_rgb"] = True
            self.history.append(results)
            return results
        finally:
            model.train(was_training)

    @staticmethod
    def _split_batch(batch):
        if len(batch) == 3:
            return batch[0], batch[1], tuple(batch[2])
        if len(batch) == 2:
            return batch[0], batch[1], tuple(
                target.get("stem", str(index))
                for index, target in enumerate(batch[1])
            )
        raise ValueError("evaluation batch must contain images and targets")

    def _run_eval_on_loader(self, model, loader, domain_name) -> Dict:
        self.evaluator.reset()
        started = time.time()
        with torch.no_grad():
            for batch in loader:
                images, targets, sample_ids = self._split_batch(batch)
                images = images.to(self.device) if isinstance(images, torch.Tensor) else images
                targets = [
                    {key: value.to(self.device) if isinstance(value, torch.Tensor) else value
                     for key, value in target.items()}
                    for target in targets
                ]
                try:
                    predictions = model(images, sample_ids=sample_ids)
                except TypeError:
                    predictions = model(images)
                self.evaluator.update(predictions, targets)
        metrics = self.evaluator.compute()
        logger.info(
            "[%s Val] mAP@0.5: %.4f (%.1fs)",
            domain_name,
            metrics.get("mAP@0.5", 0.0),
            time.time() - started,
        )
        return metrics

    def print_history(self) -> None:
        for result in self.history:
            logger.info(
                "step=%s phase=%s mAP@0.5=%.4f",
                result.get("global_step"),
                result.get("phase"),
                result.get("mAP@0.5", 0.0),
            )


In [ ]:
%%flir_module example_flir
"""Command-line entrypoint for FLIR RGB baseline and D3T curriculum training."""

import argparse
import logging

import torch

from config import (
    AugConfig,
    CurriculumConfig,
    DataConfig,
    DataLoaderConfig,
    DistillConfig,
    EMAConfig,
    EvalLoaderConfig,
    FCOSModelConfig,
    LossConfig,
    IntermediateDomainConfig,
    OptimizerConfig,
    Phase,
    PseudoLabelEvalConfig,
    SoftSAGAConfig,
    TeacherSchedule,
    TrainLoaderConfig,
    TrainingConfig,
)
from data import FLIR_CLASSES, NUM_CLASSES
from models.torchvision_fcos_adapter import ClassificationInitMode
from training.bootstrap import build_training, move_model_trio_to_device
from training.observability import configure_training_logging


logger = logging.getLogger(__name__)


def _phase_boundaries(total_iters: int) -> tuple[int, int, int]:
    """Create monotonic shortened boundaries for both normal and smoke runs."""

    first = min(total_iters, max(1, round(total_iters * 0.20)))
    second = min(total_iters, max(first, round(total_iters * 0.50)))
    third = min(total_iters, max(second, round(total_iters * 0.80)))
    return first, second, third


def _weight_identifier(value: str):
    return None if value.strip().lower() in {"none", ""} else value


def build_training_config(args) -> TrainingConfig:
    """Map every training-relevant CLI argument into one effective config."""

    # Resolve iteration boundaries and weight identifiers before building the
    # effective configuration consumed by bootstrap and checkpoint metadata.
    total_iters = int(args.total_iters)
    default_boundaries = _phase_boundaries(total_iters)
    phase1_end = (
        default_boundaries[0] if args.phase1_end is None else args.phase1_end
    )
    phase2_end = (
        default_boundaries[1] if args.phase2_end is None else args.phase2_end
    )
    phase3_end = (
        default_boundaries[2] if args.phase3_end is None else args.phase3_end
    )
    if phase3_end > total_iters:
        raise ValueError("phase3_end must not exceed total_iters")
    weights = _weight_identifier(args.weights)
    pretrained_backbone = (
        bool(args.pretrained_backbone)
        if args.pretrained_backbone is not None
        else weights is not None
    )


    # Collect phase boundaries and route proportions for the curriculum scheduler.
    curriculum = CurriculumConfig(
        phase1_end=phase1_end,
        phase2_end=phase2_end,
        phase3_end=phase3_end,
        phase2_rgb_sampling_ratio=args.phase2_rgb_ratio,
        phase3_rgb_sampling_ratio=args.phase3_rgb_ratio,
        phase4_rgb_sampling_ratio=args.phase4_rgb_ratio,
    )

    return TrainingConfig(
        # Preserve baseline detector construction, transform, and inference settings.
        model=FCOSModelConfig(
            num_classes=NUM_CLASSES,
            class_names=tuple(FLIR_CLASSES),
            weights=weights,
            classification_init_mode=ClassificationInitMode(
                args.classification_init
            ),
            pretrained_backbone=pretrained_backbone,
            trainable_backbone_layers=args.trainable_backbone_layers,
            min_sizes=tuple(args.min_sizes),
            max_size=args.max_size,
            center_sampling_radius=args.center_sampling_radius,
            score_thresh=args.score_thresh,
            nms_thresh=args.nms_thresh,
            topk_candidates=args.topk_candidates,
            detections_per_img=args.detections_per_img,
            vfl_alpha=0.75,
            vfl_gamma=2.0,
        ),
        # Teacher-specific KD selection schedules and shared distillation settings.
        distill=DistillConfig(
            hm_alpha=args.hm_alpha,
            hm_beta=args.hm_beta,
            un_regular_alpha=args.un_regular_alpha,
            rgb_teacher=TeacherSchedule(
                phase1=(0.010, 0.45),
                phase2=(0.020, 0.35),
                phase3=(0.040, 0.25),
                phase4=(0.060, 0.20),
            ),
            ir_teacher=TeacherSchedule(
                phase1=(0.002, 0.60),
                phase2=(0.005, 0.50),
                phase3=(0.015, 0.35),
                phase4=(0.030, 0.25),
            ),
        ),
        curriculum=curriculum,
        # Phase and component weights are resolved upstream of loss execution.
        loss=LossConfig(
            p1_sup_weight=1.0,
            p1_distill_weight=0.0,
            p2_sup_weight=1.0,
            p2_distill_weight=0.7,
            p3_sup_weight=0.5,
            p3_distill_weight=1.0,
            p4_sup_weight=1.0,
            p4_distill_weight=1.0,
            weight_logits=4.0,
            weight_deltas=1.0,
            weight_quality=1.0,
            weight_uhl=1.0,
        ),
        # Keep update schedules and view-generation parameters in configuration;
        # executors receive the values selected by the trainer for each step.
        optimizer=OptimizerConfig(
            base_lr=args.lr,
            warmup_iters=args.warmup_iters,
            warmup_factor=args.warmup_factor,
            milestones=tuple(args.lr_steps),
            gamma=args.lr_gamma,
        ),
        ema=EMAConfig(alpha=args.ema_alpha, start_steps=args.ema_start),
        intermediate_domain=IntermediateDomainConfig(
            mode=args.intermediate_domain,
            texture_strength=args.texture_strength,
            btf_radius=args.btf_radius,
            btf_iterations=args.btf_iterations,
        ),
        soft_saga=SoftSAGAConfig(
            alpha_near_rgb=0.75,
            alpha_intermediate=0.5,
            alpha_near_ir=0.25,
        ),
        aug=AugConfig(
            hflip_prob=args.hflip_prob,
            blur_prob=args.blur_prob,
            brightness_prob=args.brightness_prob,
            contrast_prob=args.contrast_prob,
        ),
        # Dataset location and loader settings used by bootstrap's factories.
        data=DataConfig(root=args.data_root),
        loader=DataLoaderConfig(
            train=TrainLoaderConfig(
                batch_size=args.batch_size,
                num_workers=args.workers,
            ),
            eval=EvalLoaderConfig(
                batch_size=args.eval_batch_size,
                num_workers=args.workers,
            ),
        ),
        # Run lifecycle, output, workflow, and monitoring cadence settings.
        max_iter=total_iters,
        total_iters=total_iters,
        output_dir=args.output_dir,
        device=args.device,
        workflow=args.workflow,
        teacher_mode=args.teacher_mode,
        log_interval=args.log_interval,
        eval_period=args.eval_every,
        pseudo_label_eval=PseudoLabelEvalConfig(
            enabled=args.eval_pseudo_labels,
            interval=args.pseudo_label_eval_interval,
            iou_threshold=args.pseudo_label_iou_threshold,
            sample_size=args.pseudo_label_eval_samples,
            subset_seed=args.pseudo_label_eval_seed,
        ),
    )


# Backward-compatible name used by earlier local launch scripts.
make_training_config = build_training_config
# Preserve the helper name used by existing launch scripts and tests.
_move_model_trio_to_device = move_model_trio_to_device


def main(args) -> None:
    # Configure the experiment's existing log destinations before constructing
    # the dependencies that emit setup and progress messages.
    config = make_training_config(args)
    log_path = configure_training_logging(config.output_dir)
    logger.info("Training log -> %s", log_path)
    device = torch.device(config.device)
    logger.info(
        "FLIR D3T: workflow=%s teacher_mode=%s mode=%s weights=%s device=%s total_iters=%d",
        config.workflow,
        config.teacher_mode,
        config.model.classification_init_mode.value,
        config.model.weights or "none",
        device,
        config.total_iters,
    )

    logger.info("Intermediate domain: %s", config.intermediate_domain)
    if config.workflow == "curriculum" and config.intermediate_domain.mode == "randommono_btf" and config.intermediate_domain.texture_strength:
        from intermediate_domain import require_btf
        require_btf()

    # Bootstrap wires dependencies; this entrypoint retains the sequence of
    # resume/initial evaluation, training, interruption handling, and final save.
    trainer = build_training(config, logger=logger)
    student = trainer.student
    phase_evaluator = trainer.phase_evaluator

    # Resume saved progress or evaluate the initial model using the original
    # mutually exclusive paths and initial phase label.
    if args.resume:
        trainer.load_checkpoint(args.resume)
    else:
        logger.info("Running initial IR evaluation")
        phase_evaluator.evaluate(
            student,
            global_step=trainer.global_step,
            current_phase=Phase.PHASE1_RGB_WARMUP,
            trigger_reason="initial",
        )

    # Retain final persistence and history reporting on normal completion,
    # interruption, or an exception raised inside the training loop.
    try:
        trainer.train()
    except KeyboardInterrupt:
        logger.info("Training interrupted by user")
    finally:
        trainer.save_checkpoint("final_model.pth")
        phase_evaluator.print_history()


def parse_args(argv=None):
    # Declare launch options here; build_training_config translates them into
    # one effective configuration before any training dependency is assembled.
    parser = argparse.ArgumentParser(
        description="Curriculum D3T training for aligned FLIR RGB/IR data"
    )
    parser.add_argument("--data_root", required=True)
    parser.add_argument("--intermediate-domain", choices=("randommono_btf", "saga"), default="randommono_btf")
    parser.add_argument("--texture-strength", type=float, default=0.25)
    parser.add_argument("--btf-radius", type=int, default=2)
    parser.add_argument("--btf-iterations", type=int, default=1)
    parser.add_argument("--output_dir", default="./output_flir")
    parser.add_argument("--total_iters", type=int, default=10000)
    parser.add_argument(
        "--phase1-end", "--phase1_end", dest="phase1_end", type=int, default=None
    )
    parser.add_argument(
        "--phase2-end", "--phase2_end", dest="phase2_end", type=int, default=None
    )
    parser.add_argument(
        "--phase3-end", "--phase3_end", dest="phase3_end", type=int, default=None
    )
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--eval_batch_size", type=int, default=4)
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument(
        "--lr",
        type=float,
        default=None,
        help="Actual LR; defaults to D3T's 0.016 scaled from batch 16",
    )
    parser.add_argument("--warmup-iters", type=int, default=None)
    parser.add_argument("--warmup-factor", type=float, default=1.0 / 1000.0)
    parser.add_argument("--lr-steps", type=int, nargs="*", default=(70000,))
    parser.add_argument("--lr-gamma", type=float, default=0.1)
    parser.add_argument(
        "--min-sizes",
        type=int,
        nargs="+",
        default=(640, 672, 704, 736, 768, 800),
        help="Training short-edge choices; evaluation uses the final value",
    )
    parser.add_argument("--max_size", type=int, default=1333)
    parser.add_argument("--center_sampling_radius", type=float, default=0.0)
    parser.add_argument("--score_thresh", type=float, default=0.05)
    parser.add_argument("--nms_thresh", type=float, default=0.6)
    parser.add_argument("--topk_candidates", type=int, default=1000)
    parser.add_argument("--detections_per_img", type=int, default=100)
    parser.add_argument("--trainable_backbone_layers", type=int, default=3)
    parser.add_argument(
        "--weights",
        choices=("DEFAULT", "none"),
        default="DEFAULT",
        help="Torchvision FCOS COCO weights or none",
    )
    parser.add_argument(
        "--pretrained-backbone",
        dest="pretrained_backbone",
        action="store_true",
    )
    parser.add_argument(
        "--no-pretrained-backbone",
        dest="pretrained_backbone",
        action="store_false",
    )
    parser.set_defaults(pretrained_backbone=None)
    parser.add_argument(
        "--classification-init",
        dest="classification_init",
        choices=[mode.value for mode in ClassificationInitMode],
        default=ClassificationInitMode.COCO_TOWER.value,
    )
    parser.add_argument(
        "--workflow",
        choices=("curriculum", "rgb_baseline"),
        default="curriculum",
    )
    parser.add_argument(
        "--teacher-mode",
        choices=("rgb", "ir", "two_teacher"),
        default="two_teacher",
        help="Enable one teacher for staged smoke/ablation or both for curriculum",
    )
    parser.add_argument("--phase2-rgb-ratio", type=float, default=0.7)
    parser.add_argument("--phase3-rgb-ratio", type=float, default=0.3)
    parser.add_argument(
        "--phase4-rgb-ratio", type=float, default=0.2,
        help="Fraction of phase-4 steps using supervised RGB; use 0 for IR-only",
    )
    parser.add_argument("--hm-alpha", type=float, default=0.5)
    parser.add_argument("--hm-beta", type=float, default=0.5)
    parser.add_argument("--un-regular-alpha", type=float, default=1.0)
    parser.add_argument("--ema-alpha", type=float, default=0.9996)
    parser.add_argument(
        "--ema-start",
        type=int,
        default=None,
        help="EMA initialization step; defaults to the start of phase 2",
    )
    parser.add_argument("--hflip-prob", type=float, default=0.5)
    parser.add_argument("--blur-prob", type=float, default=0.5)
    parser.add_argument("--brightness-prob", type=float, default=0.3)
    parser.add_argument("--contrast-prob", type=float, default=0.3)
    parser.add_argument("--eval_every", type=int, default=500)
    # Audit labels are independent of supervised targets and student validation.
    parser.add_argument("--eval-pseudo-labels", action="store_true",
                        help="Compare selected KD rows with IR GT; never train on IR GT")
    parser.add_argument("--pseudo-label-eval-interval", type=int, default=500,
                        help="Global iteration interval after EMA teachers are initialized")
    parser.add_argument("--pseudo-label-iou-threshold", type=float, default=0.5)
    parser.add_argument("--pseudo-label-eval-samples", type=int, default=256,
                        help="Maximum labeled IR validation images in each periodic teacher audit")
    parser.add_argument("--pseudo-label-eval-seed", type=int, default=0,
                        help="Local seed for reproducible fixed IR validation image selection")
    parser.add_argument("--log_interval", type=int, default=50)
    parser.add_argument("--resume", default=None)
    parser.add_argument("--device", default="cuda")
    return parser.parse_args(argv)


if __name__ == "__main__":
    main(parse_args())


In [ ]:
%%flir_module intermediate_domain
"""Source-only RandomMono/BTF views, shared across teacher/student levels."""

from __future__ import annotations

import math
from typing import Callable, Sequence

import numpy as np
import torch

from config import IntermediateDomainConfig


def require_btf():
    """Resolve the optional real algorithm lazily; never substitute another blur."""
    try:
        import cv2
    except ImportError as error:
        raise RuntimeError("BTF requires opencv-contrib-python-headless; install it in the training environment") from error
    function = getattr(getattr(cv2, "ximgproc", None), "bilateralTextureFilter", None)
    if function is None:
        raise RuntimeError("BTF requires opencv-contrib-python-headless with ximgproc.bilateralTextureFilter; use one OpenCV distribution in the environment")
    return function


class IntermediateDomainAugmentation:
    """Generate one endpoint per image, then render multiple alpha views.

    Alpha=1 is RGB and consumes no bridge random numbers/backend work.
    Inputs may be a BCHW tensor or a sequence of CHW tensors. Outputs preserve
    that representation, geometry, device and dtype. BTF is CPU preprocessing,
    not a trainable generator; its result is detached from input autograd.
    """

    def __init__(self, config=None, *, texture_filter: Callable | None = None):
        self.config = config or IntermediateDomainConfig()
        self._texture_filter = texture_filter

    def _filter(self, mono: np.ndarray) -> np.ndarray:
        if self._texture_filter is not None:
            return self._texture_filter(mono)
        function = require_btf()
        return function(mono, fr=self.config.btf_radius,
                        numIter=self.config.btf_iterations,
                        sigmaAlpha=-1.0, sigmaAvg=-1.0)

    def make_views(self, images, alphas: Sequence[float], *, boxes=None):
        alphas = tuple(float(alpha) for alpha in alphas)
        if any(not math.isfinite(alpha) or not 0 <= alpha <= 1 for alpha in alphas):
            raise ValueError("bridge alphas must be finite values in [0, 1]")
        stacked = isinstance(images, torch.Tensor)
        if stacked and images.ndim != 4:
            raise ValueError("images must have shape [B, 3, H, W]")
        items = tuple(images)
        if not items or any(image.ndim != 3 or image.shape[0] != 3 or not image.is_floating_point() for image in items):
            raise ValueError("images must contain non-empty floating-point RGB tensors")
        if any(image.shape[1] == 0 or image.shape[2] == 0 for image in items):
            raise ValueError("image dimensions must be positive")
        if not alphas:
            return ()
        if all(alpha == 1 for alpha in alphas):
            return tuple(images.clone() if stacked else tuple(image.clone() for image in items) for _ in alphas)
        if self.config.mode == "saga":
            from saga import SoftSAGA
            if boxes is None or len(boxes) != len(items):
                raise ValueError("legacy SAGA requires boxes for each image")
            views = [tuple(SoftSAGA().apply(image, box, alpha) for image, box in zip(items, boxes)) for alpha in alphas]
        else:
            if any(not bool(torch.isfinite(image).all()) or bool((image < 0).any()) or bool((image > 1).any()) for image in items):
                raise ValueError("bridge inputs must be finite values in [0, 1] before detector normalization")
            weights = torch.rand(len(items), 3, 1, 1)
            sums = weights.sum(dim=1, keepdim=True)
            weights = torch.where(sums > 0, weights / sums.clamp_min(1e-12), torch.full_like(weights, 1 / 3))
            endpoints = []
            for image, weight in zip(items, weights):
                mono = (image.float() * weight.to(image.device)).sum(dim=0)
                if self.config.texture_strength:
                    source = np.ascontiguousarray(mono.detach().cpu().numpy(), dtype=np.float32)
                    filtered = np.asarray(self._filter(source), dtype=np.float32)
                    if filtered.shape != source.shape or not np.isfinite(filtered).all():
                        raise ValueError("BTF must return a finite intensity image of the original size")
                    endpoint = torch.from_numpy(source + self.config.texture_strength * (filtered - source))
                    endpoint = endpoint.to(device=image.device, dtype=image.dtype).clamp(0, 1)
                else:
                    endpoint = mono.to(image.dtype)
                endpoints.append(endpoint.unsqueeze(0).expand_as(image))
            views = [tuple(image.clone() if alpha == 1 else alpha * image + (1 - alpha) * endpoint
                           for image, endpoint in zip(items, endpoints)) for alpha in alphas]
        return tuple(torch.stack(view) if stacked else view for view in views)


In [ ]:
%%flir_module loss
"""Detector-neutral D3T loss exports."""

from .d3t_criterion import D3TLossCriterion

__all__ = ["D3TLossCriterion"]


In [ ]:
%%flir_module loss.d3t_criterion
"""Detector-neutral supervised and distillation losses for D3T."""

import math
from typing import Optional, Sequence

import torch
from torch import Tensor
import torch.nn.functional as F

from models.d3t_adapter import (
    CriterionResult,
    D3TCriterion,
    DistillationPair,
    DistillationSettings,
    Losses,
    SupervisedBatch,
)


def _hm_focal_elementwise(
    logits: Tensor,
    targets: Tensor,
    alpha: float,
    gamma: float,
    weight_type: str = "iou",
) -> Tensor:
    """Return unreduced HM/varifocal-style BCE terms."""

    if logits.shape != targets.shape:
        raise ValueError("HM-Focal logits and targets must have the same shape")
    probabilities = logits.sigmoid()
    targets = targets.to(dtype=logits.dtype)
    positive = targets > 0
    negative_weight = alpha * (probabilities - targets).abs().pow(gamma)
    if weight_type == "iou":
        positive_weight = targets
    elif weight_type == "hm":
        positive_weight = torch.sqrt(
            probabilities.detach().clamp_min(0) * targets.clamp_min(0)
        )
    elif weight_type == "hm_rev":
        positive_weight = 1.0 - torch.sqrt(
            probabilities.detach().clamp_min(0) * targets.clamp_min(0)
        )
    elif weight_type in {"binary", "ones"}:
        positive_weight = torch.ones_like(targets)
    else:
        raise ValueError(f"Unsupported HM-Focal weight_type: {weight_type}")
    focal_weight = torch.where(positive, positive_weight, negative_weight)
    return F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none"
    ) * focal_weight


def _iou_loss_xyxy(
    pred_boxes: Tensor, target_boxes: Tensor, eps: float = 1e-7
) -> Tensor:
    """Return unreduced aligned ``1 - IoU`` for decoded XYXY boxes."""

    if pred_boxes.shape != target_boxes.shape or pred_boxes.shape[-1] != 4:
        raise ValueError("pred_boxes and target_boxes must both have shape [N, 4]")
    top_left = torch.maximum(pred_boxes[..., :2], target_boxes[..., :2])
    bottom_right = torch.minimum(pred_boxes[..., 2:], target_boxes[..., 2:])
    intersection_wh = (bottom_right - top_left).clamp_min(0)
    intersection = intersection_wh[..., 0] * intersection_wh[..., 1]
    pred_wh = (pred_boxes[..., 2:] - pred_boxes[..., :2]).clamp_min(0)
    target_wh = (target_boxes[..., 2:] - target_boxes[..., :2]).clamp_min(0)
    pred_area = pred_wh[..., 0] * pred_wh[..., 1]
    target_area = target_wh[..., 0] * target_wh[..., 1]
    union = pred_area + target_area - intersection
    return 1.0 - intersection / union.clamp_min(eps)


def _giou_loss_xyxy(
    pred_boxes: Tensor, target_boxes: Tensor, eps: float = 1e-7
) -> Tensor:
    """Return unreduced aligned GIoU loss for decoded XYXY boxes."""

    if pred_boxes.shape != target_boxes.shape or pred_boxes.shape[-1] != 4:
        raise ValueError("pred_boxes and target_boxes must both have shape [N, 4]")
    top_left = torch.maximum(pred_boxes[..., :2], target_boxes[..., :2])
    bottom_right = torch.minimum(pred_boxes[..., 2:], target_boxes[..., 2:])
    intersection_wh = (bottom_right - top_left).clamp_min(0)
    intersection = intersection_wh[..., 0] * intersection_wh[..., 1]
    pred_wh = (pred_boxes[..., 2:] - pred_boxes[..., :2]).clamp_min(0)
    target_wh = (target_boxes[..., 2:] - target_boxes[..., :2]).clamp_min(0)
    pred_area = pred_wh[..., 0] * pred_wh[..., 1]
    target_area = target_wh[..., 0] * target_wh[..., 1]
    union = pred_area + target_area - intersection
    iou = intersection / union.clamp_min(eps)
    enclosing_top_left = torch.minimum(pred_boxes[..., :2], target_boxes[..., :2])
    enclosing_bottom_right = torch.maximum(pred_boxes[..., 2:], target_boxes[..., 2:])
    enclosing_wh = (enclosing_bottom_right - enclosing_top_left).clamp_min(0)
    enclosing_area = enclosing_wh[..., 0] * enclosing_wh[..., 1]
    giou = iou - (enclosing_area - union) / enclosing_area.clamp_min(eps)
    return 1.0 - giou


def _pairwise_iou_xyxy(boxes: Tensor, eps: float = 1e-7) -> Tensor:
    """Return the IoU matrix for decoded XYXY boxes from one image."""

    if boxes.ndim != 2 or boxes.shape[1] != 4:
        raise ValueError("boxes must have shape [N, 4]")
    top_left = torch.maximum(boxes[:, None, :2], boxes[None, :, :2])
    bottom_right = torch.minimum(boxes[:, None, 2:], boxes[None, :, 2:])
    intersection_wh = (bottom_right - top_left).clamp_min(0)
    intersection = intersection_wh[..., 0] * intersection_wh[..., 1]
    wh = (boxes[:, 2:] - boxes[:, :2]).clamp_min(0)
    area = wh[:, 0] * wh[:, 1]
    union = area[:, None] + area[None, :] - intersection
    return intersection / union.clamp_min(eps)


def _zero_with_grad(*tensors: Tensor) -> Tensor:
    """Create a scalar zero connected to every supplied computation graph."""

    connected = None
    for tensor in tensors:
        term = tensor.sum() * 0.0
        connected = term if connected is None else connected + term
    if connected is None:
        return torch.tensor(0.0)
    return connected


def _cat_or_none(tensors: Sequence[Tensor]) -> Optional[Tensor]:
    if not tensors:
        return None
    return torch.cat(tuple(tensors), dim=0)


@torch.no_grad()
def _select_distillation_rows(teacher_prob, teacher_quality_prob, settings):
    """Share the exact batch-global HM/top-k rule between KD and evaluation."""
    hm = teacher_prob.max(dim=1).values.pow(settings.hm_alpha)
    hm = hm * teacher_quality_prob.pow(settings.hm_beta)
    total_rows = teacher_prob.shape[0]
    count = min(total_rows, max(1, int(math.ceil(total_rows * settings.top_ratio))))
    top_indices = hm.topk(count, dim=0).indices
    top_mask = torch.zeros(total_rows, dtype=torch.bool, device=teacher_prob.device)
    top_mask[top_indices] = True
    return top_mask & (hm >= settings.min_hm), hm


@torch.no_grad()
def select_distillation_masks(predictions, settings):
    """Select teacher rows without student forward or computation of KD losses."""
    if not predictions:
        return ()
    selected, _ = _select_distillation_rows(
        torch.cat([p.class_logits for p in predictions]).sigmoid(),
        torch.cat([p.quality_logits for p in predictions]).sigmoid(), settings,
    )
    return tuple(selected.split([len(p.boxes) for p in predictions]))


class D3TLossCriterion(D3TCriterion):
    """HM-Focal supervised loss and position-wise D3T distillation."""

    def __init__(
        self,
        alpha: float = 0.75,
        gamma: float = 2.0,
        weight_type: str = "iou",
        kd_gamma: float = 2.0,
    ) -> None:
        super().__init__()
        if alpha < 0 or gamma < 0 or kd_gamma < 0:
            raise ValueError("loss exponents must be non-negative")
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.weight_type = weight_type
        self.kd_gamma = float(kd_gamma)

    @staticmethod
    def _supervised_tensors(batch: SupervisedBatch):
        predictions = batch.predictions
        targets = batch.targets
        if len(predictions) != len(targets):
            raise ValueError("supervised predictions and targets must be aligned")
        class_logits = _cat_or_none([p.class_logits for p in predictions])
        boxes = _cat_or_none([p.boxes for p in predictions])
        quality_logits = _cat_or_none([p.quality_logits for p in predictions])
        class_targets = _cat_or_none([t["class_targets"] for t in targets])
        box_targets = _cat_or_none([t["box_targets"] for t in targets])
        quality_targets = _cat_or_none([t["quality_targets"] for t in targets])
        foreground = _cat_or_none([t["foreground"] for t in targets])
        return (
            class_logits,
            boxes,
            quality_logits,
            class_targets,
            box_targets,
            quality_targets,
            foreground,
        )

    def supervised(self, batch: SupervisedBatch) -> Losses:
        (
            class_logits,
            boxes,
            quality_logits,
            class_targets,
            box_targets,
            quality_targets,
            foreground,
        ) = self._supervised_tensors(batch)
        if class_logits is None:
            zero = _zero_with_grad()
            return {"loss_cls": zero, "loss_box": zero, "loss_quality": zero}

        num_foreground = int(foreground.sum().item())
        denominator = max(1, num_foreground)
        cls_elementwise = _hm_focal_elementwise(
            class_logits,
            class_targets,
            alpha=self.alpha,
            gamma=self.gamma,
            weight_type=self.weight_type,
        )
        loss_cls = cls_elementwise.sum() / denominator
        if num_foreground:
            loss_box = _iou_loss_xyxy(
                boxes[foreground], box_targets[foreground]
            ).sum() / denominator
            loss_quality = F.binary_cross_entropy_with_logits(
                quality_logits[foreground],
                quality_targets[foreground].to(dtype=quality_logits.dtype),
                reduction="sum",
            ) / denominator
        else:
            loss_box = _zero_with_grad(boxes)
            loss_quality = _zero_with_grad(quality_logits)
        return {
            "loss_cls": loss_cls,
            "loss_box": loss_box,
            "loss_quality": loss_quality,
        }

    @staticmethod
    def _pair_tensors(pair: DistillationPair):
        student = pair.student
        teacher = pair.teacher
        if len(student) != len(teacher):
            raise ValueError("distillation pair image counts must match")
        s_cls = _cat_or_none([p.class_logits for p in student])
        t_cls = _cat_or_none([p.class_logits for p in teacher])
        s_boxes = _cat_or_none([p.boxes for p in student])
        t_boxes = _cat_or_none([p.boxes for p in teacher])
        s_quality = _cat_or_none([p.quality_logits for p in student])
        t_quality = _cat_or_none([p.quality_logits for p in teacher])
        return s_cls, t_cls, s_boxes, t_boxes, s_quality, t_quality

    def _uhl_loss(self, pair: DistillationPair, selected: Tensor) -> Tensor:
        """Build D3T overlap targets per image and supervise student logits."""

        logits = []
        targets = []
        offset = 0
        for student, teacher in zip(pair.student, pair.teacher):
            row_count = student.class_logits.shape[0]
            image_selected = selected[offset : offset + row_count]
            offset += row_count
            if int(image_selected.sum().item()) <= 1:
                continue

            selected_boxes = student.boxes[image_selected].detach()
            pairwise_iou = _pairwise_iou_xyxy(selected_boxes)
            pairwise_iou.fill_diagonal_(0.0)
            max_iou = pairwise_iou.max(dim=1).values
            overlapping = max_iou > 0
            if not overlapping.any():
                continue

            selected_logits = student.class_logits[image_selected]
            teacher_labels = (
                teacher.class_logits.detach().sigmoid().argmax(dim=1)[image_selected]
            )
            image_targets = torch.zeros_like(selected_logits)
            image_targets[overlapping, teacher_labels[overlapping]] = max_iou[
                overlapping
            ]
            logits.append(selected_logits[overlapping])
            targets.append(image_targets[overlapping])

        if not logits:
            return _zero_with_grad(*(p.class_logits for p in pair.student))
        return _hm_focal_elementwise(
            torch.cat(logits, dim=0),
            torch.cat(targets, dim=0),
            alpha=self.alpha,
            gamma=self.gamma,
            weight_type=self.weight_type,
        ).mean()

    @staticmethod
    def _metrics(
        device: torch.device,
        count: int,
        total: int,
        hm_sum: float,
    ):
        ratio = count / total if total else 0.0
        empty = float(count == 0)
        values = {
            "kd_selected_count": torch.tensor(float(count), device=device),
            "kd_selected_ratio": torch.tensor(ratio, device=device),
            "kd_hm_sum": torch.tensor(hm_sum, device=device),
            "kd_empty_mask": torch.tensor(empty, device=device),
        }
        # Short aliases make the metric contract convenient for standalone
        # callers while the prefixed names remain stable for trainer logging.
        values.update({
            "selected_count": values["kd_selected_count"],
            "selected_ratio": values["kd_selected_ratio"],
            "hm_sum": values["kd_hm_sum"],
            "empty_mask": values["kd_empty_mask"],
        })
        return values

    def distillation(
        self,
        pair: DistillationPair,
        settings: DistillationSettings,
    ):
        (
            student_cls,
            teacher_cls,
            student_boxes,
            teacher_boxes,
            student_quality,
            teacher_quality,
        ) = self._pair_tensors(pair)
        if student_cls is None:
            zero = _zero_with_grad()
            metrics = self._metrics(torch.device("cpu"), 0, 0, 0.0)
            return CriterionResult(
                losses={
                    "loss_kd_cls": zero,
                    "loss_kd_box": zero,
                    "loss_kd_quality": zero,
                    "loss_kd_uhl": zero,
                },
                metrics=metrics,
                selected_masks=(),
            )

        total_rows = student_cls.shape[0]
        if total_rows != teacher_cls.shape[0]:
            raise ValueError("distillation rows must match")
        device = student_cls.device
        with torch.no_grad():
            teacher_prob = teacher_cls.detach().sigmoid()
            teacher_quality_prob = teacher_quality.detach().sigmoid()
            selected, hm = _select_distillation_rows(teacher_prob, teacher_quality_prob, settings)
            selected_hm = hm[selected]
            hm_sum = float(selected_hm.sum().item())
            denominator = selected_hm.sum().clamp_min(1.0)
            reliability = torch.exp(
                -(1.0 - hm) / settings.uncertainty_alpha
            ).detach()

        if selected.any():
            student_prob = student_cls.sigmoid()
            cls_elementwise = F.binary_cross_entropy(
                student_prob[selected],
                teacher_prob[selected],
                reduction="none",
            ) * (
                student_prob[selected] - teacher_prob[selected]
            ).abs().pow(self.kd_gamma)
            cls_per_row = cls_elementwise.sum(dim=1)
            loss_kd_cls = (
                cls_per_row * reliability[selected]
            ).sum() / denominator

            box_per_row = _giou_loss_xyxy(
                student_boxes[selected], teacher_boxes[selected].detach()
            )
            loss_kd_box = (
                box_per_row * reliability[selected]
            ).sum() / denominator
            quality_per_row = F.binary_cross_entropy_with_logits(
                student_quality[selected],
                teacher_quality_prob[selected],
                reduction="none",
            )
            loss_kd_quality = (
                quality_per_row * reliability[selected]
            ).sum() / denominator
        else:
            loss_kd_cls = _zero_with_grad(student_cls)
            loss_kd_box = _zero_with_grad(student_boxes)
            loss_kd_quality = _zero_with_grad(student_quality)

        loss_kd_uhl = self._uhl_loss(pair, selected)

        return CriterionResult(
            losses={
                "loss_kd_cls": loss_kd_cls,
                "loss_kd_box": loss_kd_box,
                "loss_kd_quality": loss_kd_quality,
                "loss_kd_uhl": loss_kd_uhl,
            },
            metrics=self._metrics(device, int(selected.sum().item()), total_rows, hm_sum),
            # Reuse the mask above rather than recomputing top-k for diagnostics.
            # Splitting preserves batch-global selection and per-image identity.
            selected_masks=tuple(selected.detach().split(
                [p.class_logits.shape[0] for p in pair.teacher]
            )),
        )


__all__ = [
    "D3TLossCriterion",
    "select_distillation_masks",
    "_hm_focal_elementwise",
    "_iou_loss_xyxy",
    "_zero_with_grad",
]


In [ ]:
%%flir_module models.d3t_adapter
"""Detector-independent contracts used by the D3T training path.

Detector adapters own transforms, target assignment, decoding, postprocessing,
and the rules used to establish student/teacher correspondence.  The criterion
only receives the normalized tensors declared in this module.
"""

from abc import ABC, abstractmethod
from contextlib import contextmanager
from dataclasses import dataclass, field
import math
from typing import Any, Dict, Mapping, Optional, Sequence, Tuple

import torch
from torch import Tensor, nn

Targets = Sequence[Mapping[str, Any]]
Losses = Dict[str, Tensor]
SampleIds = Tuple[str, ...]


@dataclass(frozen=True)
class DistillationSettings:
    """Immutable settings for one teacher's position-wise KD pass."""

    top_ratio: float
    min_hm: float
    hm_alpha: float = 0.5
    hm_beta: float = 0.5
    uncertainty_alpha: float = 1.0

    def __post_init__(self) -> None:
        if not math.isfinite(self.top_ratio) or not 0.0 < self.top_ratio <= 1.0:
            raise ValueError("top_ratio must be in (0, 1]")
        if not math.isfinite(self.min_hm) or not 0.0 <= self.min_hm <= 1.0:
            raise ValueError("min_hm must be in [0, 1]")
        if (
            not math.isfinite(self.hm_alpha)
            or not math.isfinite(self.hm_beta)
            or not math.isfinite(self.uncertainty_alpha)
            or self.hm_alpha < 0
            or self.hm_beta < 0
            or self.uncertainty_alpha <= 0
        ):
            raise ValueError(
                "HM exponents must be non-negative and uncertainty_alpha positive"
            )


@dataclass(frozen=True)
class CriterionResult:
    """Losses plus detector-neutral metrics from a criterion call."""

    losses: Losses
    metrics: Mapping[str, Tensor] = field(default_factory=dict)
    # Optional read-only observations of the exact KD selection, split by image.
    # Consumers must never feed audit annotations back into these masks.
    selected_masks: Optional[Tuple[Tensor, ...]] = None


@dataclass(frozen=True)
class Predictions:
    """Predictions for one image in decoded XYXY pixel coordinates.

    ``class_logits`` contains foreground-only sigmoid logits.  ``boxes`` are
    differentiable decoded boxes in transformed-image coordinates and
    ``quality_logits`` predicts IoU, never centerness.
    """

    class_logits: Tensor
    boxes: Tensor
    quality_logits: Tensor

    def __post_init__(self) -> None:
        if self.class_logits.ndim != 2 or self.class_logits.shape[1] == 0:
            raise ValueError("class_logits must have shape [N, C] with C > 0")
        n = self.class_logits.shape[0]
        if self.boxes.shape != (n, 4):
            raise ValueError("boxes must have shape [N, 4] matching class_logits")
        if self.quality_logits.shape != (n,):
            raise ValueError("quality_logits must have shape [N]")
        tensors = (self.class_logits, self.boxes, self.quality_logits)
        if any(not tensor.is_floating_point() for tensor in tensors):
            raise ValueError("Predictions must be floating point tensors")
        if len({tensor.device for tensor in tensors}) != 1:
            raise ValueError("Predictions must share a device")
        if len({tensor.dtype for tensor in tensors}) != 1:
            raise ValueError("Predictions must share a dtype")

    def detached(self) -> "Predictions":
        return Predictions(
            self.class_logits.detach(),
            self.boxes.detach(),
            self.quality_logits.detach(),
        )


@dataclass(frozen=True)
class AdapterOutput:
    predictions: Tuple[Predictions, ...]
    # Private adapter data: transformed targets, features, anchors/proposals,
    # original/transformed image sizes, and stable sample IDs.
    context: Any = None
    sample_ids: Optional[SampleIds] = None


@dataclass(frozen=True)
class SupervisedBatch:
    """Loss-ready predictions and per-image aligned targets."""

    predictions: Tuple[Predictions, ...]
    targets: Targets
    native_losses: Losses = field(default_factory=dict)

    def __post_init__(self) -> None:
        if len(self.predictions) != len(self.targets):
            raise ValueError("Supervised targets must match the prediction batch")
        for prediction, target in zip(self.predictions, self.targets):
            expected_shapes = {
                "class_targets": prediction.class_logits.shape,
                "box_targets": prediction.boxes.shape,
                "quality_targets": prediction.quality_logits.shape,
                "foreground": prediction.quality_logits.shape,
            }
            for key, shape in expected_shapes.items():
                if key not in target or target[key].shape != shape:
                    raise ValueError(f"{key} must have shape {tuple(shape)}")
                if target[key].device != prediction.class_logits.device:
                    raise ValueError(f"{key} must share the prediction device")
                if target[key].requires_grad:
                    raise ValueError(f"{key} must be detached")
            if target["foreground"].dtype != torch.bool:
                raise ValueError("foreground must be a boolean mask")


@dataclass(frozen=True)
class DistillationPair:
    """Explicitly matched student and teacher predictions."""

    student: Tuple[Predictions, ...]
    teacher: Tuple[Predictions, ...]

    def __post_init__(self) -> None:
        if len(self.student) != len(self.teacher):
            raise ValueError("Distillation requires matched image batches")
        for student, teacher in zip(self.student, self.teacher):
            if student.class_logits.shape != teacher.class_logits.shape:
                raise ValueError("Distillation requires matched rows and classes")
            if student.boxes.shape != teacher.boxes.shape:
                raise ValueError("Distillation requires matched decoded boxes")
            if student.quality_logits.shape != teacher.quality_logits.shape:
                raise ValueError("Distillation requires matched quality rows")
            if student.class_logits.device != teacher.class_logits.device:
                raise ValueError("Distillation pairs must share a device")

    def detached_teacher(self) -> "DistillationPair":
        return DistillationPair(
            self.student,
            tuple(prediction.detached() for prediction in self.teacher),
        )


class DetectorAdapter(nn.Module, ABC):
    """Adapter seam shared by detector families."""

    def select_training_resize_short_edge(self) -> Optional[int]:
        """Choose one shared training geometry when the detector needs it."""

        return None

    @contextmanager
    def resize_short_edge(self, short_edge: Optional[int]):
        """Temporarily apply a shared resize; detector adapters may override."""

        yield

    @staticmethod
    def validate_sample_ids(
        sample_ids: Optional[Sequence[str]], batch_size: int
    ) -> Optional[SampleIds]:
        """Validate stable IDs when a caller supplies them.

        The optional form keeps the original small adapter contract usable for
        inference and legacy unit fixtures.  The FCOS adapter records IDs in
        its context and rejects an ambiguous pair before position-wise KD.
        """

        if sample_ids is None:
            return None
        ids = tuple(sample_ids)
        if len(ids) != batch_size:
            raise ValueError("sample_ids must contain one ID per image")
        if any(not isinstance(value, str) or not value.strip() for value in ids):
            raise ValueError("sample_ids must contain non-empty stable strings")
        return ids

    @abstractmethod
    def forward(
        self,
        images: Sequence[Tensor],
        targets: Optional[Targets] = None,
        sample_ids: Optional[SampleIds] = None,
    ) -> AdapterOutput:
        raise NotImplementedError

    @abstractmethod
    def prepare_supervised(
        self, output: AdapterOutput, targets: Targets
    ) -> SupervisedBatch:
        raise NotImplementedError

    @abstractmethod
    def prepare_distillation(
        self, student: AdapterOutput, teacher: AdapterOutput
    ) -> DistillationPair:
        raise NotImplementedError

    @abstractmethod
    def postprocess(
        self, output: AdapterOutput
    ) -> Sequence[Mapping[str, Tensor]]:
        raise NotImplementedError


class D3TCriterion(nn.Module, ABC):
    """Shared loss formulas receive prepared data, never detector internals."""

    @abstractmethod
    def supervised(self, batch: SupervisedBatch) -> Losses:
        raise NotImplementedError

    @abstractmethod
    def distillation(
        self, pair: DistillationPair, settings: DistillationSettings
    ) -> CriterionResult:
        raise NotImplementedError


In [ ]:
%%flir_module models.d3t_wrapper
"""Detector-independent D3T orchestration."""

from dataclasses import replace
from typing import Optional, Sequence, Tuple

import torch
from torch import Tensor, nn

from .d3t_adapter import (
    AdapterOutput,
    CriterionResult,
    D3TCriterion,
    DetectorAdapter,
    DistillationSettings,
    Losses,
    Targets,
)


_DEFAULT_DISTILLATION_SETTINGS = DistillationSettings(
    top_ratio=1.0,
    min_hm=0.0,
)


class D3TWrapper(nn.Module):
    """Combine one detector adapter with one detector-neutral criterion."""

    def __init__(self, adapter: DetectorAdapter, criterion: D3TCriterion):
        super().__init__()
        self.adapter = adapter
        self.criterion = criterion

    def select_training_resize_short_edge(self) -> Optional[int]:
        """Delegate detector-specific geometry selection to the adapter."""

        return self.adapter.select_training_resize_short_edge()

    @staticmethod
    def _normalize_images(images: Sequence[Tensor] | Tensor) -> Tuple[Tensor, ...]:
        if isinstance(images, Tensor):
            if images.ndim != 4:
                raise ValueError("batched images must have shape [B, C, H, W]")
            image_list = tuple(images.unbind(dim=0))
        else:
            image_list = tuple(images)
        if not image_list:
            raise ValueError("images must contain at least one image")
        for image in image_list:
            if not isinstance(image, Tensor) or image.ndim != 3:
                raise ValueError("each image must have shape [C, H, W]")
        return image_list

    @staticmethod
    def _ids_from_targets(
        targets: Optional[Targets], batch_size: int
    ) -> Optional[Tuple[str, ...]]:
        if targets is None or not all("stem" in target for target in targets):
            return None
        stems = tuple(target["stem"] for target in targets)
        if all(isinstance(stem, str) and stem.strip() for stem in stems):
            return stems
        return None

    @staticmethod
    def _criterion_result(result) -> CriterionResult:
        if isinstance(result, CriterionResult):
            return result
        if isinstance(result, dict):
            return CriterionResult(losses=result, metrics={})
        raise TypeError(
            "criterion must return CriterionResult (or a loss dict for compatibility)"
        )

    @staticmethod
    def _merge_losses(*loss_maps: Losses) -> Losses:
        merged: Losses = {}
        for losses in loss_maps:
            duplicates = merged.keys() & losses.keys()
            if duplicates:
                raise ValueError(
                    f"Duplicate native/D3T loss names: {sorted(duplicates)}"
                )
            merged.update(losses)
        return merged

    def raw(
        self,
        images: Sequence[Tensor] | Tensor,
        targets: Optional[Targets] = None,
        sample_ids: Optional[Sequence[str]] = None,
        resize_short_edge: Optional[int] = None,
    ) -> AdapterOutput:
        image_list = self._normalize_images(images)
        if targets is not None and len(targets) != len(image_list):
            raise ValueError("targets must match the image batch length")

        resolved_ids = sample_ids
        if resolved_ids is None:
            resolved_ids = self._ids_from_targets(targets, len(image_list))
        resolved_ids = self.adapter.validate_sample_ids(resolved_ids, len(image_list))

        if resize_short_edge is not None and resize_short_edge <= 0:
            raise ValueError("resize_short_edge must be positive")
        with self.adapter.resize_short_edge(resize_short_edge):
            output = self.adapter(
                image_list,
                targets,
                sample_ids=resolved_ids,
            )
        if len(output.predictions) != len(image_list):
            raise ValueError("Adapter output must match the image batch length")
        if output.sample_ids is None and resolved_ids is not None:
            output = replace(output, sample_ids=resolved_ids)
        elif output.sample_ids is not None:
            self.adapter.validate_sample_ids(output.sample_ids, len(image_list))
            if resolved_ids is not None and tuple(output.sample_ids) != tuple(resolved_ids):
                raise ValueError("adapter output sample IDs differ from the input batch")
        return output

    def supervised_from_output(
        self, output: AdapterOutput, targets: Targets
    ) -> Losses:
        if len(output.predictions) != len(targets):
            raise ValueError("targets must match the output batch length")
        batch = self.adapter.prepare_supervised(output, targets)
        result = self._criterion_result(self.criterion.supervised(batch))
        return self._merge_losses(result.losses, batch.native_losses)

    def distill_from_outputs(
        self,
        student_output: AdapterOutput,
        teacher_output: AdapterOutput,
        settings: Optional[DistillationSettings] = None,
    ) -> CriterionResult:
        pair = self.adapter.prepare_distillation(student_output, teacher_output)
        result = self._criterion_result(
            self.criterion.distillation(
                pair.detached_teacher(),
                settings or _DEFAULT_DISTILLATION_SETTINGS,
            )
        )
        return result

    def forward(
        self,
        images: Sequence[Tensor] | Tensor,
        targets: Optional[Targets] = None,
        sample_ids: Optional[Sequence[str]] = None,
    ):
        if self.training and targets is None:
            raise ValueError("Training requires targets; use raw() for unlabeled images")
        output = self.raw(images, targets, sample_ids)
        if not self.training:
            return self.adapter.postprocess(output)
        return self.supervised_from_output(output, targets)

    def distill(
        self,
        student_images: Sequence[Tensor] | Tensor,
        teacher: "D3TWrapper",
        teacher_images: Sequence[Tensor] | Tensor,
        settings: Optional[DistillationSettings] = None,
        sample_ids: Optional[Sequence[str]] = None,
        teacher_sample_ids: Optional[Sequence[str]] = None,
    ) -> Losses:
        """Convenience one-teacher call; the trainer owns routing and EMA."""

        if teacher.training or any(
            module.training for module in teacher.adapter.modules()
        ):
            raise ValueError("Teacher and its adapter must be in eval mode")
        student_list = self._normalize_images(student_images)
        teacher_list = self._normalize_images(teacher_images)
        if len(student_list) != len(teacher_list):
            raise ValueError("Distillation requires matching image batch lengths")

        resize_short_edge = self.select_training_resize_short_edge()
        student_output = self.raw(
            student_list,
            sample_ids=sample_ids,
            resize_short_edge=resize_short_edge,
        )
        with torch.no_grad():
            teacher_output = teacher.raw(
                teacher_list,
                sample_ids=teacher_sample_ids,
                resize_short_edge=resize_short_edge,
            )
        return self.distill_from_outputs(
            student_output,
            teacher_output,
            settings or _DEFAULT_DISTILLATION_SETTINGS,
        ).losses


In [ ]:
%%flir_module models.fcos_factory
"""Factories for the Torchvision FCOS D3T student/teacher models."""

from dataclasses import fields, is_dataclass
from enum import Enum
import copy
from typing import Any, Callable, Optional, Tuple

from torch import nn
from torchvision.models import ResNet50_Weights
from torchvision.models.detection import (
    FCOS_ResNet50_FPN_Weights,
    fcos_resnet50_fpn,
)

from config import FCOSModelConfig, TrainingConfig
from loss.d3t_criterion import D3TLossCriterion
from .d3t_wrapper import D3TWrapper
from .torchvision_fcos_adapter import (
    ClassificationInitMode,
    FCOSIoUHead,
    TorchvisionFCOSAdapter,
)


CHECKPOINT_SCHEMA_VERSION = 1


def _resolve_fcos_weights(value):
    if value is None:
        return None
    if isinstance(value, FCOS_ResNet50_FPN_Weights):
        return value
    if isinstance(value, str):
        normalized = value.strip().upper()
        if normalized in {"DEFAULT", "COCO_V1", "FCOS_RESNET50_FPN_COCO_V1"}:
            return FCOS_ResNet50_FPN_Weights.DEFAULT
        if normalized in {"NONE", ""}:
            return None
    raise ValueError(
        "weights must be None, 'DEFAULT', or a supported FCOS weight identifier"
    )


def _serialize(value):
    if isinstance(value, Enum):
        return value.value
    if is_dataclass(value):
        return {field.name: _serialize(getattr(value, field.name)) for field in fields(value)}
    if isinstance(value, tuple):
        return [_serialize(item) for item in value]
    if isinstance(value, list):
        return [_serialize(item) for item in value]
    if isinstance(value, dict):
        return {key: _serialize(item) for key, item in value.items()}
    return value


def serialize_effective_config(config: TrainingConfig) -> dict[str, Any]:
    return _serialize(config)


def build_checkpoint_metadata(config: TrainingConfig) -> dict[str, Any]:
    return {
        "schema_version": CHECKPOINT_SCHEMA_VERSION,
        "num_classes": config.model.num_classes,
        "class_names": list(config.model.class_names),
        "classification_init_mode": config.model.classification_init_mode.value,
        "teacher_mode": config.teacher_mode,
        "config": serialize_effective_config(config),
    }


def validate_checkpoint_metadata(
    metadata: dict[str, Any], config: TrainingConfig
) -> None:
    if metadata.get("schema_version") != CHECKPOINT_SCHEMA_VERSION:
        raise ValueError(
            "Unsupported checkpoint schema; expected "
            f"{CHECKPOINT_SCHEMA_VERSION}, got {metadata.get('schema_version')}"
        )
    if metadata.get("num_classes") != config.model.num_classes:
        raise ValueError("Checkpoint class count does not match the effective config")
    if tuple(metadata.get("class_names", ())) != tuple(config.model.class_names):
        raise ValueError("Checkpoint class order does not match the effective config")
    if metadata.get("classification_init_mode") != config.model.classification_init_mode.value:
        raise ValueError(
            "Checkpoint classification initialization mode does not match the effective config"
        )
    if metadata.get("teacher_mode") != config.teacher_mode:
        raise ValueError("Checkpoint teacher mode does not match the effective config")


def _make_base_detector(
    config: TrainingConfig,
    detector_builder: Callable[..., nn.Module],
) -> nn.Module:
    model_config: FCOSModelConfig = config.model
    weights = _resolve_fcos_weights(model_config.weights)
    if weights is not None:
        weights_backbone = None
        base_num_classes = len(weights.meta.get("categories", ())) or 91
    else:
        weights_backbone = (
            ResNet50_Weights.DEFAULT if model_config.pretrained_backbone else None
        )
        base_num_classes = 91
    return detector_builder(
        weights=weights,
        weights_backbone=weights_backbone,
        num_classes=base_num_classes,
        trainable_backbone_layers=model_config.trainable_backbone_layers,
        min_size=model_config.min_sizes,
        max_size=model_config.max_size,
        center_sampling_radius=model_config.center_sampling_radius,
        score_thresh=model_config.score_thresh,
        nms_thresh=model_config.nms_thresh,
        topk_candidates=model_config.topk_candidates,
        detections_per_img=model_config.detections_per_img,
    )


def build_fcos_d3t_model(
    config: TrainingConfig,
    *,
    base_detector: Optional[nn.Module] = None,
    detector_builder: Optional[Callable[..., nn.Module]] = None,
) -> D3TWrapper:
    """Build one student wrapper from the effective FCOS configuration."""

    if not isinstance(config, TrainingConfig):
        raise TypeError("config must be a TrainingConfig")
    if base_detector is None:
        base_detector = _make_base_detector(
            config, detector_builder or fcos_resnet50_fpn
        )
    if not hasattr(base_detector, "head"):
        raise TypeError("base_detector must expose a Torchvision FCOS head")
    base_detector.head = FCOSIoUHead(
        base_detector.head,
        num_classes=config.model.num_classes,
        classification_init_mode=config.model.classification_init_mode,
    )
    # The injected sentinel path is used by tests and still receives the same
    # effective transform/detection settings as a freshly constructed model.
    transform = getattr(base_detector, "transform", None)
    if transform is not None:
        transform.min_size = config.model.min_sizes
        transform.max_size = config.model.max_size
    for name in (
        "center_sampling_radius",
        "score_thresh",
        "nms_thresh",
        "topk_candidates",
        "detections_per_img",
    ):
        if hasattr(base_detector, name):
            setattr(base_detector, name, getattr(config.model, name))

    adapter = TorchvisionFCOSAdapter(
        base_detector,
        class_names=config.model.class_names,
    )
    criterion = D3TLossCriterion(
        alpha=config.model.vfl_alpha,
        gamma=config.model.vfl_gamma,
        weight_type=config.model.vfl_weight_type,
    )
    wrapper = D3TWrapper(adapter, criterion)
    wrapper.effective_config = config
    wrapper.classification_init_mode = config.model.classification_init_mode
    return wrapper


def build_fcos_d3t_trio(
    config: TrainingConfig,
    *,
    base_detector: Optional[nn.Module] = None,
    detector_builder: Optional[Callable[..., nn.Module]] = None,
) -> Tuple[D3TWrapper, Optional[D3TWrapper], Optional[D3TWrapper]]:
    """Build student plus independent RGB/IR teachers."""

    student = build_fcos_d3t_model(
        config,
        base_detector=base_detector,
        detector_builder=detector_builder,
    )
    rgb_teacher = (
        copy.deepcopy(student)
        if config.teacher_mode in {"rgb", "two_teacher"}
        else None
    )
    ir_teacher = (
        copy.deepcopy(student)
        if config.teacher_mode in {"ir", "two_teacher"}
        else None
    )
    for teacher in (rgb_teacher, ir_teacher):
        if teacher is None:
            continue
        teacher.eval()
        for parameter in teacher.parameters():
            parameter.requires_grad = False
    return student, rgb_teacher, ir_teacher


__all__ = [
    "CHECKPOINT_SCHEMA_VERSION",
    "ClassificationInitMode",
    "build_checkpoint_metadata",
    "build_fcos_d3t_model",
    "build_fcos_d3t_trio",
    "serialize_effective_config",
    "validate_checkpoint_metadata",
]


In [ ]:
%%flir_module models.torchvision_fcos_adapter
"""Torchvision FCOS adapter for detector-neutral D3T training.

Only this module knows Torchvision's FCOS head layout, anchor assignment, box
coder, and native detection postprocessing.  The public output is decoded XYXY
boxes plus foreground/quality logits.
"""

from dataclasses import dataclass
from enum import Enum
import copy
from collections import OrderedDict
from contextlib import contextmanager
from typing import Any, Mapping, Optional, Sequence, Tuple

import torch
from torch import Tensor, nn
import torch.nn.functional as F

from torchvision.models.detection.fcos import FCOSClassificationHead, FCOSHead
from torchvision.models.detection.transform import resize_boxes

from .d3t_adapter import (
    AdapterOutput,
    DetectorAdapter,
    DistillationPair,
    Predictions,
    SupervisedBatch,
    Targets,
)


class ClassificationInitMode(str, Enum):
    """Which classification tower is retained from the COCO FCOS model."""

    COCO_TOWER = "coco_tower"
    RANDOM_HEAD = "random_head"


class _RegressionHeadWithoutCenterness(nn.Module):
    """Torchvision regression tower and predictor without the old ctrness head."""

    def __init__(self, conv: nn.Module, bbox_reg: nn.Module, num_anchors: int):
        super().__init__()
        self.conv = conv
        self.bbox_reg = bbox_reg
        self.num_anchors = num_anchors

    def forward(self, features: Sequence[Tensor]) -> Tensor:
        outputs = []
        for feature in features:
            regression_feature = self.conv(feature)
            regression = F.relu(self.bbox_reg(regression_feature))
            batch, _, height, width = regression.shape
            regression = regression.view(
                batch, self.num_anchors, 4, height, width
            )
            regression = regression.permute(0, 3, 4, 1, 2).reshape(
                batch, -1, 4
            )
            outputs.append(regression)
        return torch.cat(outputs, dim=1)


def _new_quality_predictor(in_channels: int, num_anchors: int) -> nn.Conv2d:
    predictor = nn.Conv2d(
        in_channels, num_anchors, kernel_size=3, stride=1, padding=1
    )
    nn.init.normal_(predictor.weight, mean=0.0, std=0.01)
    nn.init.zeros_(predictor.bias)
    return predictor


def _flatten_level(output: Tensor, channels_per_anchor: int) -> Tensor:
    batch, _, height, width = output.shape
    output = output.view(
        batch, -1, channels_per_anchor, height, width
    )
    return output.permute(0, 3, 4, 1, 2).reshape(
        batch, -1, channels_per_anchor
    )


class FCOSIoUHead(nn.Module):
    """FCOS head with a three-class predictor and an IoU quality branch.

    The regression tower and bbox predictor are copied from ``base_head`` in
    both modes.  In ``COCO_TOWER`` the classification convolution tower is
    copied too; ``RANDOM_HEAD`` obtains a freshly initialized tower from a new
    Torchvision classification head.  Neither mode retains the COCO
    classification predictor or centerness predictor.
    """

    def __init__(
        self,
        base_head: FCOSHead,
        num_classes: int,
        classification_init_mode: ClassificationInitMode,
    ) -> None:
        super().__init__()
        if num_classes <= 0:
            raise ValueError("num_classes must be positive")
        if not isinstance(base_head, FCOSHead):
            raise TypeError("base_head must be a torchvision FCOSHead")
        mode = ClassificationInitMode(classification_init_mode)

        base_classification = base_head.classification_head
        base_regression = base_head.regression_head
        first_conv = next(
            (module for module in base_classification.conv if isinstance(module, nn.Conv2d)),
            None,
        )
        if first_conv is None:
            raise ValueError("FCOS classification head has no convolution")
        in_channels = first_conv.in_channels
        num_anchors = base_classification.num_anchors
        num_convs = sum(
            isinstance(module, nn.Conv2d)
            for module in base_classification.conv
        )

        if mode is ClassificationInitMode.COCO_TOWER:
            classification_head = copy.deepcopy(base_classification)
            classification_head.num_classes = num_classes
            classification_head.cls_logits = nn.Conv2d(
                in_channels,
                num_anchors * num_classes,
                kernel_size=3,
                stride=1,
                padding=1,
            )
            nn.init.normal_(classification_head.cls_logits.weight, std=0.01)
            prior_probability = 0.01
            nn.init.constant_(
                classification_head.cls_logits.bias,
                -torch.log(
                    torch.tensor((1.0 - prior_probability) / prior_probability)
                ).item(),
            )
        else:
            classification_head = FCOSClassificationHead(
                in_channels=in_channels,
                num_anchors=num_anchors,
                num_classes=num_classes,
                num_convs=num_convs,
            )

        self.classification_head = classification_head
        self.regression_head = _RegressionHeadWithoutCenterness(
            conv=copy.deepcopy(base_regression.conv),
            bbox_reg=copy.deepcopy(base_regression.bbox_reg),
            num_anchors=num_anchors,
        )
        self.quality_predictor = _new_quality_predictor(
            in_channels, num_anchors
        )
        self.classification_init_mode = mode
        self.num_classes = num_classes
        self.num_anchors = num_anchors

    @property
    def quality_logits(self) -> nn.Conv2d:
        """Alias retained for callers that name the output branch directly."""

        return self.quality_predictor

    @property
    def quality_head(self) -> nn.Conv2d:
        """Readable alias for the standalone IoU quality predictor."""

        return self.quality_predictor

    def forward(self, features: Sequence[Tensor]) -> dict[str, Tensor]:
        if not features:
            raise ValueError("FCOSIoUHead requires at least one feature level")
        cls_outputs = []
        quality_outputs = []
        for feature in features:
            cls_feature = self.classification_head.conv(feature)
            cls_outputs.append(
                _flatten_level(
                    self.classification_head.cls_logits(cls_feature),
                    self.num_classes,
                )
            )
            quality_outputs.append(
                _flatten_level(
                    self.quality_predictor(cls_feature),
                    1,
                )
            )

        return {
            "cls_logits": torch.cat(cls_outputs, dim=1),
            "bbox_regression": self.regression_head(features),
            "quality_logits": torch.cat(quality_outputs, dim=1),
        }


@dataclass(frozen=True)
class FCOSAdapterContext:
    transformed_images: Any
    transformed_targets: Optional[Tuple[Mapping[str, Any], ...]]
    original_image_sizes: Tuple[Tuple[int, int], ...]
    features: Tuple[Tensor, ...]
    anchors_per_image: Tuple[Tensor, ...]
    anchors_per_level: Tuple[Tuple[Tensor, ...], ...]
    num_anchors_per_level: Tuple[int, ...]
    raw_head_outputs: Mapping[str, Tensor]
    sample_ids: Optional[Tuple[str, ...]] = None
    class_names: Tuple[str, ...] = ("person", "car", "bicycle")

    @property
    def transformed_image_sizes(self) -> Tuple[Tuple[int, int], ...]:
        return tuple(tuple(size) for size in self.transformed_images.image_sizes)


def _aligned_box_iou(pred_boxes: Tensor, target_boxes: Tensor, eps: float = 1e-7) -> Tensor:
    if pred_boxes.shape != target_boxes.shape or pred_boxes.shape[-1] != 4:
        raise ValueError("aligned boxes must both have shape [N, 4]")
    top_left = torch.maximum(pred_boxes[:, :2], target_boxes[:, :2])
    bottom_right = torch.minimum(pred_boxes[:, 2:], target_boxes[:, 2:])
    intersection = (bottom_right - top_left).clamp_min(0)
    intersection_area = intersection[:, 0] * intersection[:, 1]
    pred_wh = (pred_boxes[:, 2:] - pred_boxes[:, :2]).clamp_min(0)
    target_wh = (target_boxes[:, 2:] - target_boxes[:, :2]).clamp_min(0)
    union = (
        pred_wh[:, 0] * pred_wh[:, 1]
        + target_wh[:, 0] * target_wh[:, 1]
        - intersection_area
    )
    return intersection_area / union.clamp_min(eps)


class TorchvisionFCOSAdapter(DetectorAdapter):
    """Run Torchvision FCOS once and expose D3T-compatible tensors."""

    def __init__(
        self,
        detector: nn.Module,
        class_names: Sequence[str] = ("person", "car", "bicycle"),
    ) -> None:
        super().__init__()
        required = (
            "transform",
            "backbone",
            "anchor_generator",
            "box_coder",
            "postprocess_detections",
        )
        missing = [name for name in required if not hasattr(detector, name)]
        if missing:
            raise TypeError(f"FCOS detector is missing required members: {missing}")
        if not isinstance(getattr(detector, "head", None), FCOSIoUHead):
            raise TypeError("detector.head must be an FCOSIoUHead")
        names = tuple(class_names)
        if len(names) != detector.head.num_classes:
            raise ValueError("class_names must match the FCOS predictor class count")
        if len(set(names)) != len(names):
            raise ValueError("class_names must be unique")

        self.detector = detector
        self.class_names = names
        self.num_classes = len(names)

    def select_training_resize_short_edge(self) -> int:
        """Choose once so student and EMA teachers use identical anchors."""

        return int(
            self.detector.transform.torch_choice(
                self.detector.transform.min_size
            )
        )

    @contextmanager
    def resize_short_edge(self, short_edge: Optional[int]):
        """Force one shared short edge while preserving the configured schedule."""

        if short_edge is None:
            yield
            return
        previous = self.detector.transform.min_size
        self.detector.transform.min_size = (int(short_edge),)
        try:
            yield
        finally:
            self.detector.transform.min_size = previous

    @staticmethod
    def _normalize_images(images: Sequence[Tensor] | Tensor) -> Tuple[Tensor, ...]:
        if isinstance(images, Tensor):
            if images.ndim != 4:
                raise ValueError("batched images must have shape [B, C, H, W]")
            images = tuple(images.unbind(dim=0))
        else:
            images = tuple(images)
        if not images:
            raise ValueError("images must contain at least one image")
        if any(not isinstance(image, Tensor) or image.ndim != 3 for image in images):
            raise ValueError("each image must have shape [C, H, W]")
        return images

    def _resolve_sample_ids(
        self,
        sample_ids: Optional[Sequence[str]],
        targets: Optional[Targets],
        batch_size: int,
    ) -> Optional[Tuple[str, ...]]:
        if sample_ids is None and targets is not None:
            stems = tuple(target.get("stem") for target in targets)
            if all(isinstance(stem, str) and stem.strip() for stem in stems):
                sample_ids = stems
        return self.validate_sample_ids(sample_ids, batch_size)

    def forward(
        self,
        images: Sequence[Tensor] | Tensor,
        targets: Optional[Targets] = None,
        sample_ids: Optional[Tuple[str, ...]] = None,
    ) -> AdapterOutput:
        image_list = self._normalize_images(images)
        target_list = None if targets is None else list(targets)
        if target_list is not None and len(target_list) != len(image_list):
            raise ValueError("targets must match the image batch length")
        resolved_ids = self._resolve_sample_ids(
            sample_ids, target_list, len(image_list)
        )

        original_sizes = tuple(
            (int(image.shape[-2]), int(image.shape[-1])) for image in image_list
        )
        transformed_images, transformed_targets = self.detector.transform(
            list(image_list), target_list
        )
        features = self.detector.backbone(transformed_images.tensors)
        if isinstance(features, Tensor):
            features = OrderedDict([("0", features)])
        feature_list = tuple(features.values())
        head_outputs = self.detector.head(feature_list)
        expected_keys = {"cls_logits", "bbox_regression", "quality_logits"}
        if set(head_outputs) != expected_keys:
            raise ValueError(
                f"FCOSIoUHead output keys must be {sorted(expected_keys)}"
            )

        anchors_list = tuple(
            self.detector.anchor_generator(transformed_images, list(feature_list))
        )
        anchors_per_location = tuple(
            self.detector.anchor_generator.num_anchors_per_location()
        )
        if len(anchors_per_location) != len(feature_list):
            raise ValueError("anchor generator levels do not match FPN levels")
        if any(value != 1 for value in anchors_per_location):
            raise ValueError("FCOS D3T adapter requires one anchor per location")
        num_anchors_per_level = tuple(
            int(feature.shape[-2] * feature.shape[-1] * anchors_per_location[level])
            for level, feature in enumerate(feature_list)
        )
        total_anchors = sum(num_anchors_per_level)
        for image_anchors in anchors_list:
            if image_anchors.shape != (total_anchors, 4):
                raise ValueError("anchor/head layout mismatch in FCOS adapter")

        cls_logits = head_outputs["cls_logits"]
        bbox_regression = head_outputs["bbox_regression"]
        quality_logits = head_outputs["quality_logits"]
        if cls_logits.shape[1] != total_anchors:
            raise ValueError("classification/head anchor row count mismatch")
        if bbox_regression.shape[:2] != cls_logits.shape[:2]:
            raise ValueError("regression/head anchor row count mismatch")
        if quality_logits.shape[:2] != cls_logits.shape[:2]:
            raise ValueError("quality/head anchor row count mismatch")

        predictions = []
        anchors_per_level = []
        for image_index, anchors in enumerate(anchors_list):
            levels = tuple(anchors.split(num_anchors_per_level, dim=0))
            anchors_per_level.append(levels)
            decoded_boxes = self.detector.box_coder.decode(
                bbox_regression[image_index], anchors
            )
            predictions.append(
                Predictions(
                    class_logits=cls_logits[image_index],
                    boxes=decoded_boxes,
                    quality_logits=quality_logits[image_index].squeeze(-1),
                )
            )

        context = FCOSAdapterContext(
            transformed_images=transformed_images,
            transformed_targets=(
                None
                if transformed_targets is None
                else tuple(transformed_targets)
            ),
            original_image_sizes=original_sizes,
            features=feature_list,
            anchors_per_image=anchors_list,
            anchors_per_level=tuple(anchors_per_level),
            num_anchors_per_level=num_anchors_per_level,
            raw_head_outputs=head_outputs,
            sample_ids=resolved_ids,
            class_names=self.class_names,
        )
        return AdapterOutput(
            tuple(predictions), context=context, sample_ids=resolved_ids
        )

    def _match_anchors_to_targets(
        self,
        anchors: Tensor,
        target: Mapping[str, Any],
        num_anchors_per_level: Sequence[int],
    ) -> Tensor:
        """Assign FCOS locations with D3T-compatible center sampling."""

        if anchors.ndim != 2 or anchors.shape[-1] != 4:
            raise ValueError("anchors must have shape [N, 4]")
        if sum(num_anchors_per_level) != anchors.shape[0]:
            raise ValueError("anchor levels do not cover all anchors")
        gt_boxes = target["boxes"].to(device=anchors.device, dtype=anchors.dtype)
        if gt_boxes.ndim != 2 or gt_boxes.shape[-1] != 4:
            raise ValueError("target boxes must have shape [M, 4]")
        if gt_boxes.numel() == 0:
            return torch.full(
                (anchors.shape[0],),
                -1,
                dtype=torch.int64,
                device=anchors.device,
            )

        gt_centers = (gt_boxes[:, :2] + gt_boxes[:, 2:]) / 2
        anchor_centers = (anchors[:, :2] + anchors[:, 2:]) / 2
        anchor_sizes = anchors[:, 2] - anchors[:, 0]
        radius = float(self.detector.center_sampling_radius)

        x, y = anchor_centers.unsqueeze(dim=2).unbind(dim=1)
        x0, y0, x1, y1 = gt_boxes.unsqueeze(dim=0).unbind(dim=2)
        pairwise_dist = torch.stack(
            [x - x0, y - y0, x1 - x, y1 - y], dim=2
        )
        inside_match = pairwise_dist.min(dim=2).values > 0

        lower_bound = anchor_sizes * 4
        lower_bound = lower_bound.clone()
        lower_bound[: num_anchors_per_level[0]] = 0
        upper_bound = anchor_sizes * 8
        upper_bound = upper_bound.clone()
        upper_bound[-num_anchors_per_level[-1] :] = float("inf")
        max_regression = pairwise_dist.max(dim=2).values
        level_match = (
            (max_regression >= lower_bound[:, None])
            & (max_regression <= upper_bound[:, None])
        )

        candidate = inside_match & level_match
        if radius > 0:
            center_match = (
                anchor_centers[:, None, :] - gt_centers[None, :, :]
            ).abs().amax(dim=2) < radius * anchor_sizes[:, None]
            candidate &= center_match
        gt_areas = (
            (gt_boxes[:, 2] - gt_boxes[:, 0])
            * (gt_boxes[:, 3] - gt_boxes[:, 1])
        )
        scores = candidate.to(torch.float32) * (1e8 - gt_areas[None, :])
        max_values, matched = scores.max(dim=1)
        matched[max_values < 1e-5] = -1
        return matched.to(torch.int64)

    @torch.no_grad()
    def assign_diagnostic_targets(self, output: AdapterOutput, targets):
        """Align audit-only GT to existing FCOS geometry without another forward."""
        context = output.context
        if not isinstance(context, FCOSAdapterContext):
            raise ValueError("FCOS diagnostics require FCOS context")
        if len(targets) != len(output.predictions):
            raise ValueError("diagnostic targets must align with output images")
        assigned = []
        # Use sizes from this teacher forward, not a new transform that could
        # choose a different resize. Never modify its context or training GT.
        for target, prediction, anchors, original_size, resized_size in zip(
            targets, output.predictions, context.anchors_per_image,
            context.original_image_sizes, context.transformed_image_sizes,
        ):
            if target is None:
                assigned.append(None)
                continue
            boxes = resize_boxes(target["boxes"].to(prediction.boxes), original_size, resized_size)
            labels = target["labels"].to(device=boxes.device, dtype=torch.long)
            if labels.shape != (len(boxes),) or bool(((labels < 0) | (labels >= self.num_classes)).any()):
                raise ValueError("diagnostic labels must match FCOS classes and boxes")
            aligned = {"boxes": boxes, "labels": labels}
            # Share the detector's existing location/scale/center assignment.
            # Matching GT for observation never changes the criterion's KD mask.
            aligned["matched_idxs"] = self._match_anchors_to_targets(
                anchors, aligned, context.num_anchors_per_level
            )
            assigned.append(aligned)
        return tuple(assigned)

    def prepare_supervised(
        self, output: AdapterOutput, targets: Targets
    ) -> SupervisedBatch:
        if not isinstance(output.context, FCOSAdapterContext):
            raise ValueError("FCOS supervised preparation requires FCOS context")
        context = output.context
        if context.transformed_targets is None:
            raise ValueError(
                "raw output was created without targets; run raw(images, targets)"
            )
        if len(targets) != len(output.predictions):
            raise ValueError("targets must match the output batch length")

        prepared_targets = []
        for prediction, transformed_target, anchors in zip(
            output.predictions,
            context.transformed_targets,
            context.anchors_per_image,
        ):
            matched = self._match_anchors_to_targets(
                anchors,
                transformed_target,
                context.num_anchors_per_level,
            )
            foreground = matched >= 0
            class_targets = torch.zeros(
                prediction.class_logits.shape,
                dtype=prediction.class_logits.dtype,
                device=prediction.class_logits.device,
            )
            box_targets = torch.zeros_like(prediction.boxes)
            quality_targets = torch.zeros_like(prediction.quality_logits)
            if foreground.any():
                matched_fg = matched[foreground]
                gt_boxes = transformed_target["boxes"].to(
                    device=prediction.boxes.device,
                    dtype=prediction.boxes.dtype,
                )
                gt_labels = transformed_target["labels"].to(
                    device=prediction.class_logits.device,
                    dtype=torch.long,
                )
                labels_fg = gt_labels[matched_fg]
                if (
                    labels_fg.numel()
                    and (
                        labels_fg.min() < 0
                        or labels_fg.max() >= self.num_classes
                    )
                ):
                    raise ValueError("FCOS target labels must be zero-based")
                matched_boxes = gt_boxes[matched_fg]
                box_targets[foreground] = matched_boxes.detach()
                with torch.no_grad():
                    iou = _aligned_box_iou(
                        prediction.boxes[foreground].detach(),
                        matched_boxes,
                    ).clamp(0.0, 1.0)
                quality_targets[foreground] = iou.detach()
                class_targets[foreground, labels_fg] = iou.detach().to(
                    class_targets.dtype
                )

            prepared_targets.append(
                {
                    "class_targets": class_targets.detach(),
                    "box_targets": box_targets.detach(),
                    "quality_targets": quality_targets.detach(),
                    "foreground": foreground.detach(),
                    "matched_idxs": matched.detach(),
                }
            )
        return SupervisedBatch(
            predictions=output.predictions,
            targets=tuple(prepared_targets),
            native_losses={},
        )

    def prepare_distillation(
        self, student: AdapterOutput, teacher: AdapterOutput
    ) -> DistillationPair:
        if not isinstance(student.context, FCOSAdapterContext) or not isinstance(
            teacher.context, FCOSAdapterContext
        ):
            raise ValueError("FCOS distillation requires FCOS adapter contexts")
        s_context = student.context
        t_context = teacher.context
        if s_context.sample_ids is None or t_context.sample_ids is None:
            raise ValueError("sample_ids are required for position-wise FCOS KD")
        if s_context.sample_ids != t_context.sample_ids:
            raise ValueError("student and teacher sample IDs do not correspond")
        if s_context.transformed_image_sizes != t_context.transformed_image_sizes:
            raise ValueError("student and teacher transformed image sizes differ")
        if s_context.num_anchors_per_level != t_context.num_anchors_per_level:
            raise ValueError("student and teacher FPN anchor counts differ")
        if s_context.class_names != t_context.class_names:
            raise ValueError("student and teacher class order differs")
        if len(s_context.anchors_per_image) != len(t_context.anchors_per_image):
            raise ValueError("student and teacher image counts differ")
        for s_anchors, t_anchors in zip(
            s_context.anchors_per_image, t_context.anchors_per_image
        ):
            if s_anchors.shape != t_anchors.shape or not torch.equal(
                s_anchors, t_anchors
            ):
                raise ValueError("student and teacher anchor geometry differs")
        if len(student.predictions) != len(teacher.predictions):
            raise ValueError("student and teacher image counts differ")
        for s_prediction, t_prediction in zip(
            student.predictions, teacher.predictions
        ):
            if s_prediction.class_logits.shape != t_prediction.class_logits.shape:
                raise ValueError("student and teacher class layouts differ")
            if s_prediction.boxes.shape != t_prediction.boxes.shape:
                raise ValueError("student and teacher decoded layouts differ")
        return DistillationPair(
            tuple(student.predictions), tuple(teacher.predictions)
        )

    def postprocess(
        self, output: AdapterOutput
    ) -> Sequence[Mapping[str, Tensor]]:
        if not isinstance(output.context, FCOSAdapterContext):
            raise ValueError("FCOS postprocessing requires FCOS context")
        context = output.context
        per_level_outputs = {
            key: list(value.split(context.num_anchors_per_level, dim=1))
            for key, value in context.raw_head_outputs.items()
        }
        native_outputs = self.detector.postprocess_detections(
            {
                "cls_logits": per_level_outputs["cls_logits"],
                "bbox_regression": per_level_outputs["bbox_regression"],
                # Torchvision's native postprocessor uses this third tensor in
                # sqrt(sigmoid(class) * sigmoid(quality)).
                "bbox_ctrness": per_level_outputs["quality_logits"],
            },
            [list(levels) for levels in context.anchors_per_level],
            list(context.transformed_images.image_sizes),
        )
        return self.detector.transform.postprocess(
            native_outputs,
            list(context.transformed_images.image_sizes),
            list(context.original_image_sizes),
        )


__all__ = [
    "ClassificationInitMode",
    "FCOSAdapterContext",
    "FCOSIoUHead",
    "TorchvisionFCOSAdapter",
]


In [ ]:
%%flir_module notebook_support
"""Kaggle-only orchestration around the canonical CLI and evaluator."""

from __future__ import annotations

import gc
import json
from pathlib import Path
import random
import time

import numpy as np
from PIL import Image, ImageDraw
import torch


def make_training_args(settings):
    from example_flir import parse_args
    argv = ["--data_root", str(settings["data_root"]), "--output_dir", str(settings["output_dir"])]
    options = {
        "total_iters": "--total_iters", "batch_size": "--batch_size",
        "eval_batch_size": "--eval_batch_size", "workers": "--workers",
        "weights": "--weights", "device": "--device", "teacher_mode": "--teacher-mode",
        "workflow": "--workflow", "eval_every": "--eval_every", "log_interval": "--log_interval",
        "intermediate_domain": "--intermediate-domain", "texture_strength": "--texture-strength",
        "btf_radius": "--btf-radius", "btf_iterations": "--btf-iterations",
        "resume": "--resume", "lr": "--lr", "min_sizes": "--min-sizes", "max_size": "--max_size",
        "phase1_end": "--phase1-end", "phase2_end": "--phase2-end", "phase3_end": "--phase3-end",
        "phase2_rgb_ratio": "--phase2-rgb-ratio", "phase3_rgb_ratio": "--phase3-rgb-ratio",
        "phase4_rgb_ratio": "--phase4-rgb-ratio",
    }
    for key, flag in options.items():
        value = settings.get(key)
        if value is not None:
            argv.append(flag)
            argv.extend(str(item) for item in value) if isinstance(value, (tuple, list)) else argv.append(str(value))
    if settings.get("eval_pseudo_labels", False):
        argv.append("--eval-pseudo-labels")
    return parse_args(argv)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def validate_dataset(data_root, batch_size):
    from data.datasets import FLIRRGBDataset, FLIRIRDataset, FLIRIRValDataset
    root = Path(data_root)
    for directory in ("JPEGImages", "Annotations"):
        if not (root / directory).is_dir():
            raise FileNotFoundError(f"Missing {directory} at {root}; attach the FLIR dataset and check DATA_ROOT")
    rgb = FLIRRGBDataset(str(root))
    ir = FLIRIRDataset(str(root))
    validation = FLIRIRValDataset(str(root))
    if len(rgb) < batch_size or len(ir) < batch_size or not len(validation):
        raise ValueError(f"Dataset too small/empty for drop_last training: RGB={len(rgb)}, IR={len(ir)}, IR val={len(validation)}, batch={batch_size}")
    missing = [str(root / "Annotations" / f"{stem}.xml") for stem in rgb.stems if not (root / "Annotations" / f"{stem}.xml").is_file()]
    if missing:
        raise FileNotFoundError(f"Missing RGB source annotation(s), first: {missing[:5]}")
    return {"rgb_train": len(rgb), "ir_train": len(ir), "ir_validation": len(validation)}


def preflight(args, manifest=None):
    """Check the actual dependencies/data/initial weights before model assembly."""
    import cv2
    import torchvision
    from torchvision.ops import nms
    from pycocotools.coco import COCO  # noqa: F401
    from example_flir import build_training_config
    from models.fcos_factory import serialize_effective_config
    from intermediate_domain import require_btf
    config = build_training_config(args)
    device = torch.device(args.device)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. In Kaggle select Settings > Accelerator > GPU, then restart and Run All.")
    box = torch.tensor([[0., 0., 8., 8.]], device=device)
    nms(box, torch.ones(1, device=device), .5)
    counts = validate_dataset(args.data_root, args.batch_size)
    if config.intermediate_domain.mode == "randommono_btf" and config.intermediate_domain.texture_strength:
        function = require_btf()
        result = function(np.full((12, 12), .5, np.float32), fr=config.intermediate_domain.btf_radius, numIter=1)
        if result.shape != (12, 12) or not np.isfinite(result).all():
            raise RuntimeError("OpenCV BTF capability check failed")
    # Resolve weights up front. Keep model factory initialization unchanged.
    try:
        if config.model.weights:
            from torchvision.models.detection import FCOS_ResNet50_FPN_Weights
            state = FCOS_ResNet50_FPN_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)
            del state
        elif config.model.pretrained_backbone:
            from torchvision.models import ResNet50_Weights
            state = ResNet50_Weights.DEFAULT.get_state_dict(progress=True, check_hash=True)
            del state
    except Exception as error:
        raise RuntimeError("Pretrained weights could not be loaded. Enable Kaggle Internet for the first download or populate the torchvision cache. No random-initialization fallback was used.") from error
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    effective = serialize_effective_config(config)
    (output / "effective_config.json").write_text(json.dumps(effective, indent=2), encoding="utf-8")
    if manifest is not None:
        (output / "source_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print({"torch": torch.__version__, "torchvision": torchvision.__version__, "opencv": cv2.__version__, "device": str(device), **counts})
    print(f"Training: {config.total_iters} iterations; outputs: {output}; bridge: {config.intermediate_domain}")
    return config


def save_bridge_preview(args, *, count=2):
    """Preview only source-train images; restore torch CPU RNG afterwards."""
    from data.datasets import FLIRRGBDataset
    from example_flir import build_training_config
    from intermediate_domain import IntermediateDomainAugmentation
    config = build_training_config(args)
    dataset = FLIRRGBDataset(args.data_root)
    items = [dataset[i] for i in range(min(count, len(dataset)))]
    alphas = (1., .75, .5, .25, 0.)
    state = torch.random.get_rng_state()
    start = time.perf_counter()
    try:
        torch.random.default_generator.manual_seed(123)
        views = IntermediateDomainAugmentation(config.intermediate_domain).make_views(
            tuple(item[0] for item in items), alphas,
            boxes=[item[1]["boxes"] for item in items],
        )
    finally:
        torch.random.set_rng_state(state)
    elapsed = time.perf_counter() - start
    width, height = 256, 205
    sheet = Image.new("RGB", (width * len(alphas), (height + 26) * len(items)), "white")
    draw = ImageDraw.Draw(sheet)
    for row, item in enumerate(items):
        for col, alpha in enumerate(alphas):
            array = (views[col][row].permute(1, 2, 0).cpu().numpy().clip(0, 1) * 255).round().astype(np.uint8)
            tile = Image.fromarray(array).resize((width, height))
            sheet.paste(tile, (col * width, row * (height + 26) + 26))
            draw.text((col * width + 4, row * (height + 26) + 5), f"alpha={alpha:g} | {item[2]}", fill="black")
    output = Path(args.output_dir) / "bridge_preview.png"
    output.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output)
    print(f"Bridge preprocessing: {elapsed / max(1, len(items)):.3f} s/image (CPU preview); {output}")
    return output


def select_checkpoint(output_dir, override=None):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"Requested evaluation checkpoint does not exist: {path}")
        return path
    for name in ("best_model.pth", "final_model.pth"):
        path = Path(output_dir) / name
        if path.is_file():
            return path
    raise FileNotFoundError(f"No best_model.pth or final_model.pth in {output_dir}; run training or set eval_checkpoint")


def release_training_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
%%flir_module optimization
"""Detector-neutral optimizer and iteration-based LR scheduler builders."""

from bisect import bisect_right

import torch
from torch import nn

from config import TrainingConfig


class WarmupMultiStepLR(torch.optim.lr_scheduler.LRScheduler):
    """Iteration scheduler whose complete schedule is checkpointable."""

    def __init__(
        self,
        optimizer,
        milestones,
        gamma: float,
        warmup_factor: float,
        warmup_iters: int,
        last_epoch: int = -1,
    ) -> None:
        self.milestones = tuple(milestones)
        self.gamma = float(gamma)
        self.warmup_factor = float(warmup_factor)
        self.warmup_iters = int(warmup_iters)
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        iteration = self.last_epoch
        if self.warmup_iters > 0 and iteration < self.warmup_iters:
            alpha = iteration / self.warmup_iters
            warmup = self.warmup_factor * (1.0 - alpha) + alpha
        else:
            warmup = 1.0
        decay = self.gamma ** bisect_right(self.milestones, iteration)
        return [base_lr * warmup * decay for base_lr in self.base_lrs]


def build_optimizer(model: nn.Module, config: TrainingConfig) -> torch.optim.SGD:
    """Build D3T-style SGD over every trainable student parameter exactly once."""

    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not parameters:
        raise ValueError("student model has no trainable parameters")
    settings = config.optimizer
    return torch.optim.SGD(
        parameters,
        lr=settings.base_lr,
        momentum=settings.momentum,
        weight_decay=settings.weight_decay,
    )


def build_lr_scheduler(optimizer, config: TrainingConfig):
    """Build the linear-warmup MultiStep schedule used by D3T."""

    settings = config.optimizer
    return WarmupMultiStepLR(
        optimizer,
        milestones=settings.milestones,
        gamma=settings.gamma,
        warmup_factor=settings.warmup_factor,
        warmup_iters=settings.warmup_iters,
    )


__all__ = ["WarmupMultiStepLR", "build_lr_scheduler", "build_optimizer"]


In [ ]:
%%flir_module saga
"""
SAGA — Semantic-Aware Grayscale Augmentation

Purpose: create an intermediate domain that bridges RGB and IR by converting
*only object instances* (inside bounding boxes) to grayscale while keeping the
background in RGB.

Domain role:
  RGB (color + texture)  →  MID/SAGA (gray objects, RGB background)  →  IR (grayscale)

This is NOT a simple augmentation — it is the domain bridge that enables the
curriculum learning strategy: RGB → MID → IR.
"""

import random
from typing import List, Optional

import torch
from typing import Literal

MidLevel = Literal["mid_near_rgb", "mid_intermediate", "mid_near_ir"]


class SemanticAwareGrayAugmentation:
    """
    Convert object bounding-box regions to grayscale; keep background as RGB.

    Args:
        apply_prob: probability of applying the transform to each image.
                    Set to 1.0 to always apply (deterministic).
    """

    # ITU-R BT.601 luma coefficients (standard for digital video)
    _LUMA_WEIGHTS = torch.tensor([0.299, 0.587, 0.114])

    def __init__(self, apply_prob: float = 0.5) -> None:
        if not (0.0 <= apply_prob <= 1.0):
            raise ValueError(f"apply_prob must be in [0, 1], got {apply_prob}")
        self.apply_prob = apply_prob

    # ------------------------------------------------------------------
    # Public interface
    # ------------------------------------------------------------------

    def __call__(
        self,
        image: torch.Tensor,          # [3, H, W]
        boxes: Optional[torch.Tensor],  # [N, 4]  xyxy, can be empty
    ) -> torch.Tensor:
        """
        Apply SAGA to a single image with stochastic gating.

        Returns the (possibly transformed) image — same shape as input.
        """
        if random.random() > self.apply_prob:
            return image
        return self.apply(image, boxes)

    def apply(
        self,
        image: torch.Tensor,           # [3, H, W]
        boxes: Optional[torch.Tensor], # [N, 4]  xyxy
    ) -> torch.Tensor:
        """
        Deterministic SAGA transform (no probability gating).

        - If boxes is None or empty → return image unchanged.
        - Otherwise: object regions → grayscale, background → RGB.
        """
        if image.dim() != 3 or image.shape[0] != 3:
            raise ValueError(
                f"Expected image shape [3, H, W], got {tuple(image.shape)}"
            )

        if boxes is None or boxes.numel() == 0:
            return image.clone()

        _, H, W = image.shape
        device = image.device

        # Build per-pixel object mask from all bounding boxes
        obj_mask = self._build_object_mask_static(boxes, H, W, device)  # [H, W] bool

        if not obj_mask.any():
            return image.clone()

        # Compute grayscale version of the whole image
        gray = self._to_grayscale(image)          # [H, W]
        gray_3ch = gray.unsqueeze(0).expand(3, H, W)  # [3, H, W]

        # Blend: object mask → gray, background → original RGB
        mask_3ch = obj_mask.unsqueeze(0).expand(3, H, W)
        result = torch.where(mask_3ch, gray_3ch, image)

        return result

    def apply_to_batch(
        self,
        images: torch.Tensor,            # [B, 3, H, W]
        batch_boxes: List[Optional[torch.Tensor]],  # list of [N_i, 4] or None
    ) -> torch.Tensor:
        """
        Apply SAGA to a whole batch.

        Each image is processed independently with its own boxes.
        The stochastic gating (apply_prob) is evaluated per image.
        """
        if images.dim() != 4 or images.shape[1] != 3:
            raise ValueError(
                f"Expected images shape [B, 3, H, W], got {tuple(images.shape)}"
            )
        if len(images) != len(batch_boxes):
            raise ValueError(
                f"images batch size {len(images)} != len(batch_boxes) {len(batch_boxes)}"
            )

        results = [
            self(img, boxes)
            for img, boxes in zip(images, batch_boxes)
        ]
        return torch.stack(results, dim=0)

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _to_grayscale(self, image: torch.Tensor) -> torch.Tensor:
        """
        Convert [3, H, W] RGB to [H, W] grayscale using ITU-R BT.601 luma.
        The result is replicated to 3 channels by the caller.
        """
        weights = self._LUMA_WEIGHTS.to(device=image.device, dtype=image.dtype)
        return (image * weights.view(3, 1, 1)).sum(dim=0)

    @staticmethod
    def _build_object_mask_static(
        boxes: torch.Tensor,   # [N, 4]  xyxy float
        H: int,
        W: int,
        device: torch.device,
    ) -> torch.Tensor:
        """
        Create a binary mask [H, W] that is True inside any bounding box.

        Out-of-bound coordinates are clamped to image boundaries.
        Zero-area boxes (x1 >= x2 or y1 >= y2 after clamping) are skipped.
        """
        mask = torch.zeros(H, W, dtype=torch.bool, device=device)

        for box in boxes:
            x1 = max(0, int(box[0].item()))
            y1 = max(0, int(box[1].item()))
            x2 = min(W, int(box[2].item()))
            y2 = min(H, int(box[3].item()))

            if x2 > x1 and y2 > y1:
                mask[y1:y2, x1:x2] = True

        return mask


# ---------------------------------------------------------------------------
# SoftSAGA — soft interpolation between RGB and grayscale in object regions
# ---------------------------------------------------------------------------

class SoftSAGA:
    """
    Soft Semantic-Aware Grayscale Augmentation.

    Unlike hard SAGA (binary object→gray or bg→gray), SoftSAGA blends
    object regions between RGB and grayscale using alpha:

        pixel_out = alpha * rgb_pixel + (1 - alpha) * gray_pixel

    Background stays unchanged (original RGB).

    3 levels for curriculum bridge RGB → MID → IR:
        near_rgb     : alpha ~ 0.7  (objects mostly RGB, slightly desaturated)
        intermediate : alpha ~ 0.5  (half RGB, half gray)
        near_ir      : alpha ~ 0.25 (objects mostly gray, approaching IR)
    """

    _LUMA_WEIGHTS = torch.tensor([0.299, 0.587, 0.114])

    def apply(
        self,
        image: torch.Tensor,            # [3, H, W]
        boxes: Optional[torch.Tensor],  # [N, 4] xyxy
        alpha: float,                   # 1.0=pure RGB, 0.0=pure gray
    ) -> torch.Tensor:
        """Deterministic soft blend on object regions. Background unchanged."""
        if image.dim() != 3 or image.shape[0] != 3:
            raise ValueError(f"Expected [3,H,W], got {tuple(image.shape)}")
        if boxes is None or boxes.numel() == 0:
            return image.clone()

        _, H, W = image.shape
        device = image.device

        # Grayscale version of whole image (luma)
        weights = self._LUMA_WEIGHTS.to(device=device, dtype=image.dtype)
        gray = (image * weights.view(3, 1, 1)).sum(dim=0)   # [H, W]
        gray_3ch = gray.unsqueeze(0).expand(3, H, W)         # [3, H, W]

        # Build object mask
        obj_mask = SemanticAwareGrayAugmentation._build_object_mask_static(
            boxes, H, W, device
        )
        if not obj_mask.any():
            return image.clone()

        # Soft blend in object regions: alpha*rgb + (1-alpha)*gray
        blended = alpha * image + (1.0 - alpha) * gray_3ch  # [3, H, W]
        mask_3ch = obj_mask.unsqueeze(0).expand(3, H, W)
        return torch.where(mask_3ch, blended, image)

    def apply_to_batch(
        self,
        images: torch.Tensor,                        # [B, 3, H, W]
        batch_boxes: List[Optional[torch.Tensor]],
        alpha: float,
    ) -> torch.Tensor:
        return torch.stack([
            self.apply(img, boxes, alpha)
            for img, boxes in zip(images, batch_boxes)
        ])


In [ ]:
%%flir_module scheduler
from fractions import Fraction
from typing import Literal
from config import Phase 

# Route identifiers describe the curriculum action; physical source selection
# is resolved separately so an IR-named route can still use RGB/SAGA inputs.
DomainStep = Literal[
    "p1_rgb_supervised",    # Student(RGB+GT)
    "p2_rgb_flow",          # Student(SAGA-Weak) + Teacher(RGB) + GT
    "p2_ir_flow",           # Student(SAGA-High) + Teacher(SAGA-Mid) + GT
    "p3_rgb_flow",          # Student(SAGA-Mid) + Teacher(SAGA-Weak) + GT
    "p3_ir_flow",           # Student(IR) + Teacher(SAGA-High) [NO GT]
    "p4_rgb_supervised",   # Student(RGB+GT), no KD or teacher update
    "p4_ir_focus"           # Student(IR) + Teacher(IR) [NO GT]
]

class CurriculumScheduler:
    """
    Handles within-phase alternation (RGB vs IR flows) 
    using ratios defined in TrainingConfig.
    """
    def __init__(self, config) -> None:
        self.config = config
        # Counters are internal to maintain strict alternating patterns
        
        self.Phase = Phase
        self._counters = {
            Phase.PHASE2_TRANSITION: 0,
            Phase.PHASE3_ADAPTATION:  0,
            Phase.PHASE4_IR_FOCUS:    0,
        }

    def get_next_step(self, global_step: int) -> DomainStep:
        """
        Main logic to decide what the Student and Teachers see in this iteration.
        """
        # Resolve the phase from configured iteration boundaries before choosing
        # a route. Phase transitions do not consume either data loader here.
        phase = self.config.get_phase(global_step)

        if phase == self.Phase.PHASE1_RGB_WARMUP:
            # Warmup stays on the labeled RGB path without teacher-driven routing.
            return "p1_rgb_supervised"

        elif phase == self.Phase.PHASE2_TRANSITION:
            # Transition alternates two RGB-backed view policies using the
            # configured RGB-route proportion, while retaining GT on both.
            return self._alternate(
                phase=self.Phase.PHASE2_TRANSITION,
                primary="p2_rgb_flow",
                secondary="p2_ir_flow",
                ratio=self.config.curriculum.phase2_rgb_sampling_ratio
            )

        elif phase == self.Phase.PHASE3_ADAPTATION:
            # Adaptation alternates the RGB/SAGA route and real unlabeled IR.
            # The configured proportion continues to mean the RGB-route share.
            return self._alternate(
                phase=self.Phase.PHASE3_ADAPTATION,
                primary="p3_rgb_flow",
                secondary="p3_ir_flow",
                ratio=self.config.curriculum.phase3_rgb_sampling_ratio
            )

        else: # PHASE4_IR_FOCUS
            # Retain source supervision within the fixed phase-4 budget.
            # The existing alternation gives one RGB and four IR steps at 0.2.
            return self._alternate(
                phase=self.Phase.PHASE4_IR_FOCUS,
                primary="p4_rgb_supervised",
                secondary="p4_ir_focus",
                ratio=self.config.curriculum.phase4_rgb_sampling_ratio,
            )

    @staticmethod
    def get_source_domain(step_name: str) -> str:
        """Resolve the existing physical source policy, including RGB-backed P2 IR."""
        # Supply data flow with the actual loader domain, independently of the
        # teacher name or SAGA appearance level carried by the route settings.
        if step_name in {
            "p1_rgb_supervised", "p2_rgb_flow", "p2_ir_flow", "p3_rgb_flow",
            "p4_rgb_supervised",
        }:
            return "rgb"
        if step_name in {"p3_ir_flow", "p4_ir_focus"}:
            return "ir"
        raise ValueError(f"Unsupported curriculum step: {step_name}")

    def should_evaluate_pseudo_labels(self, global_step, *, last_eval_step):
        """Check global windows; fixed IR evaluation is independent of train routes.

        The trainer separately requires initialized teachers. A delayed first
        pass does not shift later iteration boundaries or consume loader routes.
        """
        cfg = self.config.pseudo_label_eval
        completed_step = global_step + 1
        return (
            cfg.enabled
            and completed_step // cfg.interval > last_eval_step // cfg.interval
        )

    def _alternate(self, phase, primary: DomainStep, secondary: DomainStep, ratio: float) -> DomainStep:
        """Helper to alternate between two steps based on a ratio."""
        # Keep a deterministic per-phase route sequence. Persisted counters
        # allow resume to continue its position instead of restarting the cycle.
        period = self._ratio_to_period(ratio)
        n_primary = round(period * ratio)

        count = self._counters[phase]
        self._counters[phase] += 1

        position = count % period
        return primary if position < n_primary else secondary

    @staticmethod
    def _ratio_to_period(ratio: float) -> int:
        """Return a short exact period for the requested RGB ratio."""
        if not 0.0 <= ratio <= 1.0:
            raise ValueError("sampling ratio must be in [0, 1]")
        if ratio in (0.0, 1.0):
            return 1
        return Fraction(float(ratio)).limit_denominator(100).denominator

    def state_dict(self):
        # Persist route-selection progress independently from model/LR state.
        return {phase.name: count for phase, count in self._counters.items()}

    def load_state_dict(self, state):
        for phase in self._counters:
            if phase.name in state:
                self._counters[phase] = int(state[phase.name])


In [ ]:
%%flir_module trainer
"""Training orchestration for RGB baseline and dual-teacher D3T routes."""

from collections import defaultdict
import logging
import os
import time
from typing import Any, Dict, Optional

import torch
import torch.nn as nn
from torch.optim import Optimizer
from torch.utils.data import DataLoader

from config import Phase, StepRouting, TrainingConfig
from models.d3t_adapter import DistillationSettings
from models.d3t_wrapper import D3TWrapper as D3TWrapper
from scheduler import CurriculumScheduler, DomainStep
from data.augmentations import StudentAugmentor
from training import teachers
from training.data_flow import RouteDataFlow
from intermediate_domain import IntermediateDomainAugmentation
from training.loss_flow import LossFlow, component_weight
from training.observability import log_iteration, log_pseudo_label_evaluation

logger = logging.getLogger(__name__)


class CurriculumDomainAdaptationTrainer:
    """Own routing, optimization, EMA, checkpointing, and evaluation hooks."""

    def __init__(
        self,
        student: nn.Module,
        rgb_teacher: Optional[nn.Module],
        ir_teacher: Optional[nn.Module],
        optimizer: Optimizer,
        config: TrainingConfig,
        rgb_loader: DataLoader,
        ir_loader: Optional[DataLoader] = None,
        phase_evaluator: Any = None,
        lr_scheduler: Any = None,
        pseudo_label_evaluator: Any = None,
    ) -> None:
        # Retain the live dependencies and effective configuration on the
        # orchestrator so execution modules receive resolved inputs per step.
        self.student = student
        self.rgb_teacher = rgb_teacher
        self.ir_teacher = ir_teacher
        self.optimizer = optimizer
        self.config = config
        self.device = torch.device(config.device)
        self.phase_evaluator = phase_evaluator
        self.lr_scheduler = lr_scheduler
        self.augmentor = StudentAugmentor(config)

        # Prepare model state before constructing lazy data cursors. Their
        # creation order is kept stable to preserve the existing RNG behavior.
        self._setup_models()
        self.data_flow = RouteDataFlow(
            rgb_loader, ir_loader, device=self.device, augmentor=self.augmentor,
            bridge=IntermediateDomainAugmentation(config.intermediate_domain)
        )
        # Own the progress and lifecycle state that must survive checkpoint
        # save/resume; execution modules do not maintain competing copies.
        self.scheduler = CurriculumScheduler(config)
        self.global_step = 0
        self.best_map = 0.0
        self.loss_history = defaultdict(list)
        self.ema_initialized = False
        # Bootstrap supplies isolated IR validation loaders. Audit progress stays
        # separate from student validation history and best-checkpoint selection.
        self.last_pseudo_label_eval_step = 0
        self.last_full_pseudo_label_eval_step = 0
        self.pseudo_label_evaluator = pseudo_label_evaluator
        if config.pseudo_label_eval.enabled and config.workflow != "rgb_baseline" and pseudo_label_evaluator is None:
            raise ValueError("enabled pseudo-label evaluation requires an IR validation evaluator")

    def _teacher_models(self):
        # References stay on the trainer so checkpoint keys and ownership are unchanged.
        return {"rgb": self.rgb_teacher, "ir": self.ir_teacher}

    def _setup_models(self) -> None:
        teachers.prepare_models(self.student, self._teacher_models(), device=self.device)

    def _alpha_map(self):
        # Resolve appearance-level names from config before invoking data flow;
        # these levels describe SAGA strength, not the physical input domain.
        ss_cfg = self.config.soft_saga
        return {
            "rgb": 1.0,
            "weak": ss_cfg.alpha_near_rgb,
            "mid": ss_cfg.alpha_intermediate,
            "high": ss_cfg.alpha_near_ir,
            "ir": 0.0,
        }

    def _prepare_data_by_route(self, step_name: str, route: StepRouting):
        """Resolve configured values before delegating data execution."""
        source_domain = self.scheduler.get_source_domain(step_name)
        alpha_map = self._alpha_map()
        return self.data_flow.prepare(
            source_domain=source_domain,
            use_gt=route.use_gt,
            student_alpha=(
                alpha_map[route.student_saga_level] if source_domain == "rgb" else None
            ),
            teacher_alpha=(
                alpha_map[route.teacher_saga_level] if source_domain == "rgb" else None
            ),
        )

    def _ensure_ema_initialized(self) -> None:
        # Decide whether initialization is due from persisted state and config.
        # A resumed, initialized run must continue without another hard copy.
        if self.ema_initialized:
            return
        if self.global_step < self.config.ema.start_steps:
            return
        # Select the enabled references here, then ask the teacher executor to
        # copy weights; schedule decisions remain on the orchestration side.
        if self.config.teacher_mode == "two_teacher":
            active_names = ("rgb", "ir")
        else:
            active_names = (self.config.teacher_mode,)
        teachers.initialize_teachers(
            self.student, self._teacher_models(), names=active_names
        )
        self.ema_initialized = True

    def _enabled_teacher_names(self, route: StepRouting) -> tuple[str, ...]:
        # Intersect this route's teacher requests with the experiment's enabled
        # teacher mode while preserving the configured route order.
        if self.config.teacher_mode == "two_teacher":
            enabled = {"rgb", "ir"}
        else:
            enabled = {self.config.teacher_mode}
        return tuple(name for name in route.teacher_names if name in enabled)

    def _teacher_for_name(self, name: str) -> nn.Module:
        return teachers.get_teacher(self._teacher_models(), name)

    def _distillation_settings(self, teacher_name: str, phase: Phase):
        # Resolve teacher-specific selection thresholds and shared KD settings
        # for this phase before passing them to the loss executor.
        schedule = getattr(self.config.distill, f"{teacher_name}_teacher")
        top_ratio, min_hm = schedule.get_params(phase)
        return DistillationSettings(
            top_ratio=top_ratio,
            min_hm=min_hm,
            hm_alpha=self.config.distill.hm_alpha,
            hm_beta=self.config.distill.hm_beta,
            uncertainty_alpha=self.config.distill.un_regular_alpha,
        )

    @staticmethod
    def _component_weight(name: str, config) -> float:
        """Keep the existing helper while passing explicit weights to the executor."""
        return component_weight(
            name, uhl=config.weight_uhl, quality=config.weight_quality,
            deltas=config.weight_deltas, logits=config.weight_logits,
        )

    def train_one_iteration(self) -> Dict[str, Any]:
        # Route selection: the baseline uses its fixed supervised route, while
        # curriculum training asks the scheduler for the current step's route.
        self.student.train()
        if self.config.workflow == "rgb_baseline":
            step_name: DomainStep = "p1_rgb_supervised"
        else:
            step_name = self.scheduler.get_next_step(self.global_step)
        phase = self.config.get_phase(self.global_step)
        route = self.config.mid_routing.get_routing(step_name)
        if self.config.workflow == "rgb_baseline":
            route = self.config.mid_routing.p1_rgb_supervised

        # Data preparation: obtain one source batch with aligned student/teacher
        # views, optional GT, and explicit sample identity.
        data = self._prepare_data_by_route(step_name, route)
        targets = data["targets"]
        sample_ids = data["sample_ids"]
        iteration_lr = float(self.optimizer.param_groups[0]["lr"])
        self.optimizer.zero_grad(set_to_none=True)

        # Establish teacher references before any forward that can consume them.
        if self.config.workflow != "rgb_baseline":
            self._ensure_ema_initialized()

        # Student forward: choose resize once and reuse this raw output for
        # every applicable loss, avoiding a second student forward for KD.
        select_resize = getattr(
            self.student, "select_training_resize_short_edge", None
        )
        resize_short_edge = None if select_resize is None else select_resize()
        student_raw_kwargs = {
            "sample_ids": sample_ids,
        }
        if resize_short_edge is not None:
            student_raw_kwargs["resize_short_edge"] = resize_short_edge
        student_output = self.student.raw(
            data["student_images"], targets, **student_raw_kwargs
        )
        # Loss setup: resolve phase/component weights in the orchestrator and
        # give the executor a fresh accumulator for this iteration only.
        losses = LossFlow(
            uhl=self.config.loss.weight_uhl,
            quality=self.config.loss.weight_quality,
            deltas=self.config.loss.weight_deltas,
            logits=self.config.loss.weight_logits,
        )
        sup_weight, distill_weight = self.config.loss.get_phase_weights(phase)
        if self.config.workflow == "rgb_baseline":
            sup_weight, distill_weight = 1.0, 0.0

        # Supervised contribution: use transformed GT only when this route and
        # the effective phase weight request it.
        if targets is not None and route.use_gt and sup_weight > 0:
            losses.supervised(
                self.student, student_output, targets, phase_weight=sup_weight
            )

        # Distillation contribution: use only enabled, initialized references.
        # Each teacher consumes the paired view with the student's resize.
        if (
            self.config.workflow != "rgb_baseline"
            and self.ema_initialized
            and distill_weight > 0
        ):
            enabled_teacher_names = self._enabled_teacher_names(route)
            for teacher_name in enabled_teacher_names:
                teacher = self._teacher_for_name(teacher_name)
                teacher_output = teachers.forward_teacher(
                    teacher,
                    data["teacher_images"],
                    sample_ids=sample_ids,
                    resize_short_edge=resize_short_edge,
                )
                settings = self._distillation_settings(teacher_name, phase)
                losses.distill(
                    self.student,
                    student_output,
                    teacher_output,
                    settings,
                    teacher_name=teacher_name,
                    phase_weight=distill_weight,
                )

        # Preserve the existing optimizer eligibility rule, including a scalar
        # fallback when this route produced no differentiable contributions.
        total_loss = losses.total_loss
        logs = losses.logs

        if total_loss is None:
            total_loss = torch.zeros((), device=self.device)
        did_step = False
        # Student update: backward, clipping, optimizer, and LR scheduling stay
        # together and keep their original order within a successful step.
        if total_loss.requires_grad and bool(torch.isfinite(total_loss).item()):
            total_loss.backward()
            if self.config.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(
                    self.student.parameters(), self.config.grad_clip
                )
            self.optimizer.step()
            if self.lr_scheduler is not None:
                self.lr_scheduler.step()
            did_step = True

        # Teacher update: follow an eligible student update with EMA only for
        # the teacher targeted by this route and enabled by the experiment.
        if (
            did_step
            and self.ema_initialized
            and route.ema_target in self._enabled_teacher_names(route)
        ):
            teachers.update_teacher(
                self._teacher_for_name(route.ema_target),
                self.student,
                alpha=self.config.ema.alpha,
                global_step=self.global_step,
            )

        # Observation and progress: collect detached values after updates, then
        # advance the global step once and return the existing metric payload.
        logs["total_loss"] = float(total_loss.detach().item())
        logs["step_type"] = step_name
        logs["phase"] = phase.name
        logs["ema_initialized"] = float(self.ema_initialized)
        logs["lr"] = iteration_lr
        self.loss_history[step_name].append(logs["total_loss"])
        self.global_step += 1
        logs["global_step"] = self.global_step
        # Observe the current teachers after the iteration/EMA update using a
        # separate fixed dataset, never the train batch or supervised targets.
        if (
            self.pseudo_label_evaluator is not None and self.ema_initialized
            and self.scheduler.should_evaluate_pseudo_labels(
                self.global_step - 1, last_eval_step=self.last_pseudo_label_eval_step
            )
        ):
            logs["pseudo_label_evaluations"] = self._evaluate_pseudo_labels(phase)
        return logs

    def _evaluate_pseudo_labels(self, phase, *, full_dataset=False):
        """Resolve teacher/settings policy and delegate isolated IR evaluation."""
        names = ("rgb", "ir") if self.config.teacher_mode == "two_teacher" else (self.config.teacher_mode,)
        records = []
        for name in names:
            settings = self._distillation_settings(name, phase)
            result = self.pseudo_label_evaluator.evaluate(
                self._teacher_for_name(name), settings,
                iou_threshold=self.config.pseudo_label_eval.iou_threshold,
                resize_short_edge=self.config.model.min_sizes[-1], full_dataset=full_dataset,
            )
            records.append({
                **result, "global_step": self.global_step, "phase": phase.name,
                "source_domain": "ir", "teacher": name,
                "trigger": "final" if full_dataset else "periodic",
                "top_ratio": settings.top_ratio, "min_hm": settings.min_hm,
                "hm_alpha": settings.hm_alpha, "hm_beta": settings.hm_beta,
                "resize_short_edge": self.config.model.min_sizes[-1],
                "iou_threshold": self.config.pseudo_label_eval.iou_threshold,
            })
        for record in records:
            log_pseudo_label_evaluation(record, output_dir=self.config.output_dir, logger=logger)
        if full_dataset:
            self.last_full_pseudo_label_eval_step = self.global_step
        else:
            self.last_pseudo_label_eval_step = self.global_step
        return records

    def train(self) -> None:
        logger.info(
            "Starting training: total_iters=%s starting_from_step=%s workflow=%s",
            self.config.max_iter,
            self.global_step,
            self.config.workflow,
        )
        for _ in range(self.global_step, self.config.max_iter):
            # Execute one iteration before applying the existing log, validation,
            # and checkpoint cadence to the resulting global step.
            started = time.time()
            logs = self.train_one_iteration()
            if self.global_step % self.config.log_interval == 0:
                logs["iter_time"] = time.time() - started
                self._log(logs)
            # Delegate evaluation triggers and best-model callbacks to the
            # evaluator, then retain the best score in trainer checkpoint state.
            if self.phase_evaluator is not None:
                phase = self.config.get_phase(self.global_step)
                results = self.phase_evaluator.step(
                    self.student, self.global_step, phase
                )
                if results and "mAP@0.5" in results:
                    self.best_map = max(self.best_map, results["mAP@0.5"])
            # Preserve the current periodic snapshot cadence alongside the
            # separately triggered best-model and final-model saves.
            if self.global_step > 0 and self.global_step % 5000 == 0:
                self.save_checkpoint(f"checkpoint_{self.global_step:06d}.pth")

        # A normal completed run also measures every IR validation image once.
        # Persisting this flag makes resuming a completed final model idempotent.
        if (self.pseudo_label_evaluator is not None and self.ema_initialized
                and self.global_step >= self.config.max_iter
                and self.last_full_pseudo_label_eval_step != self.global_step):
            self._evaluate_pseudo_labels(
                self.config.get_phase(max(0, self.global_step - 1)), full_dataset=True,
            )

    def save_checkpoint(self, filename: str) -> None:
        from models.fcos_factory import build_checkpoint_metadata

        # Snapshot model, optimization, routing, and evaluation progress together
        # using the existing payload keys consumed by resume and evaluation.
        os.makedirs(self.config.output_dir, exist_ok=True)
        path = os.path.join(self.config.output_dir, filename)
        payload = {
            "metadata": build_checkpoint_metadata(self.config),
            "global_step": self.global_step,
            "best_map": self.best_map,
            "ema_initialized": self.ema_initialized,
            "last_pseudo_label_eval_step": self.last_pseudo_label_eval_step,
            "last_full_pseudo_label_eval_step": self.last_full_pseudo_label_eval_step,
            "pseudo_label_evaluator": (None if self.pseudo_label_evaluator is None
                                       else self.pseudo_label_evaluator.state_dict()),
            "student": self.student.state_dict(),
            "rgb_teacher": None if self.rgb_teacher is None else self.rgb_teacher.state_dict(),
            "ir_teacher": None if self.ir_teacher is None else self.ir_teacher.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "scheduler": self.scheduler.state_dict(),
            "lr_scheduler": (
                None if self.lr_scheduler is None else self.lr_scheduler.state_dict()
            ),
            "phase_evaluator": (
                self.phase_evaluator.state_dict()
                if self.phase_evaluator is not None
                and hasattr(self.phase_evaluator, "state_dict")
                else None
            ),
        }
        torch.save(payload, path)
        logger.info("Checkpoint saved -> %s", path)

    def load_checkpoint(self, path: str) -> None:
        from models.fcos_factory import validate_checkpoint_metadata

        # Validate experiment compatibility and required teacher states before
        # restoring model tensors, keeping the current rejection behavior.
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        metadata = checkpoint.get("metadata")
        if metadata is None:
            raise ValueError("Checkpoint has no D3T metadata; legacy checkpoints are unsupported")
        validate_checkpoint_metadata(metadata, self.config)
        configured_teachers = {
            "rgb_teacher": self.rgb_teacher,
            "ir_teacher": self.ir_teacher,
        }
        missing_teacher_states = [
            name
            for name, teacher in configured_teachers.items()
            if teacher is not None and checkpoint.get(name) is None
        ]
        if missing_teacher_states:
            missing = ", ".join(missing_teacher_states)
            raise ValueError(
                f"Checkpoint is missing state for configured teacher(s): {missing}"
            )
        # Restore learned state first, followed by optimization and routing
        # state, so the next iteration continues with the saved update schedule.
        self.student.load_state_dict(checkpoint["student"])
        for name, teacher in configured_teachers.items():
            if teacher is not None:
                teacher.load_state_dict(checkpoint[name])
        self.optimizer.load_state_dict(checkpoint["optimizer"])
        if "scheduler" in checkpoint:
            self.scheduler.load_state_dict(checkpoint["scheduler"])
        if self.lr_scheduler is not None:
            lr_scheduler_state = checkpoint.get("lr_scheduler")
            if lr_scheduler_state is None:
                raise ValueError("Checkpoint is missing LR scheduler state")
            self.lr_scheduler.load_state_dict(lr_scheduler_state)
        # Restore evaluation history and lifecycle flags, including the EMA
        # initialization flag that prevents an unintended teacher reset.
        evaluator_state = checkpoint.get("phase_evaluator")
        if (
            evaluator_state is not None
            and self.phase_evaluator is not None
            and hasattr(self.phase_evaluator, "load_state_dict")
        ):
            self.phase_evaluator.load_state_dict(evaluator_state)
        self.global_step = int(checkpoint.get("global_step", 0))
        self.best_map = float(checkpoint.get("best_map", 0.0))
        self.ema_initialized = bool(checkpoint.get("ema_initialized", False))
        # Older baseline checkpoints do not carry observer progress. An audit
        # is eligible after teacher initialization if a global tick has passed.
        self.last_pseudo_label_eval_step = int(checkpoint.get("last_pseudo_label_eval_step", 0))
        self.last_full_pseudo_label_eval_step = int(checkpoint.get("last_full_pseudo_label_eval_step", 0))
        audit_state = checkpoint.get("pseudo_label_evaluator")
        if self.pseudo_label_evaluator is not None and audit_state is not None:
            self.pseudo_label_evaluator.load_state_dict(audit_state)
        logger.info("Resumed from step %s", self.global_step)

    def _log(self, log: Dict[str, Any]) -> None:
        log_iteration(log, self.global_step, logger=logger)


In [ ]:
%%flir_module training
"""Training execution modules; configuration and scheduling stay with callers."""


In [ ]:
%%flir_module training.bootstrap
"""Assemble training dependencies while the entrypoint owns the run lifecycle."""

import logging
import os

import torch
from torch.utils.data import DataLoader, Subset

from config import TrainingConfig
from data import FLIR_CLASSES, NUM_CLASSES, build_dataloaders
from data.datasets import FLIRIRDataset
from data.dataloaders import ir_collate
from data.ir_annotations import IRAnnotationReader
from evaluator import DetectionEvaluator, PhaseEvaluator
from models.fcos_factory import build_fcos_d3t_trio
from optimization import build_lr_scheduler, build_optimizer
from trainer import CurriculumDomainAdaptationTrainer
from training.pseudo_label_eval import FixedIRPseudoLabelEvaluator, select_fixed_ir_subset


def move_model_trio_to_device(student, rgb_teacher, ir_teacher, device):
    """Move the student and whichever teachers are enabled to one device."""

    student = student.to(device)
    if rgb_teacher is not None:
        rgb_teacher = rgb_teacher.to(device)
    if ir_teacher is not None:
        ir_teacher = ir_teacher.to(device)
    return student, rgb_teacher, ir_teacher


def build_pseudo_label_evaluator(config, *, logger):
    """Resolve audit policy once and assemble dedicated, deterministic IR loaders."""
    audit = config.pseudo_label_eval
    if not audit.enabled or config.workflow == "rgb_baseline":
        return None
    dataset = FLIRIRDataset(config.data.root, split="validation")
    reader = IRAnnotationReader(config.data.root)
    indices = select_fixed_ir_subset(
        dataset.ir_paths, reader, sample_size=audit.sample_size, seed=audit.subset_seed,
    )
    # Each loader owns a generator so iterator/worker seeding never consumes
    # the global RNG used by training. Retain the final incomplete batch.
    def loader(data):
        return DataLoader(
            data, batch_size=config.loader.eval.batch_size,
            num_workers=config.loader.eval.num_workers, shuffle=False, drop_last=False,
            pin_memory=config.device.startswith("cuda"), collate_fn=ir_collate,
            generator=torch.Generator().manual_seed(audit.subset_seed),
        )
    logger.info("Pseudo-label IR validation: fixed_images=%d full_images=%d batch_size=%d seed=%d",
                len(indices), len(dataset), config.loader.eval.batch_size, audit.subset_seed)
    return FixedIRPseudoLabelEvaluator(
        fixed_loader=loader(Subset(dataset, indices)), full_loader=loader(dataset),
        reader=reader, device=torch.device(config.device),
        fixed_sample_ids=tuple(dataset.ir_paths[i].stem for i in indices),
    )


def build_training(
    config: TrainingConfig, *, logger: logging.Logger
) -> CurriculumDomainAdaptationTrainer:
    """Build and wire the existing factories in their original order."""
    # Data assembly: create all source/target training and validation loaders
    # from the effective config, then expose their sizes in the existing log.
    device = torch.device(config.device)
    loaders = build_dataloaders(config)
    rgb_train_loader = loaders["rgb_train"]
    ir_train_loader = loaders["ir_train"]
    rgb_val_loader = loaders["rgb_val"]
    ir_val_loader = loaders["ir_val"]
    logger.info(
        "RGB train=%d, IR train=%d, RGB val=%d, IR val=%d",
        len(rgb_train_loader.dataset),
        len(ir_train_loader.dataset),
        len(rgb_val_loader.dataset),
        len(ir_val_loader.dataset),
    )

    # Model assembly: build the student and configured teachers before binding
    # optimization state to the student's actual device-resident parameters.
    student, rgb_teacher, ir_teacher = move_model_trio_to_device(
        *build_fcos_d3t_trio(config),
        device,
    )
    optimizer = build_optimizer(student, config)
    lr_scheduler = build_lr_scheduler(optimizer, config)

    # Evaluation assembly: pair the existing AP50 metric implementation with
    # domain loaders, evaluation cadence, and best-result tracking.
    evaluator = DetectionEvaluator(
        num_classes=NUM_CLASSES,
        class_names=list(FLIR_CLASSES),
        iou_thresholds=[0.5],
    )
    phase_evaluator = PhaseEvaluator(
        evaluator=evaluator,
        ir_val_loader=ir_val_loader,
        rgb_val_loader=rgb_val_loader,
        device=device,
        eval_every_n=config.eval_period,
        vis_dir=os.path.join(config.output_dir, "vis"),
    )
    # Wire the assembled dependencies into the orchestrator. Run lifecycle
    # decisions such as resume, initial evaluation, and final save stay outside.
    trainer = CurriculumDomainAdaptationTrainer(
        student=student,
        rgb_teacher=rgb_teacher,
        ir_teacher=ir_teacher,
        optimizer=optimizer,
        config=config,
        rgb_loader=rgb_train_loader,
        ir_loader=ir_train_loader,
        phase_evaluator=phase_evaluator,
        lr_scheduler=lr_scheduler,
        pseudo_label_evaluator=build_pseudo_label_evaluator(config, logger=logger),
    )

    # Connect an evaluator improvement to saving the complete training state.
    # Bind it before initial evaluation or resume, as in the original entrypoint.
    def on_best_found(results):
        trainer.save_checkpoint("best_model.pth")
        logger.info(
            "New best IR mAP@0.5 saved: %.4f",
            results.get("mAP@0.5", 0.0),
        )

    phase_evaluator.register_best_fn(on_best_found)

    # Return a ready-to-run object without starting training during assembly.
    return trainer


In [ ]:
%%flir_module training.data_flow
"""Prepare aligned views using routing decisions supplied by the caller."""

from typing import Iterable

import torch


class RouteDataFlow:
    """Own loader cursors and view preparation, without reading training config."""

    def __init__(self, rgb_loader, ir_loader, *, device, augmentor, bridge=None):
        # Keep execution dependencies here; the trainer supplies the source,
        # supervision flag, and appearance settings separately for each step.
        self.device = device
        self.augmentor = augmentor
        if bridge is None:
            from intermediate_domain import IntermediateDomainAugmentation
            bridge = IntermediateDomainAugmentation()
        self.bridge = bridge
        # Construct lazy cursors in the same order as the original trainer.
        self._rgb_iter = iter(self._infinite(rgb_loader))
        self._ir_iter = None if ir_loader is None else iter(self._infinite(ir_loader))

    @staticmethod
    def _infinite(loader: Iterable):
        # Present each loader as a continuous stream across curriculum phases.
        # An empty loader is rejected instead of leaving training waiting forever.
        while True:
            yielded = False
            for batch in loader:
                yielded = True
                yield batch
            if not yielded:
                raise ValueError("training DataLoader is empty")

    def _move_images(self, images):
        # Accept both supported image batch representations at the device seam.
        if isinstance(images, torch.Tensor):
            return images.to(self.device)
        return tuple(image.to(self.device) for image in images)

    def _move_targets(self, targets):
        # Move tensor annotations while retaining sample metadata used to align KD.
        moved = []
        for target in targets:
            moved.append({
                key: value.to(self.device) if isinstance(value, torch.Tensor) else value
                for key, value in target.items()
            })
        return moved

    @staticmethod
    def _ids_from_targets(targets) -> tuple[str, ...]:
        # Support labeled loaders that carry identity inside targets rather than
        # as a separate batch field; correspondence must still be explicit.
        ids = tuple(target.get("stem") for target in targets)
        if any(not isinstance(value, str) or not value.strip() for value in ids):
            raise ValueError("labeled batches must carry one non-empty sample stem per image")
        return ids

    def _unpack_rgb_batch(self, batch):
        # Normalize the labeled source contract before view creation so images,
        # annotations, and identity stay associated throughout augmentation.
        if not isinstance(batch, (tuple, list)) or len(batch) not in (2, 3):
            raise ValueError("RGB loader must return (images, targets[, sample_ids])")
        images, targets = batch[0], list(batch[1])
        ids = tuple(batch[2]) if len(batch) == 3 else self._ids_from_targets(targets)
        if len(ids) != len(targets):
            raise ValueError("RGB sample_ids must match target count")
        return self._move_images(images), self._move_targets(targets), ids

    def _unpack_ir_batch(self, batch):
        # Normalize the unlabeled target contract; identity is required even
        # though real IR training batches do not provide supervised targets.
        if not isinstance(batch, (tuple, list)) or len(batch) not in (1, 2, 3):
            raise ValueError("IR loader must return (images[, sample_ids])")
        images = self._move_images(batch[0])
        if len(batch) >= 2:
            ids = tuple(batch[1])
        else:
            raise ValueError("unlabeled IR batches must carry sample_ids")
        image_count = images.shape[0] if isinstance(images, torch.Tensor) else len(images)
        if len(ids) != image_count or any(not isinstance(value, str) or not value.strip() for value in ids):
            raise ValueError("IR sample_ids must contain one non-empty ID per image")
        return images, ids

    def prepare(
        self,
        *,
        source_domain: str,
        use_gt: bool,
        student_alpha: float | None,
        teacher_alpha: float | None,
    ):
        """Execute the supplied source/augmentation settings on one batch."""
        if source_domain == "rgb":
            # Labeled-source path, including RGB-backed routes named "IR flow":
            # fetch one batch and establish shared geometry for both views and GT.
            images, targets, sample_ids = self._unpack_rgb_batch(next(self._rgb_iter))
            geometric_images, geometric_targets, _ = self.augmentor.apply_weak_aug(
                images, targets
            )
            # Build different appearance views from that same geometric batch;
            # the stronger photometric augmentation belongs to the student only.
            student_images, teacher_images = self.bridge.make_views(
                geometric_images, (student_alpha, teacher_alpha),
                boxes=[target["boxes"] for target in geometric_targets],
            )
            student_images = self.augmentor.apply_photometric_aug(student_images)
            # Publish one prepared batch for forward/loss execution. The caller's
            # supervision flag controls whether its transformed GT is exposed.
            return {
                "student_images": student_images,
                "teacher_images": teacher_images,
                "targets": geometric_targets if use_gt else None,
                "sample_ids": sample_ids,
            }

        if source_domain == "ir":
            # Real-target path: create paired views from one unlabeled IR batch.
            # No annotation-based SAGA is applied on this path.
            if self._ir_iter is None:
                raise ValueError("IR route requested without an IR DataLoader")
            images, sample_ids = self._unpack_ir_batch(next(self._ir_iter))
            geometric_images, _, _ = self.augmentor.apply_weak_aug(images)
            teacher_images = geometric_images.clone()
            student_images = self.augmentor.apply_photometric_aug(geometric_images)
            # Publish aligned target views and identity without supervised GT.
            # The model keeps the baseline detector's existing forward interface.
            return {
                "student_images": student_images,
                "teacher_images": teacher_images,
                "targets": None,
                "sample_ids": sample_ids,
            }
        raise ValueError(f"Unsupported physical source domain: {source_domain}")


In [ ]:
%%flir_module training.loss_flow
"""Execute loss computation with weights and KD settings resolved by the caller."""

import torch

from models.d3t_adapter import CriterionResult


def component_weight(name: str, *, uhl: float, quality: float, deltas: float, logits: float):
    """Apply the original name matching order to explicitly supplied weights."""
    if "uhl" in name:
        return uhl
    if "quality" in name:
        return quality
    if "box" in name or "delta" in name:
        return deltas
    if "cls" in name or "logit" in name:
        return logits
    return 1.0


class LossFlow:
    """Accumulate one iteration's losses without selecting phases or teachers."""

    def __init__(self, *, uhl: float, quality: float, deltas: float, logits: float):
        # Each instance belongs to one iteration. Resolve configuration upstream
        # and keep its differentiable total separate from detached log values.
        self._weights = dict(uhl=uhl, quality=quality, deltas=deltas, logits=logits)
        self.total_loss = None
        self.logs = {}

    def _add(self, name, value, phase_weight, log_name):
        # Keep multiplication and accumulation order for numerical parity.
        weighted = value * phase_weight * component_weight(name, **self._weights)
        self.total_loss = (
            weighted if self.total_loss is None else self.total_loss + weighted
        )
        self.logs[log_name] = float(weighted.detach().item())

    def supervised(self, student, student_output, targets, *, phase_weight: float):
        # Reuse the existing student forward to compute GT losses; apply the
        # supplied phase/component weights through the shared accumulator.
        for name, value in student.supervised_from_output(student_output, targets).items():
            self._add(name, value, phase_weight, f"sup_{name}")

    def distill(
        self, student, student_output, teacher_output, settings,
        *, teacher_name: str, phase_weight: float,
    ):
        # The caller has already selected and forwarded this teacher. Delegate
        # prediction matching and loss formulas to the existing wrapper/criterion.
        result = student.distill_from_outputs(student_output, teacher_output, settings)
        # Normalize the supported result contracts before recording losses and
        # diagnostics, preserving compatibility with wrappers that return a dict.
        if isinstance(result, dict):
            result = CriterionResult(result, {})
        if not isinstance(result, CriterionResult):
            raise TypeError("wrapper distillation must return CriterionResult")
        # Teacher-specific names keep contributions distinct when aggregating
        # multiple distillation results into the same iteration total.
        for name, value in result.losses.items():
            self._add(name, value, phase_weight, f"{name}_{teacher_name}")
        # Diagnostics are observations only; they never contribute to backward.
        for name, value in result.metrics.items():
            if isinstance(value, torch.Tensor) and value.numel() == 1:
                self.logs[f"{name}_{teacher_name}"] = float(value.detach().item())
        # The observer may inspect selection metadata from this same KD call.
        # Only the loss/metric fields above enter the existing training path.
        return result


In [ ]:
%%flir_module training.observability
"""Logging setup and presentation, without changing training decisions."""

from datetime import datetime
import json
import logging
from pathlib import Path


_LOG_FORMAT = "%(asctime)s  %(message)s"
_DATE_FORMAT = "%H:%M:%S"
_HANDLER_MARKER = "_d3t_training_handler"


def configure_training_logging(output_dir: str) -> Path:
    """Log the current run to both the console and its output logs directory."""

    # Give this invocation its own timestamped destination under the configured
    # output directory, using the existing filename convention.
    log_dir = Path(output_dir) / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    log_path = log_dir / f"train_{timestamp}.log"

    # Replace only handlers owned by training so reconfiguration does not leave
    # duplicate outputs or open files from a previous invocation.
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.INFO)
    for handler in list(root_logger.handlers):
        if getattr(handler, _HANDLER_MARKER, False):
            root_logger.removeHandler(handler)
            handler.close()

    # Send the same records to console and UTF-8 file with the original format.
    # Other modules can use their normal loggers through root propagation.
    formatter = logging.Formatter(_LOG_FORMAT, datefmt=_DATE_FORMAT)
    handlers = (
        logging.StreamHandler(),
        logging.FileHandler(log_path, encoding="utf-8"),
    )
    for handler in handlers:
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
        setattr(handler, _HANDLER_MARKER, True)
        root_logger.addHandler(handler)

    return log_path


def log_iteration(log: dict, global_step: int, *, logger: logging.Logger) -> None:
    """Present the current metrics with the original field selection and format."""
    # Keep the existing loss/KD field selection when constructing the detail
    # section; collecting a metric does not automatically add it to text output.
    components = []
    for key, value in sorted(log.items()):
        if (
            key != "total_loss"
            and isinstance(value, float)
            and ("loss" in key or "kd_" in key)
        ):
            components.append(f"{key}={value:.4f}")
    # Present the step context first, followed by optional timing and component
    # values. Formatting here does not modify the supplied training metrics.
    message = (
        f"[{int(log.get('global_step', global_step)):06d}] "
        f"Phase: {log.get('phase', 'N/A'):<22} | "
        f"Route: {log.get('step_type', 'N/A'):<18} | "
        f"Loss: {log.get('total_loss', 0.0):.4f} | "
        f"LR: {log.get('lr', 0.0):.6g}"
    )
    if isinstance(log.get("iter_time"), (int, float)):
        message += f" | Time: {log['iter_time']:.3f}s"
    if components:
        message += " | " + " | ".join(components)
    logger.info(message)


def log_pseudo_label_evaluation(record: dict, *, output_dir: str, logger: logging.Logger):
    """Persist audit records independently of regular iteration log cadence."""
    # Append supports resumed runs. Keep sample identity and selection settings
    # in JSONL while showing metric summaries through the existing log handlers.
    log_dir = Path(output_dir) / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    with (log_dir / "pseudo_label_eval.jsonl").open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(record, allow_nan=False, sort_keys=True) + "\n")
    logger.info(
        "[pseudo-label-eval] step=%s phase=%s teacher=%s scope=%s images=%s metrics=%s",
        record["global_step"], record["phase"], record["teacher"],
        record.get("scope", "train_batch"), record.get("images", "N/A"),
        json.dumps(record["metrics"], allow_nan=False, sort_keys=True),
    )


__all__ = ["configure_training_logging", "log_iteration", "log_pseudo_label_evaluation"]


In [ ]:
%%flir_module training.pseudo_label_eval
"""Evaluate dense teacher supervision on fixed and full IR validation sets."""

from contextlib import contextmanager
import random

import numpy as np
from PIL import Image
import torch

from loss.d3t_criterion import select_distillation_masks
from models.torchvision_fcos_adapter import _aligned_box_iou


def _ratio(numerator, denominator):
    return numerator / denominator if denominator else None


@torch.no_grad()
def evaluate_pseudo_labels(adapter, output, selected_masks, annotations, *, iou_threshold):
    """Report FCOS-assigned row validity and unique GT coverage before NMS.

    Multiple selected locations may correctly cover one object. These metrics
    describe KD supervision, not detection precision/recall or teacher AP.
    """
    if selected_masks is None or not (
        len(output.predictions) == len(selected_masks) == len(annotations)
    ):
        raise ValueError("diagnostic predictions, masks, and annotations must align")
    if not 0. <= iou_threshold <= 1.:
        raise ValueError("diagnostic IoU threshold must be in [0, 1]")
    assigned = adapter.assign_diagnostic_targets(output, [r.target for r in annotations])
    counts = dict(available_images=0, missing_images=0, invalid_images=0,
                  selected_rows_all=0, selected_rows=0, selected_foreground_rows=0,
                  finite_foreground_rows=0, nonfinite_selected_rows=0,
                  valid_rows=0, correct_class_rows=0, gt_count=0, covered_gt=0)
    iou_sum = quality_error_sum = 0.
    for prediction, selected, record, target in zip(
        output.predictions, selected_masks, annotations, assigned
    ):
        if selected.dtype != torch.bool or selected.shape != prediction.quality_logits.shape:
            raise ValueError("diagnostic mask must align with prediction rows")
        counts["selected_rows_all"] += int(selected.sum().item())
        counts[f"{record.status}_images"] += 1
        # Unknown annotations contribute only to availability statistics. A
        # valid empty annotation still counts selected rows as background.
        if target is None:
            continue
        counts["selected_rows"] += int(selected.sum().item())
        counts["gt_count"] += len(target["boxes"])
        matched = target["matched_idxs"]
        foreground = selected & (matched >= 0)
        counts["selected_foreground_rows"] += int(foreground.sum().item())
        finite = (torch.isfinite(prediction.boxes).all(dim=1)
                  & torch.isfinite(prediction.class_logits).all(dim=1)
                  & torch.isfinite(prediction.quality_logits))
        counts["nonfinite_selected_rows"] += int((selected & ~finite).sum().item())
        foreground &= finite
        count = int(foreground.sum().item())
        counts["finite_foreground_rows"] += count
        if not count:
            continue

        # Quality is trained only on foreground and predicts box IoU. Compare
        # it with the same assigned GT definition, without background zeroes.
        gt_indices = matched[foreground]
        iou = _aligned_box_iou(prediction.boxes[foreground], target["boxes"][gt_indices])
        correct_class = (prediction.class_logits[foreground].argmax(dim=1)
                         == target["labels"][gt_indices])
        valid = correct_class & (iou >= iou_threshold)
        counts["correct_class_rows"] += int(correct_class.sum().item())
        counts["valid_rows"] += int(valid.sum().item())
        counts["covered_gt"] += int(gt_indices[valid].unique().numel())
        iou_sum += float(iou.sum().item())
        quality_error_sum += float((prediction.quality_logits[foreground].sigmoid() - iou).abs().sum().item())

    # Undefined denominators stay null in JSON/logs rather than suggesting a
    # perfect or zero score for an unlabeled batch or an empty foreground set.
    foreground_count = counts["finite_foreground_rows"]
    return {
        **counts,
        "valid_fraction": _ratio(counts["valid_rows"], counts["selected_rows"]),
        "foreground_class_accuracy": _ratio(counts["correct_class_rows"], foreground_count),
        "foreground_box_iou_mean": _ratio(iou_sum, foreground_count),
        "quality_mae_foreground": _ratio(quality_error_sum, foreground_count),
        "gt_coverage": _ratio(counts["covered_gt"], counts["gt_count"]),
    }


def aggregate_pseudo_label_metrics(batches):
    """Pool counts and weighted sums so small/empty batches cannot bias ratios."""
    batches = list(batches)
    count_names = (
        "available_images", "missing_images", "invalid_images", "selected_rows_all",
        "selected_rows", "selected_foreground_rows", "finite_foreground_rows",
        "nonfinite_selected_rows", "valid_rows", "correct_class_rows", "gt_count", "covered_gt",
    )
    counts = {name: sum(batch.get(name, 0) for batch in batches) for name in count_names}
    foreground = counts["finite_foreground_rows"]
    # The per-batch means already exclude non-finite/unassigned rows. Recover
    # their sums using that exact denominator, not the number of images/batches.
    means = {}
    for name in ("foreground_box_iou_mean", "quality_mae_foreground"):
        total = sum((batch.get(name) or 0.) * batch["finite_foreground_rows"] for batch in batches)
        means[name] = _ratio(total, foreground)
    return {
        **counts, **means,
        "valid_fraction": _ratio(counts["valid_rows"], counts["selected_rows"]),
        "foreground_class_accuracy": _ratio(counts["correct_class_rows"], foreground),
        "gt_coverage": _ratio(counts["covered_gt"], counts["gt_count"]),
    }


def select_fixed_ir_subset(image_paths, reader, *, sample_size, seed):
    """Choose reproducible labeled images, retaining class coverage when possible."""
    if sample_size <= 0:
        raise ValueError("fixed IR sample size must be positive")
    labels_by_index = {}
    # Read headers/annotations only during assembly. Never iterate a training
    # loader or consume its augmentation/sampling RNG to choose audit images.
    for index, path in enumerate(image_paths):
        with Image.open(path) as image:
            width, height = image.size
        record, = reader.read((path.stem,), ((height, width),), hflipped=False)
        if record.status == "available":
            labels_by_index[index] = set(record.target["labels"].tolist())
    if not labels_by_index:
        raise ValueError("no valid IR validation annotations for pseudo-label evaluation")
    candidates = list(labels_by_index)
    random.Random(seed).shuffle(candidates)
    limit = min(sample_size, len(candidates))
    classes = sorted(set().union(*labels_by_index.values()))
    selected = []
    for label in classes:
        if len(selected) == limit:
            break
        if any(label in labels_by_index[i] for i in selected):
            continue
        selected.append(next(i for i in candidates if label in labels_by_index[i]))
    selected.extend(i for i in candidates if i not in selected)
    return tuple(sorted(selected[:limit]))


@contextmanager
def _preserve_evaluation_state(model, generator):
    """Leave training RNG and every pre-existing model mode intact, even on errors."""
    modes = [(module, module.training) for module in model.modules()]
    python_rng, numpy_rng = random.getstate(), np.random.get_state()
    cpu_rng = torch.get_rng_state()
    cuda_rng = torch.cuda.get_rng_state_all() if torch.cuda.is_initialized() else None
    loader_rng = generator.get_state() if generator is not None else None
    try:
        model.eval()
        yield
    finally:
        for module, was_training in modes:
            module.training = was_training
        random.setstate(python_rng)
        np.random.set_state(numpy_rng)
        torch.set_rng_state(cpu_rng)
        if cuda_rng is not None:
            torch.cuda.set_rng_state_all(cuda_rng)
        if loader_rng is not None:
            generator.set_state(loader_rng)


class FixedIRPseudoLabelEvaluator:
    """Run isolated teacher-only passes using caller-supplied loaders and settings."""

    def __init__(self, *, fixed_loader, full_loader, reader, device, fixed_sample_ids):
        self.fixed_loader = fixed_loader
        self.full_loader = full_loader
        self.reader = reader
        self.device = device
        self.fixed_sample_ids = tuple(fixed_sample_ids)

    def state_dict(self):
        return {"fixed_sample_ids": list(self.fixed_sample_ids),
                "batch_size": self.fixed_loader.batch_size}

    def load_state_dict(self, state):
        # Comparing the manifest makes resumed metrics refer to the same images
        # and batch-global top-k groups; a changed dataset is not silently mixed.
        if state != self.state_dict():
            raise ValueError("pseudo-label evaluation subset or batch size changed on resume")

    def evaluate(self, teacher, settings, *, iou_threshold, resize_short_edge, full_dataset=False):
        loader = self.full_loader if full_dataset else self.fixed_loader
        batch_metrics, sample_ids = [], []
        # No student forward, target-dependent selection, backward, or EMA is
        # performed here. Teacher weights stay fixed throughout the entire pass.
        with _preserve_evaluation_state(teacher, loader.generator), torch.inference_mode():
            for images, ids in loader:
                images = (images.to(self.device) if isinstance(images, torch.Tensor)
                          else tuple(image.to(self.device) for image in images))
                output = teacher.raw(images, sample_ids=ids, resize_short_edge=resize_short_edge)
                annotations = self.reader.read(ids, output.context.original_image_sizes, hflipped=False)
                masks = select_distillation_masks(output.predictions, settings)
                batch_metrics.append(evaluate_pseudo_labels(
                    teacher.adapter, output, masks, annotations, iou_threshold=iou_threshold,
                ))
                sample_ids.extend(ids)
        if not batch_metrics:
            raise ValueError("pseudo-label evaluation loader is empty")
        if not full_dataset and tuple(sample_ids) != self.fixed_sample_ids:
            raise ValueError("fixed pseudo-label evaluation image order changed")
        return {
            "scope": "full_ir_validation" if full_dataset else "fixed_ir_validation",
            "images": len(sample_ids), "batches": len(batch_metrics),
            "batch_size": loader.batch_size, "sample_ids": sample_ids,
            "metrics": aggregate_pseudo_label_metrics(batch_metrics),
        }


In [ ]:
%%flir_module training.teachers
"""Execute teacher operations requested by the trainer, without schedule policy."""

import torch

from ema import copy_student_to_teacher, ema_update


def prepare_models(student, teachers, *, device):
    """Preserve device transfer order and freeze the supplied teachers."""
    # Establish model placement first, then make every supplied teacher an
    # inference-only reference. Teacher enablement is decided by the caller.
    student.to(device)
    for teacher in teachers.values():
        if teacher is None:
            continue
        teacher.to(device)
        teacher.eval()
        for parameter in teacher.parameters():
            parameter.requires_grad = False


def initialize_teachers(student, teachers, *, names):
    """Hard-copy only the names selected by the caller at its chosen start step."""
    # Execute one initialization request. The trainer owns the start-step check
    # and persisted initialization flag, including the behavior after resume.
    for name in names:
        teacher = teachers[name]
        if teacher is None:
            raise ValueError(f"EMA start requires the '{name}' teacher")
        copy_student_to_teacher(teacher, student)


def get_teacher(teachers, name):
    # Resolve a requested reference and restore its inference mode before use.
    # Route/teacher-mode filtering happens upstream, rather than in this lookup.
    teacher = teachers.get(name)
    if teacher is None:
        raise ValueError(f"teacher '{name}' is not configured")
    teacher.eval()
    return teacher


def forward_teacher(teacher, images, *, sample_ids, resize_short_edge):
    """Reuse the student-selected geometry and keep teacher forward outside autograd."""
    # Produce KD targets from the prepared teacher view with the same sample
    # identity and resize selected for the student forward.
    with torch.no_grad():
        kwargs = {"sample_ids": sample_ids}
        if resize_short_edge is not None:
            kwargs["resize_short_edge"] = resize_short_edge
        return teacher.raw(images, **kwargs)


def update_teacher(teacher, student, *, alpha, global_step):
    """Run the existing EMA primitive with caller-supplied parameters."""
    # The trainer issues this request after an eligible student update; the
    # existing EMA primitive retains responsibility for parameter/buffer math.
    ema_update(teacher, student, alpha=alpha, global_step=global_step)


In [ ]:
%%flir_module utils.helper
from typing import List
from pathlib import Path


def ir_stem_to_rgb_filename(stem: str) -> str:
    """FLIR_XXXXX_PreviewData  →  FLIR_XXXXX_RGB.jpg"""
    base = stem.replace("_PreviewData", "")
    return f"{base}_RGB.jpg"


def read_split_file(split_file: Path) -> List[str]:
    """Read ImageSets/Main/*.txt — one stem per line, ignoring blank lines."""
    with open(split_file, "r") as f:
        return [line.strip() for line in f if line.strip()]

In [ ]:
# Register the embedded source as real Python modules without project files.
import importlib.abc
import importlib.util
import linecache
import sys

class EmbeddedProjectFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    _flir_embedded_finder = True

    def __init__(self, sources):
        self.sources = sources

    def find_spec(self, fullname, path=None, target=None):
        item = self.sources.get(fullname)
        if item is None:
            return None
        return importlib.util.spec_from_loader(fullname, self, is_package=item["is_package"])

    def create_module(self, spec):
        return None

    def get_source(self, fullname):
        return self.sources[fullname]["source"]

    def get_filename(self, fullname):
        item = self.sources[fullname]
        path = item["path"] or fullname.replace(".", "/") + "/__init__.py"
        return "/__flir_embedded__/" + path

    def exec_module(self, module):
        source = self.get_source(module.__name__)
        filename = self.get_filename(module.__name__)
        module.__file__ = filename
        module.__embedded_flir__ = True
        if self.sources[module.__name__]["is_package"]:
            module.__path__ = [filename.rsplit("/", 1)[0]]
        linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
        exec(compile(source, filename, "exec"), module.__dict__)


def install_embedded_project(sources):
    sys.meta_path[:] = [finder for finder in sys.meta_path if not getattr(finder, "_flir_embedded_finder", False)]
    for name in sorted(sources, key=len, reverse=True):
        sys.modules.pop(name, None)
    finder = EmbeddedProjectFinder(sources)
    sys.meta_path.insert(0, finder)
    return finder

missing_modules = sorted(set(EXPECTED_SOURCE_MANIFEST) - set(EMBEDDED_SOURCES))
if missing_modules:
    raise RuntimeError("Run all module cells before this loader cell. Missing: " + ", ".join(missing_modules))
install_embedded_project(EMBEDDED_SOURCES)
SOURCE_MANIFEST = {name: {key: value for key, value in item.items() if key != "source"}
                   for name, item in EMBEDDED_SOURCES.items()}
print("Loaded", len(EMBEDDED_SOURCES), "embedded project modules; no checkout is required.")


## Experiment settings
The original CLI defaults remain unless overridden here. `workers=0`
avoids multiprocessing import issues with in-memory notebook modules.
Set `intermediate_domain="saga"` for legacy SAGA, or
`texture_strength=0.0` for a RandomMono-only ablation.
`resume` resumes training; `eval_checkpoint` selects a checkpoint for the
final eval cell and otherwise defaults to best, falling back to final.


In [ ]:
SETTINGS = {'data_root': '/kaggle/input/datasets/tungduong123qwe/flir-dataset/align',
 'output_dir': '/kaggle/working/output_randommono_btf',
 'total_iters': 10000,
 'batch_size': 4,
 'eval_batch_size': 4,
 'workers': 0,
 'device': 'cuda',
 'weights': 'DEFAULT',
 'workflow': 'curriculum',
 'teacher_mode': 'two_teacher',
 'intermediate_domain': 'randommono_btf',
 'texture_strength': 0.25,
 'btf_radius': 2,
 'btf_iterations': 1,
 'eval_every': 500,
 'log_interval': 50,
 'eval_pseudo_labels': False,
 'resume': None,
 'eval_checkpoint': None,
 'seed': 42}


In [ ]:
import example_flir
import notebook_support

train_args = notebook_support.make_training_args(SETTINGS)
effective_config = notebook_support.preflight(train_args, SOURCE_MANIFEST)
preview_path = notebook_support.save_bridge_preview(train_args)
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(preview_path)))


In [ ]:
# Same entrypoint as `python example_flir.py ...`, called directly.
# Re-seed after the preview so visualization cannot alter training RNG.
notebook_support.seed_everything(SETTINGS["seed"])
example_flir.main(train_args)
notebook_support.release_training_memory()
print("Training completed. Checkpoints are in:", train_args.output_dir)


## Final IR COCO evaluation
The last cell loads the saved student through the original evaluator.
It reports AP/AP50/AP75, size metrics and per-class metrics, and writes a
JSON result. You can rerun this cell with another `eval_checkpoint`.


In [ ]:
import json
from pathlib import Path
import evaluate_coco

notebook_support.release_training_memory()
eval_checkpoint = notebook_support.select_checkpoint(
    SETTINGS["output_dir"], SETTINGS.get("eval_checkpoint")
)
metrics_path = Path(SETTINGS["output_dir"]) / f"coco_ir_{eval_checkpoint.stem}.json"
eval_args = evaluate_coco.parse_args([
    "--checkpoint", str(eval_checkpoint), "--data-root", str(SETTINGS["data_root"]),
    "--domain", "ir", "--batch-size", str(SETTINGS["eval_batch_size"]),
    "--workers", str(SETTINGS["workers"]), "--device", SETTINGS["device"],
    "--output", str(metrics_path),
])
coco_metrics = evaluate_coco.evaluate(eval_args)
print("Evaluated checkpoint:", eval_checkpoint)
print("Metrics saved:", metrics_path)
print(json.dumps(coco_metrics, indent=2))
